> ### ⚠️ Repository note ⚠️
> This notebook stored in the repository is a cleaned version with outputs removed, because the original fully rendered notebook exceeds GitHub’s practical file-size limits.
>
> **Full notebook including all results and figures(HTML):** [Open Full Notebook](https://rsalehin.github.io/documentation/)
>
> The corresponding figure files are also provided separately in the repository for direct inspection.

# ML4SCI DeepLense — GSoC 2026
## Common Test I — Multi-Class Classification of Dark Matter Substructure

This notebook benchmarks standard, pretrained, and equivariant architectures for three-class classification of simulated strong-lensing images with dark-matter substructure.

**Headline result:** EqDenseNet-C8 achieved the best benchmark performance with the strongest parameter efficiency.

**Quick links:** [Benchmark Setup](#3-training--evaluation-framework) · [Results Summary](#5-comprehensive-results-summary) · [Failure Analysis](#7-sphere-class-analysis--failure-mode-analysis) · [Discussion](#9-discussion)

## Table of Contents

- [1. Environment Setup & Dependencies](#1-environment-setup--dependencies)
- [2. Exploratory Data Analysis](#2-exploratory-data-analysis)
  - [2.1 Class Visualisation and Pixel Intensity Distribution](#21-class-visualisation-and-pixel-intensity-distribution)
  - [2.2 Observations](#22-observations)
- [3. Training & Evaluation Framework](#3-training--evaluation-framework)
  - [3.1 Data Pipeline & Two-Way Split](#31-data-pipeline--two-way-split)
  - [3.2 Key Design Decisions](#32-key-design-decisions)
  - [3.3 Evaluation Protocol](#33-evaluation-protocol)
  - [3.4 Training Loop](#34-training-loop)
  - [3.5 Reproducibility — Load or Train](#35-reproducibility--load-or-train)
- [4. Architecture Evaluations](#4-architecture-evaluations)
  - [4.1 AlexNet](#41-alexnet)
  - [4.2 VGG-16](#42-vgg-16)
  - [4.3 ResNet-18](#43-resnet-18)
  - [4.4 ResNet-50](#44-resnet-50)
  - [4.5 DenseNet-121](#45-densenet-121)
  - [4.6 EfficientNet-B3](#46-efficientnet-b3)
  - [4.7 ViT-Base](#47-vit-base)
  - [4.8 Equivariant Neural Network (D4 Symmetry)](#48-equivariant-neural-network-d4-symmetry)
  - [4.9 Equivariant Residual Network (E-ResNet)](#49-equivariant-residual-network-e-resnet)
  - [4.10 Equivariant DenseNet (C8)](#410-equivariant-densenet-c8)
  - [4.11 Top-6 Soft-Voting Ensemble](#411-top-6-soft-voting-ensemble)
- [5. Comprehensive Results Summary](#5-comprehensive-results-summary)
  - [5.1 Macro-Averaged ROC Curves](#51-macro-averaged-roc-curves)
  - [5.2 Sphere-Class Precision-Recall Comparison](#52-sphere-class-precision-recall-comparison)
  - [5.3 Parameter Efficiency](#53-parameter-efficiency)
  - [5.4 Cross Architecture Confusion Matrix Grid](#54-cross-architecture-confusion-matrix-grid)
  - [5.5 Key Takeaways](#55-key-takeaways)
- [6. Understanding the Models](#6-understanding-the-models)
  - [6.1 Class Activation Maps — EqDenseNet-C8](#61-class-activation-maps--eqdensenet-c8)
  - [6.2 Ring Concentration: EqDenseNet-C8](#62-ring-concentration-eqdensenet-c8)
  - [6.3 Interpretability: ResNet-50 vs E-ResNet](#63-interpretability-resnet-50-vs-e-resnet)
    - [6.3.1 Results](#631-results)
  - [6.4 Disagreement Analysis — When the Models Diverge](#64-disagreement-analysis--when-the-models-diverge)
    - [6.4.1 Case 2 — ResNet-50 Correct, E-ResNet Wrong](#641-case-2--resnet-50-correct-e-resnet-wrong)
    - [6.4.2 Case 3 — E-ResNet Correct, ResNet-50 Wrong](#642-case-3--e-resnet-correct-resnet-50-wrong)
    - [6.4.3 Case 4 — Both Wrong](#643-case-4--both-wrong)
    - [6.4.4 Disagreement Summary](#644-disagreement-summary)
  - [6.5 Interpretability: ViT vs DenseNet](#65-interpretability-vit-vs-densenet)
    - [6.5.1 Setup: ViT Attention Patching and Attention Rollout](#651-setup-vit-attention-patching-and-attention-rollout)
    - [6.5.2 Initialisation](#652-initialisation)
    - [6.5.3 Image Selection](#653-image-selection)
    - [6.5.4 Spatial Attention Grid](#654-spatial-attention-grid)
    - [6.5.5 DenseNet Resolution Diagnostic](#655-densenet-resolution-diagnostic)
    - [6.5.6 ViT Ring Concentration and Attention Consistency](#656-vit-ring-concentration-and-attention-consistency)
    - [6.5.7 Ring Concentration: DenseNet vs ViT](#657-ring-concentration-densenet-vs-vit)
    - [6.5.8 Summary](#658-summary)
  - [6.6 Predictive Uncertainty — Deep Ensemble (M=8)](#66-predictive-uncertainty--deep-ensemble-m8)
    - [6.6.1 Entropy Distribution Results](#661-entropy-distribution-results)
    - [6.6.2 High-Entropy Sphere Images](#662-high-entropy-sphere-images)
    - [6.6.3 Two Distinct Sphere Failure Populations](#663-two-distinct-sphere-failure-populations)
    - [6.6.4 Practical Implication for Survey Pipelines](#664-practical-implication-for-survey-pipelines)
  - [6.7 Ablation Study — E-ResNet Component Analysis](#67-ablation-study--e-resnet-component-analysis)
    - [6.7.1 Hypothesis Assessment](#671-hypothesis-assessment)
  - [6.8 Rotation Invariance Verification](#68-rotation-invariance-verification)
    - [6.8.1 Equivariance Verification Test](#681-equivariance-verification-test)
    - [6.8.2 Stage 1 — Theoretical Invariance (Untrained Models)](#682-stage-1--theoretical-invariance-untrained-models)
    - [6.8.3 Stage 2 — Empirical Invariance After Training](#683-stage-2--empirical-invariance-after-training)
    - [6.8.4 Theoretical vs Empirical Invariance](#684-theoretical-vs-empirical-invariance)
    - [6.8.5 Connection to §6.7 and Deployment Recommendation](#685-connection-to-67-and-deployment-recommendation)
- [7. Sphere Class Analysis & Failure Mode Analysis](#7-sphere-class-analysis--failure-mode-analysis)
  - [7.1 Cross-Architecture Sphere Confusion Pattern](#71-cross-architecture-sphere-confusion-pattern)
    - [7.1.1 Results](#711-results)
    - [7.1.2 Interpretation](#712-interpretation)
    - [7.1.3 Future DeepLense Project Activity — Failure-Cohort Analysis for Hard Sphere Cases](#713-future-deeplense-project-activity--failure-cohort-analysis-for-hard-sphere-cases)
  - [7.2 E-ResNet and EqDenseNet-C8 Failure Mode Analysis](#72-e-resnet-and-eqdensenet-c8-failure-mode-analysis)
    - [7.2.1 Sphere False Negatives (Fig. 12a–12b)](#721-sphere-false-negatives-fig-12a12b)
    - [7.2.2 Sphere ↔ Vortex Confusion (Fig. 12c–12d)](#722-sphere--vortex-confusion-fig-12c12d)
    - [7.2.3 Classification Accuracy vs Ring Brightness (Fig. 12e)](#723-classification-accuracy-vs-ring-brightness-fig-12e)
    - [7.2.4 Perturbation Position vs Failure (Fig. 12f)](#724-perturbation-position-vs-failure-fig-12f)
  - [7.3 Sphere Class Analysis on E-DenseNet-C8](#73-sphere-class-analysis-on-e-densenet-c8)
    - [7.3.1 EqDenseNet-C8 Failure Mode Analysis](#731-eqdensenet-c8-failure-mode-analysis)
- [8. Residual Image Approach (Experimental)](#8-residual-image-approach-experimental)
  - [8.1 Motivation](#81-motivation)
  - [8.2 Analytical Fitting (Abandoned)](#82-analytical-fitting-abandoned)
  - [8.3 CAE Architecture & Training](#83-cae-architecture--training)
  - [8.4 Residual Visualisation & Statistics](#84-residual-visualisation--statistics)
  - [8.5 Residual Classifier: DenseNet-121](#85-residual-classifier-densenet-121)
  - [8.6 Residual Classifier: ResNet-18](#86-residual-classifier-resnet-18)
  - [8.7 Summary & Interpretation](#87-summary--interpretation)
- [9. Discussion](#9-discussion)
  - [9.1 The Sphere Class is Systematically Harder](#91-the-sphere-class-is-systematically-harder)
  - [9.2 Skip Connections Strongly Improve Performance in This Benchmark](#92-skip-connections-strongly-improve-performance-in-this-benchmark)
  - [9.3 Equivariant Inductive Bias Improves Parameter Efficiency](#93-equivariant-inductive-bias-improves-parameter-efficiency)
  - [9.4 The Role of Augmentation in Equivariant Networks](#94-the-role-of-augmentation-in-equivariant-networks)
  - [9.5 Architecture Priors Aligned with the Task Appear Advantageous](#95-architecture-priors-aligned-with-the-task-appear-advantageous)
  - [9.6 Deep Ensemble Uncertainty Reveals Two Distinct Failure Groups](#96-deep-ensemble-uncertainty-reveals-two-distinct-failure-groups)
  - [9.7 The Residual Image Approach — Scientific Value and Practical Ceiling](#97-the-residual-image-approach--scientific-value-and-practical-ceiling)
- [10. Limitations](#10-limitations)
- [11. Future Work](#11-future-work)
  - [11.1 Architectural Directions](#111-architectural-directions)
  - [11.2 Detection and Estimation Extensions](#112-detection-and-estimation-extensions)
  - [11.3 Robustness and Calibration](#113-robustness-and-calibration)
  - [11.4 Interpretability and Analysis](#114-interpretability-and-analysis)
- [12. Future DeepLense Research](#12-future-deeplense-research)

## Executive Summary

- EqDenseNet-C8 achieved the strongest overall benchmark result.
- Sphere is the hardest class across all strong models.
- Equivariant models were highly parameter-efficient and more rotation-stable.
- Residual-image classification retained some signal but underperformed raw-image classification.

##<a></a> 1. Environment Setup & Dependencies

All experiments were run on Google Colab with a single NVIDIA A100 GPU.

Three packages require explicit installation beyond the Colab base environment:

- `escnn` — E(2)-equivariant steerable CNNs, used for the ENN and E-ResNet architectures
- `lenstronomy` — gravitational lens modelling, used in the analytical fitting attempt (Section 8.2)
- `grad-cam` — Grad-CAM implementation, used for interpretability analysis (Section 6.1)

**numpy is pinned to <2** because both `escnn` and `lenstronomy` require numpy 1.x.
Colab ships numpy 2.4.x by default. The runtime must be restarted after installation
for the version change to take effect.

In [ ]:
# Install
!pip install lenstronomy --quiet
!pip install escnn --quiet
!pip install grad-cam --quiet
!pip install "numpy<2"
# ⚠️  RESTART RUNTIME after running this cell, then continue from the next cell.

In [ ]:
import numpy as np
print(np.__version__)   # must show 1.26.x

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────
import os, zipfile, glob, copy, time

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── ML / Metrics ──────────────────────────────────────────────────────────────
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    classification_report, precision_recall_curve,
    average_precision_score, brier_score_loss
)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import calibration_curve

# ── Deep Learning ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models
import torchvision.transforms as T
from tqdm import tqdm

# ── Environment ───────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)


In [ ]:
# ── Environment snapshot for reproducibility ─────────────────────────────
import pkg_resources

packages = [
    'numpy', 'torch', 'torchvision', 'escnn', 'lenstronomy',
    'grad-cam', 'scikit-learn', 'matplotlib', 'seaborn', 'tqdm'
]

print('Package versions used in this notebook:')
print('─' * 40)
for pkg in packages:
    try:
        version = pkg_resources.get_distribution(pkg).version
        print(f'  {pkg:<20} {version}')
    except pkg_resources.DistributionNotFound:
        print(f'  {pkg:<20} not found')

print('─' * 40)
print(f'  {"Python":<20}', end=' ')
import sys; print(sys.version.split()[0])
print(f'  {"CUDA":<20}', end=' ')
print(torch.version.cuda if torch.cuda.is_available() else 'not available')
print(f'  {"cuDNN":<20}', end=' ')
print(torch.backends.cudnn.version() if torch.cuda.is_available() else 'not available')

The output of the cell above is the exact environment used to produce all
results in this notebook. Python 3.12.12, CUDA 12.8, cuDNN 91002,
single NVIDIA GPU runtime. To reproduce, install the pinned versions
provided in `requirements.txt` in the repository root.



## <a id="2-exploratory-data-analysis"></a>2. Exploratory Data Analysis

The dataset comprises simulated strong gravitational lensing images across
three dark matter substructure classes, constructed using the DeepLenseSim
pipeline (Toomey et al.). Each image is a 150×150 single-channel numpy array
simulated under a Sheared Isothermal Elliptical lens model with a Sérsic
light profile, Gaussian PSF, and Gaussian+Poissonian noise at SNR ≈ 25.

| Class | Physical description | Train | Val |
|---|---|---|---|
| No Substructure | Smooth dark matter halo, no small-scale perturbation | 10,000 | 2,500 |
| Sphere (CDM subhalo) | Point-mass subhalo appended to base halo | 10,000 | 2,500 |
| Vortex (Axion DM) | Topological vortex in axion field distribution | 10,000 | 2,500 |

The dataset is perfectly balanced across all three classes. The predefined
train/val split is used directly as provided by the dataset organisers.

In [ ]:
CLASSES      = ['no', 'sphere', 'vort']
CLASS_NAMES  = ['No Substructure', 'Sphere', 'Vortex']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# ── Paths ─────────────────────────────────────────────────────────────────────
zip_path        = '/content/drive/MyDrive/gsoc/dataset.zip'
extract_path    = '/content/lensing_data'
gsoc_drive_path = '/content/drive/MyDrive/gsoc'
os.makedirs(gsoc_drive_path, exist_ok=True)

if not os.path.exists(extract_path):
    print('Unzipping…')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print('Done.')

base_path  = '/content/lensing_data/dataset'
train_path = os.path.join(base_path, 'train')
val_path   = os.path.join(base_path, 'val')

# Sanity check
for cls in CLASSES:
    n = len(glob.glob(os.path.join(train_path, cls, '*.npy')))
    print(f'  train/{cls}: {n} files')

# ── Dataset ───────────────────────────────────────────────────────────────────
class LensingDataset(Dataset):
    def __init__(self, root_dir, augment=False):
        self.augment = augment
        self.filepaths, self.labels = [], []
        for cls in CLASSES:
            fps = glob.glob(os.path.join(root_dir, cls, '*.npy'))
            self.filepaths.extend(fps)
            self.labels.extend([CLASS_TO_IDX[cls]] * len(fps))

        self._aug = T.Compose([
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(90),
        ])

    def __len__(self): return len(self.filepaths)

    def __getitem__(self, idx):
        img = np.load(self.filepaths[idx]).astype(np.float32)

        if img.ndim == 2:
            img = img[np.newaxis]
        elif img.shape[-1] == 1:
            img = img.transpose(2, 0, 1)

        # Per-sample min-max normalisation
        img_min, img_max = img.min(), img.max()
        if img_max - img_min > 0:
            img = (img - img_min) / (img_max - img_min)

        t = torch.from_numpy(img)
        if self.augment:
            t = self._aug(t)
        return t, torch.tensor(self.labels[idx], dtype=torch.long)

# ── Loaders ───────────────────────────────────────────────────────────────────
train_ds = LensingDataset(train_path, augment=True)
val_ds   = LensingDataset(val_path,   augment=False)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain: {len(train_ds):,}  |  Val: {len(val_ds):,}')
for cls, name in zip(CLASSES, CLASS_NAMES):
    n = sum(1 for p in val_ds.filepaths if f'/{cls}/' in p)
    print(f'  {name}: {n:,}')

ARCH_METRICS = {} # A shared dictionary that accumulates results from every architecture evaluation.

### 2.1 Class Visualisation and Pixel Intensity Distribution

A representative image from each class is shown alongside the pixel intensity
histogram aggregated over 50 randomly sampled training images. The raw images
are shown before normalisation to preserve the original intensity scale.

In [ ]:
def plot_eda(base_dir):
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))

    for col, (cls, name) in enumerate(zip(CLASSES, CLASS_NAMES)):
        fps = glob.glob(os.path.join(base_dir, 'train', cls, '*.npy'))
        imgs = [np.load(fp).squeeze() for fp in fps[:50]]

        # Raw image
        axes[0, col].imshow(imgs[0], cmap='magma', origin='lower')
        axes[0, col].set_title(f'{name}\n(sample image)', fontsize=11)
        axes[0, col].axis('off')

        # Pixel intensity histogram across 50 samples
        flat = np.concatenate([i.ravel() for i in imgs])
        axes[1, col].hist(flat, bins=60, color='steelblue', alpha=0.8, density=True)
        axes[1, col].set_title(f'{name}\npixel distribution (n=50)', fontsize=11)
        axes[1, col].set_xlabel('Normalised intensity')
        axes[1, col].set_ylabel('Density')

    plt.suptitle('Class Visualisation and Intensity Distributions', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

plot_eda(base_path)


### 2.2 Observations

The three classes share the same dominant ring morphology, so the task appears to depend on subtle perturbations rather than large global differences. The pixel distributions are strongly right-skewed in all classes, with most pixels near zero and a small fraction of bright ring pixels carrying most of the signal. Since only a few examples are shown, these observations are qualitative and should not be over-interpreted as definitive class-level structure.

## <a id="3-training--evaluation-framework"></a>3. Training & Evaluation Framework

### 3.1 Data Pipeline & Two-Way Split

**The organiser-provided train/ and val/ partitions are used directly. No additional held-out test set is created. Consequently, the validation set is used both for checkpoint selection and for reported performance, so the reported metrics should be interpreted as validation-set benchmark results rather than a fully independent final estimate of generalization.**

| Partition | Source | Images per class | Role |
|---|---|---|---|
| Train | `train/` folder | 10,000 | Weight updates only |
| Val | `val/` folder | 2,500 | Best-checkpoint selection & final metric reporting |

No held-out test partition is created. All evaluation metrics are reported
on the predefined `val/` partition. This split was used as provided by the
dataset organisers.

**Normalisation.** Per-sample min-max normalisation is applied in the data
loader: `x_norm = (x - x_min) / (x_max - x_min)`. Each image is normalised
independently using its own pixel range, preserving internal contrast
structure regardless of overall exposure level.

**Augmentation.** Applied to training images only — random horizontal flip,
vertical flip, and 90° rotation. These transformations are physically
motivated: a gravitational lens has no preferred orientation in the sky, so
all four 90° rotations and their reflections represent the same physical
system. The val set is never augmented.

### 3.2 Key Design Decisions

| Decision | Choice | Rationale |
|---|---|---|
| Optimiser | AdamW (wd=1e-3) | Weight decay regularises large pretrained models without penalising batch norm parameters |
| Scheduler | CosineAnnealingLR | Smooth LR decay without manual step tuning, completes full annealing cycle |
| Loss | CrossEntropyLoss | Standard for balanced multi-class classification |
| Gradient clipping | max_norm=1.0 | Prevents gradient explosions in deep architectures during early training |
| Checkpoint criterion | Lowest validation loss | Consistent across all architectures for fair comparison |
| Early stopping | Patience=15 epochs | Halts training when val loss does not improve, prevents overfitting |

### 3.3 Evaluation Protocol

Every architecture is evaluated using four complementary metrics:

**ROC curve and macro AUC** — primary metric. AUC is threshold-independent
and robust to class balance. Macro AUC averages the one-vs-rest AUC across
all three classes equally.

**Confusion matrix** — reveals the specific direction of misclassifications.
Within this experimental setup, Sphere→No Substructure confusion is tracked
separately from Sphere→Vortex, as the two error types have different
implications for substructure detection.

**Precision-Recall curve (Sphere class)** — Sphere shows the lowest
per-class AUC across all evaluated architectures (Section 6). A dedicated
PR curve isolates its detection performance independently of the other
two classes.

**Calibration curve** — tests whether the model's confidence scores are
reliable. A well-calibrated model that outputs 80% confidence should be
correct 80% of the time. Poor calibration undermines the use of confidence
thresholds in deployment.

In [ ]:
def count_parameters(model):
    """Return total and trainable parameter counts."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def evaluate_on_loader(model, loader):
    """Run a full evaluation pass; return labels, probs, preds."""
    model.eval()
    all_labels, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs  = F.softmax(logits, dim=1)
            preds  = logits.argmax(1)
            all_labels.extend(lbls.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    return (np.array(all_labels),
            np.array(all_probs),
            np.array(all_preds))


def plot_results(model_name, labels, probs, preds, save_dir=None):
    """
    Produces a 2×2 figure:
      [ROC curves]       [Confusion matrix]
      [PR curve-Sphere]  [Calibration curve]
    """
    y_bin = label_binarize(labels, classes=[0, 1, 2])
    n_cls = 3

    # ── ROC ───────────────────────────────────────────────────────────────────
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_cls):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    all_fpr   = np.unique(np.concatenate([fpr[i] for i in range(n_cls)]))
    mean_tpr  = sum(np.interp(all_fpr, fpr[i], tpr[i]) for i in range(n_cls)) / n_cls
    macro_auc = auc(all_fpr, mean_tpr)

    # ── PR (Sphere class only) ────────────────────────────────────────────────
    sphere_prec, sphere_rec, _ = precision_recall_curve(y_bin[:, 1], probs[:, 1])
    sphere_pr_auc = average_precision_score(y_bin[:, 1], probs[:, 1])

    # ── Calibration ───────────────────────────────────────────────────────────
    # Use max-probability confidence as the confidence estimate
    max_probs  = probs.max(axis=1)
    is_correct = (preds == labels).astype(int)
    cal_frac_pos, cal_mean_pred = calibration_curve(is_correct, max_probs, n_bins=10)

    fig = plt.figure(figsize=(16, 12))
    gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.35)

    # ROC
    ax1 = fig.add_subplot(gs[0, 0])
    colors = ['aqua', 'darkorange', 'cornflowerblue']
    for i, (color, cname) in enumerate(zip(colors, CLASS_NAMES)):
        ax1.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'{cname} (AUC={roc_auc[i]:.3f})')
    ax1.plot(all_fpr, mean_tpr, 'r:', lw=3,
             label=f'Macro-Avg (AUC={macro_auc:.3f})')
    ax1.plot([0,1],[0,1],'k--', lw=1.5)
    ax1.set(xlabel='FPR', ylabel='TPR', title=f'ROC — {model_name}')
    ax1.legend(fontsize=9)

    # Confusion matrix
    ax2 = fig.add_subplot(gs[0, 1])
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2)
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('True')
    ax2.set_title(f'Confusion Matrix — {model_name}')

    # PR — Sphere
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(sphere_rec, sphere_prec, color='darkorange', lw=2,
             label=f'Sphere PR-AUC={sphere_pr_auc:.3f}')
    ax3.axhline(y=y_bin[:,1].mean(), color='k', linestyle='--', lw=1.2,
                label='Random baseline')
    ax3.set(xlabel='Recall', ylabel='Precision',
            title=f'Precision-Recall (Sphere) — {model_name}')
    ax3.legend(fontsize=9)

    # Calibration
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.plot(cal_mean_pred, cal_frac_pos, 's-', color='steelblue', lw=2,
             label='Model')
    ax4.plot([0,1],[0,1],'k--', lw=1.5, label='Perfect calibration')
    ax4.set(xlabel='Mean predicted confidence', ylabel='Fraction correct',
            title=f'Calibration Curve — {model_name}')
    ax4.legend(fontsize=9)

    plt.suptitle(model_name, fontsize=14, fontweight='bold')

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        fig.savefig(os.path.join(save_dir, f'{model_name}_results.png'),
                    dpi=120, bbox_inches='tight')
    plt.show()

    return {
        'macro_auc':     macro_auc,
        'per_class_auc': roc_auc,
        'sphere_pr_auc': sphere_pr_auc,
        'fpr': all_fpr, 'tpr': mean_tpr,
    }


def print_per_class_report(labels, preds, model_name):
    print(f'\n── Classification Report: {model_name} ──────────────────────')
    print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))


### 3.4 Training Loop

All architectures share a single training function. Training runs for a
maximum of 50 epochs with early stopping at patience=15. The best checkpoint
— defined as the epoch with lowest validation loss — is saved to Drive and
reloaded for final evaluation. Training curves and all four evaluation plots
are produced automatically at the end of each run.

In [ ]:
def train_and_evaluate(model, model_name,
                       epochs=50, lr=1e-4,
                       patience=15,
                       pretrained=True):
    """
    Full pipeline:
      1. Train with early stopping on validation loss.
      2. Reload best checkpoint.
      3. Evaluate on the held-out test set.
      4. Report all metrics.
    """
    print(f'\n{"="*60}')
    print(f'  {model_name}')
    total_p, trainable_p = count_parameters(model)
    print(f'  Parameters: {total_p/1e6:.2f}M total  |  {trainable_p/1e6:.2f}M trainable')
    print(f'  Pretrained: {pretrained}')
    print(f'  LR: {lr}  |  Max epochs: {epochs}  |  Patience: {patience}')
    print(f'={"="*60}')

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    best_val_loss  = float('inf')
    best_weights   = None
    patience_count = 0
    history        = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        # ── Training ──────────────────────────────────────────────────────────
        model.train()
        train_loss, train_correct = 0.0, 0

        for imgs, lbls in tqdm(train_loader,
                               desc=f'Epoch {epoch}/{epochs}',
                               leave=False):
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss    += loss.item()
            train_correct += (logits.argmax(1) == lbls).sum().item()

        scheduler.step()

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        val_loss, val_correct = 0.0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                logits  = model(imgs)
                val_loss    += criterion(logits, lbls).item()
                val_correct += (logits.argmax(1) == lbls).sum().item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss   = val_loss   / len(val_loader)
        val_acc        = val_correct / len(val_ds)

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)

        print(f'Epoch {epoch:3d} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Val Loss: {avg_val_loss:.4f} | '
              f'Val Acc: {val_acc:.4f}')

        # ── Early stopping ────────────────────────────────────────────────────
        if avg_val_loss < best_val_loss:
            best_val_loss  = avg_val_loss
            best_weights   = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f'\nEarly stopping at epoch {epoch} '
                      f'(no improvement in val loss for {patience} epochs).')
                break

    # ── Save best weights ─────────────────────────────────────────────────────
    save_path = os.path.join(gsoc_drive_path, f'{model_name}_best.pth')
    torch.save(best_weights, save_path)
    print(f'\nBest weights saved → {save_path}')

    # ── Plot training curves ──────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs_ran = len(history['train_loss'])
    ax1.plot(range(1, epochs_ran+1), history['train_loss'], label='Train')
    ax1.plot(range(1, epochs_ran+1), history['val_loss'],   label='Val')
    ax1.set(xlabel='Epoch', ylabel='Loss',
            title=f'{model_name} — Loss Curves')
    ax1.legend()
    ax2.plot(range(1, epochs_ran+1), history['val_acc'])
    ax2.set(xlabel='Epoch', ylabel='Val Accuracy',
            title=f'{model_name} — Validation Accuracy')
    plt.tight_layout(); plt.show()

    # ── Final evaluation on HELD-OUT TEST SET ─────────────────────────────────
    print('\nEvaluating best checkpoint on held-out test set…')
    model.load_state_dict(best_weights)
    labels, probs, preds = evaluate_on_loader(model, val_loader)

    print_per_class_report(labels, preds, model_name)

    fig_dir = os.path.join(gsoc_drive_path, 'figures')
    metrics = plot_results(model_name, labels, probs, preds, save_dir=fig_dir)

    test_acc = (preds == labels).mean()
    sphere_recall = (
        (preds[labels == 1] == 1).sum() / (labels == 1).sum()
    )

    ARCH_METRICS[model_name] = {
        **metrics,
        'test_accuracy':  test_acc,
        'sphere_recall':  float(sphere_recall),
        'total_params_M': total_p / 1e6,
        'pretrained':     pretrained,
        'labels': labels, 'probs': probs, 'preds': preds,
    }

    print(f'\n  Test Accuracy : {test_acc:.4f}')
    print(f'  Macro AUC     : {metrics["macro_auc"]:.4f}')
    print(f'  Sphere Recall : {float(sphere_recall):.4f}')

    return model


### 3.5 Reproducibility — Load or Train

To avoid retraining on runtime reconnection, a `load_or_train` wrapper checks
whether saved weights exist in Drive before initiating training. If weights
are found, the model is loaded directly and evaluated. If not, the full
training pipeline runs and weights are saved. All results in this notebook
were produced with SEED=42.

In [ ]:
def load_or_train(model, model_name, epochs=50, lr=1e-4, pretrained=True):
    """
    If saved weights exist in Drive, load them and evaluate directly.
    If not, train from scratch and save.
    """
    save_path = os.path.join(gsoc_drive_path, f'{model_name}_best.pth')

    if os.path.exists(save_path):
        print(f'\n[SKIP] {model_name} — weights found at {save_path}')
        print('Loading saved weights and evaluating...')
        model = model.to(device)
        model.load_state_dict(torch.load(save_path, map_location=device))

        labels, probs, preds = evaluate_on_loader(model, val_loader)
        print_per_class_report(labels, preds, model_name)
        fig_dir = os.path.join(gsoc_drive_path, 'figures')
        metrics = plot_results(model_name, labels, probs, preds, save_dir=fig_dir)

        test_acc = (preds == labels).mean()
        sphere_recall = (preds[labels == 1] == 1).sum() / (labels == 1).sum()

        ARCH_METRICS[model_name] = {
            **metrics,
            'test_accuracy':  test_acc,
            'sphere_recall':  float(sphere_recall),
            'total_params_M': sum(p.numel() for p in model.parameters()) / 1e6,
            'pretrained':     pretrained,
            'labels': labels, 'probs': probs, 'preds': preds,
        }
        print(f'  Test Accuracy : {test_acc:.4f}')
        print(f'  Macro AUC     : {metrics["macro_auc"]:.4f}')
        return model
    else:
        return train_and_evaluate(model, model_name, epochs=epochs,
                                  lr=lr, pretrained=pretrained)

## <a id="4-architecture-evaluations"></a>4. Architecture Evaluations

Eleven architectures are evaluated on the same dataset, training protocol, and
evaluation metrics. The ordering follows an intentional narrative — from
sequential baselines that establish the lower bound, through increasingly
powerful general architectures, to physics-motivated equivariant networks
that encode the rotational symmetry of gravitational lensing directly into
their weight structure.

All models receive single-channel 150×150 input. Pretrained ImageNet weights
are used where available, with the first convolutional layer and final
classification head replaced to match the input and output dimensions of this
task. Equivariant architectures are trained from scratch — no ImageNet
pretraining exists for group-equivariant networks.

### 4.1 AlexNet

AlexNet (Krizhevsky et al., 2012) serves as a low-complexity convolutional baseline. Its architecture is purely sequential and lacks residual pathways, explicit multi-scale fusion, and attention mechanisms, so it is less well suited than newer models for capturing subtle local perturbations on top of a dominant shared lens morphology. It is included as a reference point, not because it is guaranteed to be the worst model, but because it represents a simpler and older design family.

To adapt it to this dataset, the first convolutional layer is replaced with a single-channel version initialized by averaging the pretrained RGB filters, and the final classifier is replaced with a 3-way output layer.

In [ ]:
alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
w = alexnet.features[0].weight.data.clone()
alexnet.features[0] = nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2)
alexnet.features[0].weight.data = w.mean(dim=1, keepdim=True)
alexnet.classifier[6] = nn.Linear(alexnet.classifier[6].in_features, 3)

# Lower LR: AlexNet is an older architecture sensitive to large updates
alexnet = load_or_train(alexnet, 'AlexNet', epochs=50, lr=1e-5, pretrained=True)


### 4.2 VGG-16

VGG-16 provides a deeper sequential CNN baseline than AlexNet. It increases representational capacity through stacked
3×3 convolutions, but still relies on a purely feed-forward architecture without skip connections or attention. This makes it useful as a comparison against later architectures that are explicitly designed to preserve gradient flow and reuse intermediate features.

The model is adapted to grayscale input by replacing the first convolution with a single-channel version initialized from the mean of the pretrained RGB filters, and the classifier head is replaced with a 3-class output layer. Since standard VGG-16 does not include batch normalization, its optimization behavior is generally less robust than that of modern residual networks.

In [ ]:
vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
w = vgg16.features[0].weight.data.clone()
vgg16.features[0] = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1)
vgg16.features[0].weight.data = w.mean(dim=1, keepdim=True)
vgg16.classifier[6] = nn.Linear(vgg16.classifier[6].in_features, 3)

vgg16 = load_or_train(vgg16, 'VGG-16', epochs=50, lr=1e-5, pretrained=True)

### 4.3 ResNet-18

**Role:** First residual baseline in the benchmark.

ResNet-18 is the first architecture in this study to introduce **identity skip connections**, allowing features and gradients to propagate through shortcut paths in addition to the main convolutional path. Relative to purely sequential models such as AlexNet and VGG-16, this makes optimization more stable and improves feature reuse across depth. For this task, that is a plausible architectural advantage because the discriminative signal is a small perturbation superimposed on a dominant shared lens morphology.

This does **not** by itself prove that ResNet-18 will outperform the sequential baselines, but it makes it a more appropriate modern reference architecture for testing whether residual learning improves classification of subtle substructure patterns.

**Modification:** The first convolutional layer is replaced to accept single-channel input, and the final fully connected layer is replaced with a 3-class output head. The new first-layer weights are initialized by averaging the pretrained RGB filters across channels rather than summing them, which preserves the approximate activation scale expected by the pretrained network.

In [ ]:
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# ── Correct channel adaptation: .mean() not .sum() ────────────────────────────
# .mean() preserves the expected activation scale.
# .sum() would triple the first-layer output variance, destabilising BN.
w = resnet18.conv1.weight.data.clone()
resnet18.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet18.conv1.weight.data = w.mean(dim=1, keepdim=True)
resnet18.fc = nn.Linear(resnet18.fc.in_features, 3)

resnet18 = load_or_train(resnet18, 'ResNet-18', epochs=50, lr=1e-4)

### 4.4 ResNet-50


**Role:** Higher-capacity residual architecture within the ResNet family.

ResNet-50 extends the residual-learning framework of ResNet-18 by using deeper **bottleneck blocks** of the form $(1\times1 \rightarrow 3\times3 \rightarrow 1\times1)$. These blocks increase representational capacity while controlling parameter growth through channel compression and expansion around the central spatial convolution. Relative to ResNet-18, ResNet-50 therefore provides a useful comparison point for asking whether a substantially deeper and more expressive residual model improves classification performance on subtle lensing perturbations.

This comparison should not be interpreted as testing depth alone, since ResNet-50 differs from ResNet-18 not only in depth and width but also in block design. However, both models belong to the same residual architecture family, so the pair offers a reasonable within-family capacity comparison.

**Modification:** As in ResNet-18, the first convolutional layer is replaced to accept single-channel input using the mean of the pretrained RGB filters, and the final fully connected layer is replaced with a 3-class output head.

In [ ]:
resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
w = resnet50.conv1.weight.data.clone()
resnet50.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet50.conv1.weight.data = w.mean(dim=1, keepdim=True)
resnet50.fc = nn.Linear(resnet50.fc.in_features, 3)


resnet50 = load_or_train(resnet50, 'ResNet-50', epochs=50, lr=1e-4)


### 4.5 DenseNet-121

**Role:** Dense connectivity baseline with strong feature reuse.

DenseNet-121 extends the idea of skip connectivity by **concatenating** feature maps from all preceding layers within a dense block, rather than combining them through elementwise addition as in ResNet. This design encourages strong feature reuse and preserves access to earlier representations throughout the network depth.

For this task, that is a plausible advantage because the discriminative signal is expected to consist of subtle local perturbations superimposed on a dominant shared lens morphology. An architecture that retains access to both earlier low-level structure and later higher-level representations may therefore be well suited to this setting. This should still be treated as an architectural hypothesis rather than a guaranteed advantage.

**Modification:** The first convolutional layer is replaced to accept single-channel input using the mean of the pretrained RGB filters, and the final classifier layer is replaced with a 3-class linear head.

In [ ]:
densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
w = densenet.features.conv0.weight.data.clone()
densenet.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
densenet.features.conv0.weight.data = w.mean(dim=1, keepdim=True)
densenet.classifier = nn.Linear(densenet.classifier.in_features, 3)
densenet = load_or_train(densenet, 'DenseNet-121', epochs=50, lr=1e-4, pretrained=True)

### 4.6 EfficientNet-B3

**Role:** Parameter-efficient modern CNN baseline.

EfficientNet-B3 is included as a modern convolutional architecture designed to balance accuracy and efficiency through coordinated scaling of network depth, width, and resolution. Its core MBConv blocks use depthwise-separable convolutions and squeeze-and-excitation mechanisms, allowing relatively strong representational capacity at lower parameter cost than many conventional CNNs.

For this task, EfficientNet-B3 is of particular interest because it combines modern architectural efficiency with fairly aggressive early spatial reduction. That creates a task-specific question: whether the network can preserve the subtle small-scale perturbation patterns relevant to substructure classification, or whether some of that information is attenuated too early in the feature hierarchy. This is a reasonable concern, but it should be treated as a hypothesis to be tested rather than assumed in advance.

**Modification:** The stem convolution is replaced to accept single-channel input by averaging the pretrained RGB filters across channels, and the final classifier layer is replaced with a 3-class output head.

In [ ]:
effnet = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
w = effnet.features[0][0].weight.data.clone()
new_stem = nn.Conv2d(1, 40, kernel_size=3, stride=2, padding=1, bias=False)
new_stem.weight.data = w.mean(dim=1, keepdim=True)
effnet.features[0][0] = new_stem
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 3)
effnet = load_or_train(effnet, 'EfficientNet-B3', epochs=50, lr=1e-4, pretrained=True)

### 4.7 ViT-Base

**Role:** Transformer-based global-context baseline.

Vision Transformer (ViT-Base) represents the image as a sequence of fixed-size patches and applies self-attention across the full patch set. Unlike convolutional networks, which build locality and translation-equivariant structure into the architecture, ViT relies more heavily on learned representations and pretrained priors to organize spatial information.

This makes ViT an interesting comparison point for the present task. On one hand, global attention may help relate distant parts of the Einstein ring and capture large-scale structural context. On the other hand, the class signal is expected to reside in subtle local perturbations, and transformer architectures generally rely more strongly on data scale and pretraining quality than CNNs. In this setting, the main question is therefore whether the global-context advantage outweighs the weaker built-in locality bias.

**Modification:** The patch-projection layer is replaced to accept single-channel input by averaging the pretrained RGB filters across channels, and the classification head is replaced with a 3-class output layer. Because the pretrained ViT expects $(224\times224)$ input, the $(150\times150)$ grayscale images are upsampled in the forward pass before patch embedding. This preserves compatibility with the pretrained weights, but it also introduces an additional interpolation step that may smooth fine-scale structure, so the ViT results should be interpreted with that caveat.

In [ ]:
class ViTWrapper(nn.Module):
    """
    ViT-Base adapted for single-channel 150x150 input.

    The pretrained ViT-Base requires 224x224 input for its 16x16 patch embeddings.
    Rather than modifying the patch embedding kernel (which destroys pretrained
    spatial frequency structure), images are bicubically upsampled to 224x224
    in the forward pass. This preserves all pretrained weights intact.

    """
    def __init__(self, n_classes=3):
        super().__init__()
        self.vit = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

        # ── 1-channel patch embedding ──────────────────────────────────────────
        old_weight = self.vit.conv_proj.weight.data        # [768, 3, 16, 16]
        self.vit.conv_proj = nn.Conv2d(1, 768, kernel_size=16, stride=16)
        self.vit.conv_proj.weight.data = old_weight.mean(dim=1, keepdim=True)

        # ── Classification head ────────────────────────────────────────────────
        self.vit.heads.head = nn.Linear(self.vit.heads.head.in_features, n_classes)

    def forward(self, x):
        # Upsample to 224x224 — preserves pretrained patch embedding structure
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        return self.vit(x)


vit_model = ViTWrapper()
vit_model = load_or_train(vit_model, 'ViT-Base', epochs=50, lr=1e-5, pretrained=True)

### 4.8 Equivariant Neural Network ($D_4$ Symmetry)

**Role:** Physics-motivated symmetry-aware architecture.

The preceding architectures treat rotational and reflection symmetry as properties that must be learned from augmented data. In contrast, this model builds discrete symmetry directly into the network through **group-equivariant convolutions** implemented with the `e2cnn` library. The relevant symmetry group is the dihedral group $D_4$, consisting of the eight symmetries of the square: the identity, rotations by $90^\circ$, $180^\circ$, and $270^\circ$, together with reflections.

Under this construction, if the input is transformed by an element of $D_4$, the intermediate feature representation transforms according to the corresponding group action by design. This is the key distinction from ordinary CNNs with augmentation: the symmetry constraint is encoded directly in the weight-sharing structure rather than learned only empirically from transformed examples.

The physical motivation is clear. A gravitational lens does not have a preferred orientation on the sky, so encoding discrete rotational and reflection symmetry is well aligned with the structure of the problem. At the same time, the scope of the symmetry prior should be stated precisely: $D_4$ covers only the eight discrete symmetries of the square, not continuous rotational equivariance.

This model is trained from scratch rather than initialized from ImageNet-pretrained weights, so its performance reflects the value of the symmetry prior without transfer from natural-image pretraining.

In [ ]:
import torch.nn.functional as F
from escnn import gspaces
from escnn import nn as enn

class EquivariantLensNet(nn.Module):
    """
    Shallow D₄-equivariant network. From-scratch baseline for Test I.
    Increased width: hid1=24, hid2=48, hid3=64 — third block added.
    Original 0.05M was insufficient capacity for 3-class 150×150 task.
    """
    def __init__(self, n_classes=3):
        super().__init__()
        self.r2_act = gspaces.flipRot2dOnR2(N=4)

        in_type  = enn.FieldType(self.r2_act,  1 * [self.r2_act.trivial_repr])
        hid1     = enn.FieldType(self.r2_act, 24 * [self.r2_act.regular_repr])
        hid2     = enn.FieldType(self.r2_act, 48 * [self.r2_act.regular_repr])
        hid3     = enn.FieldType(self.r2_act, 64 * [self.r2_act.regular_repr])

        self.input_type = in_type

        # Pad 150→152 before equivariant chain (even dims required)
        self.block1 = enn.SequentialModule(
            enn.R2Conv(in_type, hid1, kernel_size=5, padding=2, bias=False),
            enn.InnerBatchNorm(hid1),
            enn.ReLU(hid1, inplace=False),
            enn.PointwiseAvgPool(hid1, kernel_size=2, stride=2)   # 152→76
        )
        self.block2 = enn.SequentialModule(
            enn.R2Conv(hid1, hid2, kernel_size=5, padding=2, bias=False),
            enn.InnerBatchNorm(hid2),
            enn.ReLU(hid2, inplace=False),
            enn.PointwiseAvgPool(hid2, kernel_size=2, stride=2)   # 76→38
        )
        self.block3 = enn.SequentialModule(
            enn.R2Conv(hid2, hid3, kernel_size=3, padding=1, bias=False),
            enn.InnerBatchNorm(hid3),
            enn.ReLU(hid3, inplace=False),
            enn.PointwiseAvgPool(hid3, kernel_size=2, stride=2)   # 38→19
        )

        # 19×19 — spatial_pool kernel=19
        self.spatial_pool = enn.PointwiseAvgPool(hid3, kernel_size=19)
        self.gpool        = enn.GroupPooling(hid3)

        # After GroupPooling: 64 invariant scalars
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.3),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = F.pad(x, (1, 1, 1, 1))   # 150→152
        x = enn.GeometricTensor(x, self.input_type)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.spatial_pool(x)
        x = self.gpool(x)
        return self.fc(x.tensor)


# ── Verify ────────────────────────────────────────────────────────────────────
_m = EquivariantLensNet().to(device)
_m.eval()

total_p = sum(p.numel() for p in _m.parameters())
print(f'Total params: {total_p/1e6:.3f}M')

with torch.no_grad():
    _x  = torch.zeros(1, 1, 150, 150).to(device)
    _out = _m(_x)
    print(f'Output shape : {_out.shape}')

print('\nD₄ invariance verification:')
torch.manual_seed(0)
_x_t    = torch.randn(1, 1, 150, 150).to(device)
max_err = 0.0
with torch.no_grad():
    _out0 = _m(_x_t)
    for g_el in _m.input_type.gspace.fibergroup.elements:
        _x_tfm = _m.input_type.transform(_x_t.cpu(), g_el).to(device)
        _outt  = _m(_x_tfm)
        _div   = (_out0 - _outt).abs().max().item()
        max_err = max(max_err, _div)
        print(f'  {str(g_el):<30} |Δlogit| = {_div:.2e}')
print(f'\n  Max |Δlogit| : {max_err:.2e}  '
      f'{"✓" if max_err < 1e-5 else "✗"}')
del _m, _x_t

enn_model = EquivariantLensNet()
enn_model = load_or_train(
    enn_model, 'Equivariant-D4',
    epochs=50, lr=1e-3, pretrained=False
)

### 4.9 Equivariant Residual Network (E-ResNet)

**Role:** Combines discrete symmetry encoding with residual learning.

E-ResNet integrates $D_4$-equivariant convolutions with residual skip connections. Architecturally, it combines two ideas explored separately above: explicit encoding of discrete rotational and reflection symmetry, and residual pathways that improve optimization in deeper networks.

The symmetry prior is physically well motivated. Strong gravitational lenses do not have a preferred orientation on the sky, so it is reasonable to encode invariance to the eight elements of the dihedral group $D_4$ directly into the network rather than relying only on data augmentation to encourage it empirically. At the same time, the residual structure makes the model easier to optimize than a purely sequential equivariant network, especially when trained from scratch.

This architecture should be interpreted as a compact, symmetry-aware alternative to the larger pretrained CNNs in the benchmark. Because no standard ImageNet-pretrained weights exist for this model class, its performance reflects the value of the architectural inductive bias and residual design without transfer from natural-image pretraining.

**Implementation note:** The model uses carefully chosen padding and strided operations to preserve spatial consistency under the discrete group action as well as possible on the pixel grid. This matters because equivariance in practice depends not only on the abstract group construction but also on how the image lattice is sampled and downsampled.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 4.9 — Equivariant Residual Network (E-ResNet)
#
# Combines D₄ equivariant convolutions with residual skip connections.
# Stem padding=3 ensures even spatial dimensions at every strided operation:
#   150×150 → 76×76 (stem)  → 38×38 (layer1)  → 19×19 (layer2)
# ─────────────────────────────────────────────────────────────────────────────

class EResBlock(nn.Module):
    """
    D₄-equivariant residual block with optional strided shortcut.

    The identity shortcut uses enn.SequentialModule with InnerBatchNorm
    on the projection conv — this is required to match saved checkpoint
    keys: layer{n}.downsample.0 (R2Conv) and layer{n}.downsample.1 (BN).
    Using a bare R2Conv shortcut without BN produces a key mismatch
    when loading weights trained with this definition.
    """
    def __init__(self, in_type, out_type, stride=1):
        super().__init__()
        self.conv1      = enn.R2Conv(in_type,  out_type, kernel_size=3,
                                     padding=1, stride=stride, bias=False)
        self.bn1        = enn.InnerBatchNorm(out_type)
        self.relu1      = enn.ReLU(out_type, inplace=True)
        self.conv2      = enn.R2Conv(out_type, out_type, kernel_size=3,
                                     padding=1, bias=False)
        self.bn2        = enn.InnerBatchNorm(out_type)
        self.relu2      = enn.ReLU(out_type, inplace=True)
        self.downsample = None
        if stride != 1 or in_type != out_type:
            self.downsample = enn.SequentialModule(
                enn.R2Conv(in_type, out_type, kernel_size=1,
                           stride=stride, bias=False),
                enn.InnerBatchNorm(out_type)
            )

    def forward(self, x):
        identity = x
        out      = self.relu1(self.bn1(self.conv1(x)))
        out      = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu2(out + identity)


class EquivariantResNet(nn.Module):
    """
    D₄-equivariant residual network trained from scratch.

    Physical motivation: gravitational lenses have no preferred orientation.
    D₄ equivariance (rotations by 90°/180°/270° + reflections) is encoded
    directly into the weight structure via escnn, guaranteeing identical
    predictions for all 8 symmetry group elements by construction rather
    than relying on augmentation to learn approximate invariance.

    Architecture:
        stem   → 76×76   (kernel=5, padding=3, stride=2)  ← even ✓
        layer1 → 38×38   (EResBlock, stride=2)             ← even ✓
        layer2 → 19×19   (EResBlock, stride=2)
        gpool  → group pooling (GeometricTensor → standard tensor)
        fc     → AdaptiveAvgPool2d(1) → Linear(64, 3)

    Parameters: ~0.39M — trained from scratch (no ImageNet pretraining
    exists for group-equivariant architectures).

    NOTE: stem padding=3 is required for even spatial dimensions.
    padding=2 produces 75×75 (odd), breaking D₄ invariance at the
    pixel level due to stride-2 coordinate misalignment after rotation.
    """
    def __init__(self, n_classes=3):
        super().__init__()
        act = gspaces.flipRot2dOnR2(N=4)

        self.in_type = enn.FieldType(act, 1  * [act.trivial_repr])
        t16          = enn.FieldType(act, 16 * [act.regular_repr])
        t32          = enn.FieldType(act, 32 * [act.regular_repr])
        t64          = enn.FieldType(act, 64 * [act.regular_repr])

        self.stem   = enn.SequentialModule(
            enn.R2Conv(self.in_type, t16,
                       kernel_size=5,
                       padding=3,        # ← padding=3 required (not 2)
                       stride=2, bias=False),
            enn.InnerBatchNorm(t16),
            enn.ReLU(t16, inplace=True)
        )
        self.layer1 = EResBlock(t16, t32, stride=2)
        self.layer2 = EResBlock(t32, t64, stride=2)
        self.gpool  = enn.GroupPooling(t64)
        self.fc     = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = enn.GeometricTensor(x, self.in_type)
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.gpool(x)
        return self.fc(x.tensor)


# ── Verify spatial dimensions before loading/training ─────────────────────────
_m = EquivariantResNet().to(device)
_x = torch.zeros(1, 1, 150, 150).to(device)
with torch.no_grad():
    _g  = enn.GeometricTensor(_x, _m.in_type)
    _s  = _m.stem(_g)
    h1  = _s.tensor.shape[2]
    _l1 = _m.layer1(_s)
    h2  = _l1.tensor.shape[2]
print(f'E-ResNet spatial trace: 150 → {h1} → {h2} → ...'
      f'  {"✓ all even" if h1%2==0 and h2%2==0 else "✗ odd dimension — check padding"}')
del _m, _x, _g, _s, _l1

# ── Train or load ─────────────────────────────────────────────────────────────
e_resnet = EquivariantResNet()
e_resnet = load_or_train(
    e_resnet, 'E-ResNet',
    epochs=60, lr=1e-3, pretrained=False
)

### 4.10 Equivariant DenseNet ($C_8$)

**Role:** Tests a different symmetry-aware connectivity pattern within the equivariant setting.

Whereas E-ResNet combines $D_4$-equivariant convolutions with residual skip connections, this model combines **$C_8$ rotational equivariance** with **DenseNet-style dense connectivity**. The purpose of this comparison is not to isolate a single variable cleanly, since both the symmetry group and the connectivity pattern change, but to examine whether a denser feature-reuse architecture paired with finer discrete rotational structure is effective for this task.

The group $C_8$ represents rotations at $45^\circ$ increments without reflections. Relative to $D_4$, this provides finer angular coverage of orientation while imposing a different symmetry prior: rotational equivariance only, rather than rotational-plus-reflection equivariance. That distinction matters physically, because the relevance of reflection symmetry is less clear than the relevance of orientation symmetry itself.

Architecturally, the model replaces residual addition with dense concatenative connectivity, so each layer within a dense block receives the accumulated feature maps from all preceding layers. This encourages strong feature reuse and preservation of earlier representations across depth. In the present setting, that may be beneficial if substructure cues depend on subtle local deviations that can otherwise be attenuated in deeper feature hierarchies.

The network is built entirely from $C_8$-equivariant convolutions and is trained from scratch, since standard ImageNet-pretrained weights are not available for this model class. After the final equivariant feature extraction stage, group pooling is applied to obtain rotation-invariant features before classification.

**Implementation note:** Because this model uses rotations only, without reflections, its practical equivariance behavior is less constrained by parity-related spatial alignment issues than the corresponding $D_4$ case. Even so, the empirical behavior of equivariant models still depends on discretization and downsampling choices, so the symmetry prior should be treated as structurally enforced but not numerically idealized.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 4.10 — Equivariant DenseNet (C8 — 8-fold discrete rotation)
#
# Motivation: E-ResNet uses D₄ (8 discrete symmetries of the square).
# Real gravitational lenses appear at arbitrary position angles —
# C8 (rotations by 45° increments) provides finer angular coverage
# while remaining tractable to train from scratch.
#
# Architecture: DenseNet-style dense connectivity built entirely from
# C8-equivariant R2Conv layers. Each dense layer receives concatenated
# feature maps from all preceding layers within the block.
#
# Spatial dimensions (150×150 input, stem stride=2, padding=2):
#   150×150 → 75×75 (stem)
# Note: C8 uses rot2dOnR2(N=8) — no reflections, so odd spatial
# dimensions do not break the equivariance guarantee (unlike D4).
#
# Group: C8 = cyclic group of order 8 (rotations only, no reflections)
# Parameters: 0.093M (trained from scratch, weights saved to Drive)
# ─────────────────────────────────────────────────────────────────────────────

import torch.nn.functional as F
from escnn import gspaces
from escnn import nn as enn

# ── Dense Layer ───────────────────────────────────────────────────────────────

class EqDenseLayer(nn.Module):
    """
    Single BN→ReLU→Conv layer inside an equivariant dense block.
    Pre-activation order (BN→ReLU→Conv) matches original DenseNet paper.
    """
    def __init__(self, gspace, in_type, growth_rate):
        super().__init__()
        self.in_type  = in_type
        self.out_type = enn.FieldType(gspace, growth_rate * [gspace.regular_repr])
        self.block    = enn.SequentialModule(
            enn.InnerBatchNorm(self.in_type),
            enn.ReLU(self.in_type),
            enn.R2Conv(self.in_type, self.out_type,
                       kernel_size=3, padding=1, bias=False)
        )

    def forward(self, x):
        return self.block(x)


# ── Dense Block ───────────────────────────────────────────────────────────────

class EqDenseBlock(nn.Module):
    """
    Equivariant dense block: each layer receives concatenated outputs
    of all preceding layers. Field types are accumulated by direct sum
    (concatenation of representations).
    """
    def __init__(self, gspace, in_type, num_layers, growth_rate):
        super().__init__()
        self.gspace       = gspace
        self.dense_layers = nn.ModuleList()
        self.in_types     = []

        current_type = in_type
        for _ in range(num_layers):
            layer = EqDenseLayer(gspace, current_type, growth_rate)
            self.dense_layers.append(layer)
            self.in_types.append(current_type)
            current_type = enn.FieldType(
                gspace,
                current_type.representations + layer.out_type.representations
            )

        self.out_type = current_type

    def forward(self, x):
        features = [x]

        for i, layer in enumerate(self.dense_layers):
            if len(features) == 1:
                cat = features[0]
            else:
                cat_tensor = torch.cat([f.tensor for f in features], dim=1)
                cat = enn.GeometricTensor(cat_tensor, self.in_types[i])
            features.append(layer(cat))

        cat_tensor = torch.cat([f.tensor for f in features], dim=1)
        return enn.GeometricTensor(cat_tensor, self.out_type)


# ── Transition Layer ──────────────────────────────────────────────────────────

class EqTransition(nn.Module):
    """
    BN→ReLU→1×1 Conv (compression) → 2×2 AvgPool.
    Reduces feature map count by compression factor and halves spatial dims.
    Spatial avg pool is equivariant: commutes with group action.
    """
    def __init__(self, gspace, in_type, compression=0.5):
        super().__init__()
        n_out         = max(1, int(len(in_type.representations) * compression))
        self.out_type = enn.FieldType(gspace, n_out * [gspace.regular_repr])
        self.conv     = enn.SequentialModule(
            enn.InnerBatchNorm(in_type),
            enn.ReLU(in_type),
            enn.R2Conv(in_type, self.out_type, kernel_size=1, bias=False)
        )

    def forward(self, x):
        x = self.conv(x)
        pooled = F.avg_pool2d(x.tensor, kernel_size=2, stride=2)
        return enn.GeometricTensor(pooled, self.out_type)


# ── Full Model ────────────────────────────────────────────────────────────────

class EquivariantDenseNet(nn.Module):
    """
    C8-equivariant DenseNet trained from scratch on single-channel input.

    C8 (rot2dOnR2(N=8)) encodes rotational symmetry at 45° increments —
    finer coverage than D₄'s 90° increments, without reflections.

    Architecture:
        Stem      : R2Conv 5×5, stride=2, padding=2  → 75×75
        DenseBlock 1 (num_layers=4) + Transition      → ~37×37
        DenseBlock 2 (num_layers=4) + Transition      → ~18×18
        DenseBlock 3 (num_layers=4)
        InnerBN → ReLU → GroupPool → AdaptiveAvgPool(1) → Linear(3)

    GroupPooling collapses the C8 group dimension (max over 8 orientations),
    producing rotation-invariant features before the classifier.
    """
    def __init__(self, num_classes=3, growth_rate=6,
                 block_config=(4, 4, 4), init_features=8, N=8):
        super().__init__()
        gspace      = gspaces.rot2dOnR2(N=N)
        self.gspace = gspace

        in_type      = enn.FieldType(gspace, [gspace.trivial_repr])
        stem_type    = enn.FieldType(gspace, init_features * [gspace.regular_repr])
        self.in_type = in_type

        # Stem: R2Conv + BN + ReLU, stride=2
        self.stem = enn.SequentialModule(
            enn.R2Conv(in_type, stem_type,
                       kernel_size=5, padding=2, stride=2, bias=False),
            enn.InnerBatchNorm(stem_type),
            enn.ReLU(stem_type)
        )

        # Dense blocks + transitions
        self.blocks      = nn.ModuleList()
        self.transitions = nn.ModuleList()

        current_type = stem_type
        for i, num_layers in enumerate(block_config):
            block = EqDenseBlock(gspace, current_type, num_layers, growth_rate)
            self.blocks.append(block)
            current_type = block.out_type

            if i < len(block_config) - 1:
                trans = EqTransition(gspace, current_type, compression=0.5)
                self.transitions.append(trans)
                current_type = trans.out_type

        # Final norm + group pool
        self.final_bn   = enn.InnerBatchNorm(current_type)
        self.final_relu = enn.ReLU(current_type)
        self.group_pool = enn.GroupPooling(current_type)

        # After GroupPooling: one trivial repr per field → n_features scalars
        n_features = len(current_type.representations)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(n_features, num_classes)
        )

    def forward(self, x):
        x = enn.GeometricTensor(x, self.in_type)
        x = self.stem(x)

        for i, block in enumerate(self.blocks):
            x = block(x)
            if i < len(self.transitions):
                x = self.transitions[i](x)

        x = self.final_bn(x)
        x = self.final_relu(x)
        x = self.group_pool(x)
        return self.classifier(x.tensor)


# ── Instantiate and report ────────────────────────────────────────────────────

eq_densenet = EquivariantDenseNet(
    num_classes=3,
    growth_rate=6,       # 6 regular reprs per dense layer = 48 channels (C8)
    block_config=(4, 4, 4),
    init_features=8,     # 8 regular reprs in stem = 64 channels (C8)
    N=8                  # C8: 8-fold rotation
)

total_p = sum(p.numel() for p in eq_densenet.parameters())
print(f'EquivariantDenseNet (C8)')
print(f'  Group          : C8 (rot2dOnR2, N=8) — rotations at 45° increments')
print(f'  Growth rate    : 6 regular reprs per layer ({6*8} channels)')
print(f'  Block config   : (4, 4, 4)')
print(f'  Total params   : {total_p/1e6:.3f}M')
print(f'  Pretrained     : No (no ImageNet weights for equivariant architectures)')

# ── Load saved weights or train from scratch ──────────────────────────────────
# Checkpoint was saved as a raw state dict:
#   torch.save(model.state_dict(), path)
# Keys are directly model parameter names — no wrapper dict.
# Retraining only occurs if the weights file is not found on Drive.

eq_densenet_path = os.path.join(gsoc_drive_path, 'EqDenseNet-C8_best.pth')

if os.path.exists(eq_densenet_path):
    # Raw state dict — load directly, no key unpacking needed
    state_dict = torch.load(eq_densenet_path, map_location=device)
    eq_densenet.load_state_dict(state_dict)
    eq_densenet.to(device)
    eq_densenet.eval()
    print(f'\nLoaded saved weights from: {eq_densenet_path}')
    print(f'  Parameters verified: {sum(p.numel() for p in eq_densenet.parameters())/1e6:.3f}M')
else:
    print('\nNo saved weights found — training from scratch.')
    train_and_evaluate(
        eq_densenet,
        model_name='EqDenseNet-C8',
        epochs=60,
        lr=1e-3,        # from-scratch equivariant models need higher LR than pretrained
        patience=15,
        pretrained=False
    )

In [ ]:
# ── Display training curves ───────────────────────────────────────────────────
# Saved during initial training run — display without retraining.

from IPython.display import Image, display

training_fig_path = os.path.join(gsoc_drive_path, 'figures',
                                 'fig5_10a_eqdensenet_c8_training.png')
if os.path.exists(training_fig_path):
    print('EqDenseNet-C8 — Training Curves')
    display(Image(training_fig_path))
else:
    print(f'Training figure not found at: {training_fig_path}')

# ── Run evaluation and display results ───────────────────────────────────────
# Runs inference on val set, computes all metrics, saves and displays
# the 4-panel evaluation figure (ROC + CM + PR + Calibration).

from sklearn.metrics import (confusion_matrix, roc_auc_score,
                             classification_report, average_precision_score)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

eq_densenet.eval()
all_probs, all_preds, all_labels = [], [], []

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs   = imgs.to(device)
        logits = eq_densenet(imgs)
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(probs.argmax(1))
        all_labels.append(labels.numpy())

probs_arr  = np.concatenate(all_probs,  axis=0)
preds_arr  = np.concatenate(all_preds,  axis=0)
labels_arr = np.concatenate(all_labels, axis=0)
y_bin      = label_binarize(labels_arr, classes=[0, 1, 2])

# ── Metrics ───────────────────────────────────────────────────────────────────
macro_auc     = roc_auc_score(y_bin, probs_arr, multi_class='ovr', average='macro')
per_class_auc = roc_auc_score(y_bin, probs_arr, multi_class='ovr', average=None)
accuracy      = (preds_arr == labels_arr).mean()
cm            = confusion_matrix(labels_arr, preds_arr)
sphere_recall = cm[1, 1] / cm[1].sum()
sphere_pr_auc = average_precision_score(
    (labels_arr == 1).astype(int), probs_arr[:, 1]
)

print(f'\n{"="*60}')
print(f'  EqDenseNet-C8  |  Val Set Results')
print(f'{"="*60}')
print(classification_report(
    labels_arr, preds_arr,
    target_names=CLASS_NAMES, digits=4
))
print(f'  Macro AUC         : {macro_auc:.4f}')
print(f'  AUC No Sub        : {per_class_auc[0]:.4f}')
print(f'  AUC Sphere        : {per_class_auc[1]:.4f}')
print(f'  AUC Vortex        : {per_class_auc[2]:.4f}')
print(f'  Sphere PR-AUC     : {sphere_pr_auc:.4f}')
print(f'  Test Accuracy     : {accuracy*100:.2f}%')
print(f'  Sphere Recall     : {sphere_recall:.4f}')
print(f'{"="*60}')

# ── Store in ARCH_METRICS ─────────────────────────────────────────────────────
ARCH_METRICS['EqDenseNet-C8'] = {
    'probs':          probs_arr,
    'preds':          preds_arr,
    'labels':         labels_arr,
    'macro_auc':      macro_auc,
    'per_class_auc':  per_class_auc.tolist(),
    'sphere_pr_auc':  sphere_pr_auc,
    'test_accuracy':  accuracy,
    'sphere_recall':  sphere_recall,
    'total_params_M': total_p / 1e6,
    'pretrained':     False,
}
print('Stored in ARCH_METRICS["EqDenseNet-C8"].')

# ── 4-panel evaluation figure ─────────────────────────────────────────────────
from sklearn.metrics import roc_curve, precision_recall_curve

COLORS = ['#3498db', '#e74c3c', '#2ecc71']

fig = plt.figure(figsize=(18, 14))
fig.suptitle('EqDenseNet-C8', fontsize=15, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# ROC
ax_roc = fig.add_subplot(gs[0, 0])
for c, (cls, col) in enumerate(zip(CLASS_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, c], probs_arr[:, c])
    ax_roc.plot(fpr, tpr, color=col, lw=2,
                label=f'{cls} (AUC={per_class_auc[c]:.3f})')
mean_fpr = np.linspace(0, 1, 500)
mean_tpr = np.mean([
    np.interp(mean_fpr, *roc_curve(y_bin[:, c], probs_arr[:, c])[:2])
    for c in range(3)
], axis=0)
ax_roc.plot(mean_fpr, mean_tpr, 'k--', lw=2,
            label=f'Macro-Avg (AUC={macro_auc:.3f})')
ax_roc.plot([0, 1], [0, 1], 'gray', lw=1, linestyle=':')
ax_roc.set_xlabel('FPR', fontsize=11)
ax_roc.set_ylabel('TPR', fontsize=11)
ax_roc.set_title('ROC — EqDenseNet-C8', fontsize=11, fontweight='bold')
ax_roc.legend(fontsize=9)
ax_roc.grid(alpha=0.3)

# Confusion Matrix
ax_cm = fig.add_subplot(gs[0, 1])
im = ax_cm.imshow(cm, cmap='Blues', aspect='auto')
for r in range(3):
    for c in range(3):
        ax_cm.text(c, r, f'{cm[r, c]}',
                   ha='center', va='center', fontsize=12,
                   color='white' if cm[r, c] > cm.max() * 0.5 else 'black',
                   fontweight='bold')
ax_cm.set_xticks([0, 1, 2])
ax_cm.set_yticks([0, 1, 2])
ax_cm.set_xticklabels(['No Sub', 'Sphere', 'Vortex'], fontsize=10)
ax_cm.set_yticklabels(['No Sub', 'Sphere', 'Vortex'], fontsize=10)
ax_cm.set_xlabel('Predicted', fontsize=11)
ax_cm.set_ylabel('True', fontsize=11)
ax_cm.set_title('Confusion Matrix — EqDenseNet-C8', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax_cm)

# Sphere Precision-Recall
ax_pr = fig.add_subplot(gs[1, 0])
prec, rec, _ = precision_recall_curve((labels_arr == 1).astype(int), probs_arr[:, 1])
ax_pr.plot(rec, prec, color='#e74c3c', lw=2,
           label=f'Sphere PR-AUC={sphere_pr_auc:.3f}')
ax_pr.axhline(1/3, color='black', linestyle='--', lw=1.5,
              label='Random baseline')
ax_pr.set_xlabel('Recall', fontsize=11)
ax_pr.set_ylabel('Precision', fontsize=11)
ax_pr.set_title('Precision-Recall (Sphere) — EqDenseNet-C8',
                fontsize=11, fontweight='bold')
ax_pr.legend(fontsize=9)
ax_pr.grid(alpha=0.3)

# Calibration
ax_cal = fig.add_subplot(gs[1, 1])
for c, (cls, col) in enumerate(zip(CLASS_NAMES, COLORS)):
    prob_true, prob_pred = calibration_curve(
        (labels_arr == c).astype(int), probs_arr[:, c], n_bins=10
    )
    ax_cal.plot(prob_pred, prob_true, 'o-', color=col, lw=2,
                markersize=5, label=cls)
ax_cal.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
ax_cal.set_xlabel('Mean predicted confidence', fontsize=11)
ax_cal.set_ylabel('Fraction correct', fontsize=11)
ax_cal.set_title('Calibration Curve — EqDenseNet-C8',
                 fontsize=11, fontweight='bold')
ax_cal.legend(fontsize=9)
ax_cal.grid(alpha=0.3)

plt.tight_layout()

eval_fig_path = os.path.join(gsoc_drive_path, 'figures',
                              'fig5_10b_eqdensenet_c8_eval.png')
plt.savefig(eval_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {eval_fig_path}')

### 4.11 Top-6 Soft-Voting Ensemble

**Role:** Tests whether averaging complementary high-performing models improves discrimination beyond any single architecture.

This ensemble combines the six strongest individual models by averaging their predicted class probabilities on the validation set:
- EqDenseNet-$C_8$
- DenseNet-121
- E-ResNet
- ResNet-50
- ResNet-18
- EfficientNet-B3

The ensemble uses **soft voting** rather than hard voting. Instead of taking only the final class label from each model, it averages the full probability vectors and then predicts the class with the highest mean probability. This is preferable here because it retains confidence information and allows uncertain and confident predictions to contribute differently.

Scientifically, the ensemble is useful for a simple reason: the top models are not architecturally identical. They combine different inductive biases, including pretrained residual CNNs, dense feature-reuse networks, parameter-efficient modern CNNs, and symmetry-aware equivariant models trained from scratch. If their errors are only partially correlated, probability averaging can reduce variance and smooth out architecture-specific mistakes.

At the same time, the ensemble should be interpreted cautiously. It is not a new model class, but a post hoc combination of the best-performing models on the same benchmark split. Its performance therefore shows the value of predictive complementarity within this benchmark, not the emergence of a simpler or more interpretable scientific mechanism.

The ensemble is evaluated using the same validation labels and reported with the same metrics as the individual models, including macro AUC, per-class AUC, classification report, confusion matrix, and Sphere PR-AUC. Because it aggregates multiple networks, parameter count is not directly comparable to the single-model entries in the benchmark table.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ensemble — Top-6 Soft Vote
# Average softmax probabilities across the 6 best-performing models.
# Soft voting (probability averaging) is more informative than hard voting
# because it weights confident predictions more than uncertain ones.
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
from sklearn.metrics import (confusion_matrix, roc_auc_score,
                             classification_report)
from sklearn.preprocessing import label_binarize

TOP6 = ['EqDenseNet-C8', 'DenseNet-121', 'E-ResNet',
        'ResNet-50', 'ResNet-18', 'EfficientNet-B3']

# Verify all are in ARCH_METRICS
missing = [m for m in TOP6 if m not in ARCH_METRICS]
if missing:
    print(f'Missing from ARCH_METRICS: {missing}')
    print('Run the evaluation cells for those models first.')
else:
    print(f'All {len(TOP6)} models found in ARCH_METRICS.')

# ── Soft vote ─────────────────────────────────────────────────────────────────

prob_stack    = np.stack(
    [ARCH_METRICS[m]['probs'] for m in TOP6], axis=0
)                                              # (6, 7500, 3)
labels_arr    = np.array(ARCH_METRICS['ResNet-18']['labels'])

mean_probs    = prob_stack.mean(axis=0)        # (7500, 3)
ensemble_pred = mean_probs.argmax(axis=1)      # (7500,)

# ── Metrics ───────────────────────────────────────────────────────────────────

y_bin     = label_binarize(labels_arr, classes=[0, 1, 2])
macro_auc = roc_auc_score(y_bin, mean_probs,
                          multi_class='ovr', average='macro')
per_class_auc = roc_auc_score(y_bin, mean_probs,
                               multi_class='ovr', average=None)

accuracy      = (ensemble_pred == labels_arr).mean()
cm            = confusion_matrix(labels_arr, ensemble_pred)
sphere_recall = cm[1, 1] / cm[1].sum()

# Sphere PR-AUC
from sklearn.metrics import average_precision_score
sphere_pr_auc = average_precision_score(
    (labels_arr == 1).astype(int), mean_probs[:, 1]
)

# ── Report ────────────────────────────────────────────────────────────────────

print(f'\n{"═"*60}')
print(f'  Top-6 Soft Ensemble')
print(f'  Members: {", ".join(TOP6)}')
print(f'{"═"*60}')
print(f'\n── Classification Report ────────────────────────────────')
print(classification_report(
    labels_arr, ensemble_pred,
    target_names=CLASS_NAMES, digits=4
))

print(f'── Key Metrics ──────────────────────────────────────────')
print(f'  Macro AUC         : {macro_auc:.4f}')
print(f'  AUC No Sub        : {per_class_auc[0]:.4f}')
print(f'  AUC Sphere        : {per_class_auc[1]:.4f}')
print(f'  AUC Vortex        : {per_class_auc[2]:.4f}')
print(f'  Sphere PR-AUC     : {sphere_pr_auc:.4f}')
print(f'  Test Accuracy     : {accuracy*100:.2f}%')
print(f'  Sphere Recall     : {sphere_recall:.4f}')
print(f'  Parameters        : N/A (ensemble)')
print(f'  Pretrained        : Mixed')

print(f'\n── Confusion Matrix ─────────────────────────────────────')
print(f'  {"":20s}  {"No Sub":>8}  {"Sphere":>8}  {"Vortex":>8}')
for i, cls in enumerate(CLASS_NAMES):
    row = cm[i]
    print(f'  {cls:20s}  {row[0]:>8}  {row[1]:>8}  {row[2]:>8}')

# ── Store in ARCH_METRICS for inclusion in benchmark table ───────────────────

ARCH_METRICS['Ensemble-Top6'] = {
    'probs':          mean_probs,
    'preds':          ensemble_pred,
    'labels':         labels_arr,
    'macro_auc':      macro_auc,
    'test_accuracy':  accuracy,
    'sphere_recall':  sphere_recall,
    'total_params_M': None,
    'pretrained':     'Mixed',
}

ARCH_METRICS['Ensemble-Top6'].update({
    'per_class_auc': per_class_auc.tolist(),  # [no_sub, sphere, vortex]
    'sphere_pr_auc': sphere_pr_auc,
})

print(f'\nStored as ARCH_METRICS["Ensemble-Top6"]')
print('Add to benchmark table: rank 0 (above individual models if it leads).')

In [ ]:
from sklearn.metrics import roc_curve
import numpy as np

# Compute macro-averaged ROC for ensemble
fpr_list, tpr_list = [], []
for c in range(3):
    f, t, _ = roc_curve((labels_arr == c).astype(int), mean_probs[:, c])
    fpr_list.append(f)
    tpr_list.append(t)

# Interpolate to common FPR axis
mean_fpr = np.linspace(0, 1, 500)
mean_tpr = np.mean([np.interp(mean_fpr, f, t)
                    for f, t in zip(fpr_list, tpr_list)], axis=0)

ARCH_METRICS['Ensemble-Top6']['fpr'] = mean_fpr
ARCH_METRICS['Ensemble-Top6']['tpr'] = mean_tpr
print('ROC curve stored for Ensemble-Top6.')

In [ ]:
# ── Soft Ensemble (Top-6) — Evaluation Figure ─────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                             precision_recall_curve, average_precision_score)
from sklearn.calibration import calibration_curve

ens     = ARCH_METRICS['Ensemble-Top6']
probs   = np.array(ens['probs'])
preds   = np.array(ens['preds'])
labels  = np.array(ens['labels'])
y_bin   = label_binarize(labels, classes=[0, 1, 2])

CLASS_NAMES = ['No Substructure', 'Sphere', 'Vortex']
COLORS      = ['#3498db', '#e74c3c', '#2ecc71']
cm          = confusion_matrix(labels, preds)

fig = plt.figure(figsize=(18, 14))
fig.suptitle('Soft Ensemble (Top-6)', fontsize=15, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# ── ROC ───────────────────────────────────────────────────────────────────────
ax_roc = fig.add_subplot(gs[0, 0])
for c, (cls, col) in enumerate(zip(CLASS_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_bin[:, c], probs[:, c])
    auc = roc_auc_score(y_bin[:, c], probs[:, c])
    ax_roc.plot(fpr, tpr, color=col, lw=2,
                label=f'{cls} (AUC={auc:.3f})')

# Macro averaged
mean_fpr = np.linspace(0, 1, 500)
mean_tpr = np.mean([
    np.interp(mean_fpr, *roc_curve(y_bin[:, c], probs[:, c])[:2])
    for c in range(3)
], axis=0)
macro_auc = ens['macro_auc']
ax_roc.plot(mean_fpr, mean_tpr, 'k--', lw=2,
            label=f'Macro-Avg (AUC={macro_auc:.3f})')
ax_roc.plot([0, 1], [0, 1], 'gray', lw=1, linestyle=':')
ax_roc.set_xlabel('FPR', fontsize=11)
ax_roc.set_ylabel('TPR', fontsize=11)
ax_roc.set_title('ROC — Soft Ensemble (Top-6)', fontsize=11, fontweight='bold')
ax_roc.legend(fontsize=9)
ax_roc.grid(alpha=0.3)

# ── Confusion Matrix ──────────────────────────────────────────────────────────
ax_cm = fig.add_subplot(gs[0, 1])
im = ax_cm.imshow(cm, cmap='Blues', aspect='auto')
for r in range(3):
    for c in range(3):
        ax_cm.text(c, r, f'{cm[r,c]}',
                   ha='center', va='center', fontsize=12,
                   color='white' if cm[r,c] > cm.max()*0.5 else 'black',
                   fontweight='bold')
ax_cm.set_xticks([0, 1, 2])
ax_cm.set_yticks([0, 1, 2])
ax_cm.set_xticklabels(['No Sub', 'Sphere', 'Vortex'], fontsize=10)
ax_cm.set_yticklabels(['No Sub', 'Sphere', 'Vortex'], fontsize=10)
ax_cm.set_xlabel('Predicted', fontsize=11)
ax_cm.set_ylabel('True', fontsize=11)
ax_cm.set_title('Confusion Matrix — Soft Ensemble (Top-6)',
                fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax_cm)

# ── Sphere Precision-Recall ───────────────────────────────────────────────────
ax_pr = fig.add_subplot(gs[1, 0])
prec, rec, _ = precision_recall_curve((labels == 1).astype(int), probs[:, 1])
pr_auc       = ens['sphere_pr_auc']
ax_pr.plot(rec, prec, color='#e74c3c', lw=2,
           label=f'Sphere PR-AUC={pr_auc:.3f}')
ax_pr.axhline(1/3, color='black', linestyle='--', lw=1.5,
              label='Random baseline')
ax_pr.set_xlabel('Recall', fontsize=11)
ax_pr.set_ylabel('Precision', fontsize=11)
ax_pr.set_title('Precision-Recall (Sphere) — Soft Ensemble (Top-6)',
                fontsize=11, fontweight='bold')
ax_pr.legend(fontsize=9)
ax_pr.grid(alpha=0.3)

# ── Calibration ───────────────────────────────────────────────────────────────
ax_cal = fig.add_subplot(gs[1, 1])
for c, (cls, col) in enumerate(zip(CLASS_NAMES, COLORS)):
    prob_true, prob_pred = calibration_curve(
        (labels == c).astype(int), probs[:, c], n_bins=10
    )
    ax_cal.plot(prob_pred, prob_true, 'o-', color=col, lw=2,
                markersize=5, label=cls)
ax_cal.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect calibration')
ax_cal.set_xlabel('Mean predicted confidence', fontsize=11)
ax_cal.set_ylabel('Fraction correct', fontsize=11)
ax_cal.set_title('Calibration Curve — Soft Ensemble (Top-6)',
                 fontsize=11, fontweight='bold')
ax_cal.legend(fontsize=9)
ax_cal.grid(alpha=0.3)

plt.tight_layout()
save_path = os.path.join(gsoc_drive_path, 'figures', 'fig5_11_ensemble_eval.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {save_path}')

# Also copy to assets/
import shutil
import os

assets_dir = 'assets'
os.makedirs(assets_dir, exist_ok=True)
shutil.copy(save_path, os.path.join(assets_dir, 'fig5_11_ensemble_eval.png'))
print(f'Copied to assets/fig5_11_ensemble_eval.png')

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(
    np.array(ARCH_METRICS['Ensemble-Top6']['labels']),
    np.array(ARCH_METRICS['Ensemble-Top6']['preds']),
    target_names=CLASS_NAMES, digits=4
))

## <a id="5-comprehensive-results-summary"></a>5. Comprehensive Results Summary

All eleven architectures are evaluated on the predefined val partition using
the same training protocol and evaluation metrics. Results are sorted by
macro-averaged AUC. Three figures follow the table — the macro-averaged ROC
comparison, the Sphere-class precision-recall comparison, and a parameter
efficiency scatter plot.

Two metrics deserve particular attention beyond macro AUC. **Sphere Recall**
measures what fraction of true Sphere subhalos the model correctly identifies.
A missed Sphere detection is a missed dark matter substructure — physically
the most consequential error type. **Sphere PR-AUC** measures detection
quality for the hardest class independently of the other two, and is the
most sensitive indicator of whether a model has genuinely learned the
Sphere signal or is merely benefiting from the easier No Substructure /
Vortex boundary.

In [ ]:
import pandas as pd

rows = []
for name, m in ARCH_METRICS.items():
    params = m['total_params_M']
    rows.append({
        'Architecture':   name,
        'Pretrained':     '✅' if m['pretrained'] else '❌',
        'Params (M)':     f"{params:.1f}" if params is not None else 'N/A',
        'Macro-AUC':      f"{m['macro_auc']:.4f}",
        'AUC (No Sub)':   f"{m['per_class_auc'][0]:.4f}",
        'AUC (Sphere)':   f"{m['per_class_auc'][1]:.4f}",
        'AUC (Vortex)':   f"{m['per_class_auc'][2]:.4f}",
        'Sphere PR-AUC':  f"{m['sphere_pr_auc']:.4f}",
        'Test Acc.':      f"{m['test_accuracy']:.4f}",
        'Sphere Recall':  f"{m['sphere_recall']:.4f}",
    })

df = pd.DataFrame(rows)
df = df.sort_values('Macro-AUC', ascending=False).reset_index(drop=True)
df.index += 1

print('\n── Comprehensive Results Table (sorted by Macro-AUC) ──────────────────')
print(df.to_string())
print('\n* All metrics evaluated on the val partition (7,500 images).')
print('* Sphere Recall and Sphere PR-AUC are the most discriminating metrics')
print('  for this dataset (see Section 6.1).')
print('* Ensemble-Top6: soft vote across EqDenseNet-C8, DenseNet-121,')
print('  E-ResNet, ResNet-50, ResNet-18, EfficientNet-B3.')

### 5.1 Macro-Averaged ROC Curves

The ROC comparison reveals three performance tiers clearly. The top cluster
— DenseNet-121, E-ResNet, ResNet-50, ResNet-18, EfficientNet-B3 — all
achieve macro AUC above 0.989, with curves tightly grouped near the
top-left corner. ViT-Base forms a distinct second tier at 0.976, performing
well but noticeably below the convolutional models. VGG-16 and the two
from-scratch architectures — Equivariant-D4 and AlexNet — form a third tier
where performance degrades substantially.

The separation between tiers is not continuous — there is a visible gap
between EfficientNet-B3 (0.9898) and ViT-Base (0.9761), and another between
ViT-Base and VGG-16 (0.8944). Within this experimental setup, these gaps
are consistent with a pattern where skip-connection convolutional models
form the top cluster, the attention-based architecture forms a middle tier,
and architectures without skip connections form the lower tier.

In [ ]:
# ── Diagnose ARCH_METRICS key structure ───────────────────────────────────────
print('ARCH_METRICS entries and keys:\n')
for name, m in ARCH_METRICS.items():
    has_fpr = 'fpr' in m
    has_tpr = 'tpr' in m
    has_auc = 'macro_auc' in m
    print(f'  {name:<25} fpr={has_fpr}  tpr={has_tpr}  macro_auc={has_auc}')
    if not has_fpr:
        print(f'    Keys present: {list(m.keys())}')

In [ ]:
from sklearn.metrics import roc_curve, auc

# Recompute fpr/tpr from stored probs/labels for EqDenseNet-C8
_labels = ARCH_METRICS['EqDenseNet-C8']['labels']
_probs  = ARCH_METRICS['EqDenseNet-C8']['probs']
_preds  = ARCH_METRICS['EqDenseNet-C8']['preds']

# Binarize for macro-averaged ROC (same pattern as other models)
from sklearn.preprocessing import label_binarize
y_bin = label_binarize(_labels, classes=[0, 1, 2])

all_fpr = np.linspace(0, 1, 300)
tprs    = []
roc_auc = {}

for c in range(3):
    fpr_c, tpr_c, _ = roc_curve(y_bin[:, c], _probs[:, c])
    roc_auc[c]      = auc(fpr_c, tpr_c)
    tprs.append(np.interp(all_fpr, fpr_c, tpr_c))

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[0]  = 0.0
mean_tpr[-1] = 1.0

ARCH_METRICS['EqDenseNet-C8']['fpr'] = all_fpr
ARCH_METRICS['EqDenseNet-C8']['tpr'] = mean_tpr

print('EqDenseNet-C8 fpr/tpr recomputed.')
print(f'  macro_auc : {ARCH_METRICS["EqDenseNet-C8"]["macro_auc"]:.4f}')
print(f'  fpr shape : {all_fpr.shape}')
print(f'  tpr shape : {mean_tpr.shape}')

# Verify all entries now have fpr/tpr
print('\nVerification:')
for name, m in ARCH_METRICS.items():
    print(f'  {name:<25} fpr={"fpr" in m}  tpr={"tpr" in m}')

In [ ]:
plt.figure(figsize=(14, 9))

colors = [
    'navy', 'darkgreen', 'purple', 'crimson', 'darkorange',
    'teal', 'magenta', 'goldenrod', 'dodgerblue',
    '#e74c3c',   # EqDenseNet-C8 — bright red to stand out
    '#2ecc71',   # Ensemble-Top6 — bright green
]

# Sort by AUC descending
sorted_models = sorted(ARCH_METRICS.items(),
                       key=lambda x: x[1]['macro_auc'], reverse=True)

for i, (name, m) in enumerate(sorted_models):
    # Thicker line for top models and ensemble
    lw    = 2.8 if name in ('EqDenseNet-C8', 'Ensemble-Top6') else 2.0
    alpha = 1.0 if name in ('EqDenseNet-C8', 'Ensemble-Top6') else 0.85
    ls    = '--' if name == 'Ensemble-Top6' else '-'
    plt.plot(m['fpr'], m['tpr'],
             color=colors[i % len(colors)],
             lw=lw, alpha=alpha, linestyle=ls,
             label=f"{name}  (AUC={m['macro_auc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5)
plt.xlim([0, 1]); plt.ylim([0, 1.02])
plt.xlabel('False Positive Rate (macro-averaged)', fontsize=13)
plt.ylabel('True Positive Rate (macro-averaged)', fontsize=13)
plt.title('Macro-Averaged ROC — All Architectures\n(Val Set)', fontsize=14)
plt.legend(loc='lower right', fontsize=9.5, framealpha=0.9)
plt.grid(alpha=0.3)
plt.tight_layout()

fig_dir = os.path.join(gsoc_drive_path, 'figures')
os.makedirs(fig_dir, exist_ok=True)
plt.savefig(os.path.join(fig_dir, 'macro_roc_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Sphere-Class Precision-Recall Comparison

The Sphere-class PR curves reveal considerably more differentiation than the ROC
curves suggest. Models that appear nearly indistinguishable on macro-averaged ROC
separate visibly here — Sphere PR-AUC spans from 0.9932 (EqDenseNet-C8) down to
0.4380 (AlexNet), a range of 0.555 compressed into a much narrower macro AUC gap.

**EqDenseNet-C8 leads at 0.9932**, followed closely by the Ensemble-Top6 (0.9923)
and DenseNet-121 (0.9903). The equivariant models occupy both ends of the top tier:
EqDenseNet-C8 at the top and Equivariant-D4 (0.9574) at the bottom of the skip-connection
cluster — confirming that equivariance alone is not sufficient without sufficient
architectural depth and dense connectivity.

The three-tier structure visible in the ROC comparison is even more pronounced here:

- **Tier 1** (EqDenseNet-C8 through EfficientNet-B3, PR-AUC 0.975–0.993): curves
  remain near precision = 1.0 across the full recall range before collapsing sharply
  only at recall > 0.95.
- **Tier 2** (ViT-Base, 0.9413): visible separation from Tier 1 begins around
  recall = 0.8, with precision declining more steeply.
- **Tier 3** (VGG-16, 0.8065; AlexNet, 0.4380): VGG-16 shows a continuous
  precision decline from recall = 0.0; AlexNet at 0.4380 is only marginally above
  the random baseline of 0.33, indicating near-chance Sphere discrimination.

The random baseline at 0.33 is the expected PR-AUC for a model that assigns equal
probability to all three classes. AlexNet's proximity to this floor confirms that
the skip-connection gap — not parameter count — is the primary driver of Sphere
detection failure.

In [ ]:
# Sphere-class Precision-Recall comparison across all models
plt.figure(figsize=(12, 8))

for i, (name, m) in enumerate(
        sorted(ARCH_METRICS.items(),
               key=lambda x: x[1]['sphere_pr_auc'], reverse=True)):
    y_bin = label_binarize(m['labels'], classes=[0,1,2])
    prec, rec, _ = precision_recall_curve(y_bin[:, 1], m['probs'][:, 1])
    plt.plot(rec, prec, color=colors[i % len(colors)], lw=2.0,
             label=f"{name}  (PR-AUC={m['sphere_pr_auc']:.4f})")

baseline = (label_binarize(ARCH_METRICS[list(ARCH_METRICS.keys())[0]]['labels'],
                            classes=[0,1,2])[:, 1]).mean()
plt.axhline(baseline, color='k', linestyle='--', lw=1.5,
            label=f'Random baseline ({baseline:.2f})')

plt.xlabel('Recall', fontsize=13)
plt.ylabel('Precision', fontsize=13)
plt.title('Sphere-Class Precision-Recall Comparison\n(Test Set)', fontsize=14)
plt.legend(loc='upper right', fontsize=10, framealpha=0.9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'sphere_pr_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()


### 5.3 Parameter Efficiency

The parameter efficiency scatter plots macro AUC against parameter count on a
log scale, with triangles marking from-scratch training and circles marking
ImageNet-pretrained models.

The central finding is immediately visible: **EqDenseNet-C8 occupies the top-left
corner** — the highest macro AUC in the entire benchmark (0.9974) at the lowest
parameter count of any competitive model (0.093M), trained entirely from scratch.
No other architecture comes close to this position. E-ResNet (triangle, ~0.39M)
sits just to its right, also in the top-AUC band and well left of every pretrained
model — confirming that the equivariant family as a whole dominates the efficiency
frontier.

The pretrained models cluster in the middle-right region: DenseNet-121, ResNet-18,
EfficientNet-B3, and ResNet-50 form a dense group around 7–25M parameters, all
achieving macro AUC near 1.0 but none surpassing EqDenseNet-C8. ViT-Base (85M
parameters) falls noticeably below this pretrained cluster despite being the largest
model in the group — the attention-based architecture does not translate its parameter
count into competitive AUC on this task.

The Equivariant-D4 triangle sits isolated in the lower-left — low parameters but
also substantially lower AUC (~0.986 macro) relative to E-ResNet and EqDenseNet-C8.
This isolates the contribution of depth and dense connectivity within the equivariant
family: symmetry encoding alone is not sufficient without sufficient architectural
capacity.

VGG-16 (134M parameters, macro AUC 0.8944) occupies the worst position on the plot —
the clearest demonstration that scale without skip connections provides no benefit for
this task. AlexNet (57M, 0.659) is the outlier in the opposite direction. The
Ensemble-Top6 is excluded as it has no single parameter count.

In [ ]:
# Parameter efficiency: Macro-AUC vs. log(Parameters)
fig, ax = plt.subplots(figsize=(10, 6))

for i, (name, m) in enumerate(ARCH_METRICS.items()):
    params_M = m['total_params_M']

    # Skip ensemble — no single parameter count
    if params_M is None:
        continue

    auc_val = m['macro_auc']
    marker  = 'o' if m['pretrained'] else '^'
    ax.scatter(params_M, auc_val, s=120, color=colors[i % len(colors)],
               marker=marker, zorder=5)
    ax.annotate(name, (params_M, auc_val),
                textcoords='offset points', xytext=(6, 3), fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('Parameter Count (M, log scale)', fontsize=12)
ax.set_ylabel('Macro-AUC (Val Set)', fontsize=12)
ax.set_title('Parameter Efficiency\n(○ pretrained  △ from scratch)', fontsize=13)
ax.grid(alpha=0.3)

# Add note about excluded ensemble
ax.text(0.99, 0.02,
        'Ensemble-Top6 excluded (no single parameter count)',
        transform=ax.transAxes, fontsize=8,
        color='gray', ha='right', va='bottom')

plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'param_efficiency.png'),
            dpi=150, bbox_inches='tight')
plt.show()

###5.4 Cross Architechture Confusion Matrix Grid


Confusion matrices for all 10 architectures (Ensemble wasn't included), evaluated on the validation set
(7,500 images, 2,500 per class) and sorted by macro AUC descending. Values are
row-normalised fractions with raw counts in parentheses.

Three structural patterns are consistent across all well-converged models:

**No Substructure is the easiest class.** All Tier 1 models achieve recall ≥ 0.98
on No Substructure. The smooth Einstein ring morphology is sufficiently distinct
that even VGG-16 (0.99, 2472/2500) correctly identifies it at high rates.

**Sphere is the universal bottleneck.** Sphere recall is the lowest per-class
metric for every model without exception. The best result is EqDenseNet-C8 at 0.94
(2362/2500); the worst among Tier 1 is EfficientNet-B3 at 0.88 (2212/2500).
Sphere misclassifications flow predominantly into No Substructure — the compact,
approximately symmetric CDM subhalo perturbation is physically similar to the
smooth-arc edge effects it produces. The Sphere → Vortex confusion rate is
consistently low across all models, confirming this is a Sphere/No-Sub ambiguity,
not a Sphere/Vortex ambiguity.

**Vortex recall is high but not as high as No Substructure.** Tier 1 models achieve
0.97–0.98 on Vortex. The elongated, asymmetric perturbation is geometrically distinct
enough that most models resolve it cleanly, but a small fraction leaks into Sphere
and No Sub at the low-SNR boundary.

**Tier and architecture summary:**

| Tier | Models | Sphere Recall |
|:-----|:-------|:-------------:|
| Tier 1 (AUC > 0.989) | EqDenseNet-C8, DenseNet-121, E-ResNet, ResNet-50, ResNet-18, EfficientNet-B3 | 0.88–0.94 |
| Tier 2 (ViT-Base) | ViT-Base/16 | 0.78 |
| Tier 3 (no skip / shallow) | VGG-16, Equivariant-D4, AlexNet | 0.50–0.80 |

The Equivariant-D4 (0.80 Sphere recall, AUC 0.9782) sits anomalously between
Tier 2 and Tier 1 — it is the only from-scratch model in the lower tier, confirming
that equivariance without sufficient depth and skip connections does not deliver Tier 1
performance. EqDenseNet-C8 and E-ResNet demonstrate that the missing ingredient is
architectural capacity, not the symmetry prior itself.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cross-Architecture Confusion Matrix Grid
# All 10 models evaluated on val_ds, sorted by macro AUC
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize
import torch
import torch.nn.functional as F
import os

# ── 1. All 10 models ──────────────────────────────────────────────────────────

all_models = {
    'DenseNet-121':     densenet,
    'EqDenseNet-C8':    eq_densenet,    # ← new
    'E-ResNet':         e_resnet,
    'ResNet-50':        resnet50,
    'ResNet-18':        resnet18,
    'EfficientNet-B3':  effnet,
    'ViT-Base/16':      vit_model,
    'VGG-16':           vgg16,
    'Equivariant-D4':   enn_model,
    'AlexNet':          alexnet,
}

# ── 2. Inference helper ───────────────────────────────────────────────────────

def get_preds_and_probs(model, loader, device):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            try:
                logits = model(imgs)
            except Exception:
                x_geo = enn.GeometricTensor(imgs, model.in_type)
                logits = model(x_geo)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            all_labels.extend(labels.numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# ── 3. Evaluate all models ────────────────────────────────────────────────────

results = {}

for name, model in all_models.items():
    print(f'  Evaluating {name}...', end=' ', flush=True)
    labels, preds, probs = get_preds_and_probs(model, val_loader, device)
    cm  = confusion_matrix(labels, preds)
    y_b = label_binarize(labels, classes=[0, 1, 2])
    auc = roc_auc_score(y_b, probs, multi_class='ovr', average='macro')
    results[name] = {'cm': cm, 'auc': auc}
    print(f'AUC = {auc:.4f}')

# ── 4. Sort by macro AUC descending ──────────────────────────────────────────

sorted_names = sorted(results, key=lambda n: results[n]['auc'], reverse=True)
print('\nFinal ranking:')
for i, n in enumerate(sorted_names, 1):
    print(f'  {i:2d}. {n:20s}  AUC = {results[n]["auc"]:.4f}')

# ── 5. Plot 2×5 grid (10 models) ─────────────────────────────────────────────

# Tier boundaries updated for 10 models:
#   Tier 1 (green):  ranks 1-6  — AUC > 0.989
#   Tier 2 (orange): rank 7     — ViT-Base
#   Tier 3 (red):    ranks 8-10 — no skip / shallow

TIER_COLORS = ['#2ecc71', '#f39c12', '#e74c3c']

def tier(rank):
    if rank <= 6:   return 0
    elif rank == 7: return 1
    else:           return 2

n_models = len(sorted_names)
n_cols   = 5
n_rows   = 2   # 2×5 = 10

fig = plt.figure(figsize=(28, 12))
fig.patch.set_facecolor('#0f0f0f')
gs  = gridspec.GridSpec(n_rows, n_cols, figure=fig, hspace=0.48, wspace=0.35)
cmap = plt.cm.Blues
last_im = None

for idx, name in enumerate(sorted_names):
    row, col = divmod(idx, n_cols)
    ax = fig.add_subplot(gs[row, col])

    cm  = results[name]['cm']
    auc = results[name]['auc']
    tc  = TIER_COLORS[tier(idx)]

    # Row-normalise
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    last_im = ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1, aspect='auto')

    # Annotations
    for r in range(3):
        for c in range(3):
            pct   = cm_norm[r, c]
            raw   = cm[r, c]
            color = 'white' if pct > 0.55 else '#cccccc'
            ax.text(c, r, f'{pct:.2f}\n({raw})',
                    ha='center', va='center',
                    fontsize=7.5, color=color, fontweight='bold')

    short = ['No Sub', 'Sphere', 'Vortex']
    ax.set_xticks([0, 1, 2]);  ax.set_xticklabels(short, fontsize=7.5, color='#aaaaaa')
    ax.set_yticks([0, 1, 2]);  ax.set_yticklabels(short, fontsize=7.5, color='#aaaaaa', rotation=45)
    ax.tick_params(colors='#aaaaaa', length=0)

    if row == n_rows - 1: ax.set_xlabel('Predicted', fontsize=8, color='#888888', labelpad=4)
    if col == 0:          ax.set_ylabel('True',      fontsize=8, color='#888888', labelpad=4)

    ax.set_title(f'#{idx+1}  {name}\nMacro AUC = {auc:.4f}',
                 fontsize=9, fontweight='bold', color=tc, pad=6)

    for spine in ax.spines.values():
        spine.set_edgecolor(tc)
        spine.set_linewidth(1.8)

    ax.set_facecolor('#1a1a1a')

# Colour bar
cbar_ax = fig.add_axes([0.92, 0.15, 0.012, 0.7])
cb = fig.colorbar(last_im, cax=cbar_ax)
cb.set_label('Fraction of true class', color='#aaaaaa', fontsize=9)
cb.ax.yaxis.set_tick_params(color='#aaaaaa')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='#aaaaaa', fontsize=8)

# Main title
fig.suptitle(
    'Cross-Architecture Confusion Matrices — All 10 Models\n'
    'Sorted by Macro AUC  ·  Values: row-normalised fraction (raw count)',
    fontsize=13, fontweight='bold', color='white', y=1.01
)

# Tier legend
legend_elements = [
    Patch(facecolor=TIER_COLORS[0], label='Tier 1 — AUC > 0.989  (skip/dense connections)'),
    Patch(facecolor=TIER_COLORS[1], label='Tier 2 — ViT-Base  (attention)'),
    Patch(facecolor=TIER_COLORS[2], label='Tier 3 — No skip connections / shallow equivariant'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           frameon=False, fontsize=8.5, labelcolor='#cccccc',
           bbox_to_anchor=(0.46, -0.03))

# ── 6. Save ───────────────────────────────────────────────────────────────────

save_path = os.path.join(gsoc_drive_path, 'figures', 'fig8_1_cross_arch_confusion.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Saved → {save_path}')

### 5.5 Key Takeaways

Four main patterns emerge from the benchmark.

**1. Architectures with skip connections perform substantially better than the plain sequential baselines.**  
The largest performance gap is between AlexNet/VGG-16 and the residual or dense architectures. In this benchmark, skip-connected models are consistently stronger, although this should not be interpreted as a controlled causal comparison because architecture, training dynamics, and pretraining are not fully disentangled.

**2. The equivariant models are highly parameter-efficient.**  
EqDenseNet-C8 achieves the highest macro AUC in the benchmark while using far fewer parameters than the large pretrained CNN baselines, and E-ResNet is also competitive at small model size. This is consistent with the symmetry-aware inductive bias being well matched to the task, but the result should be interpreted as empirical rather than definitive proof of mechanism.

**3. Macro AUC alone hides meaningful differences on the Sphere class.**  
The top models are relatively close in macro AUC, but more clearly separated by Sphere Recall and Sphere PR-AUC. These class-specific metrics therefore provide a more informative view of performance on the most difficult and scientifically important class.

**4. The Top-6 soft-voting ensemble does not improve over EqDenseNet-C8 alone.**  
In this benchmark, the ensemble performs slightly worse than EqDenseNet-C8 on the main reported metrics. This suggests that simple uniform probability averaging is not beneficial here, possibly because the strongest single model already captures much of the useful signal and the remaining members do not add enough complementary error structure.

##<a></a>6. Understanding the Models

Sections 6.1 through 6.5 examine spatial attention patterns, predictive
uncertainty, ablation results, and equivariance properties of the trained
models. These analyses describe what the models attend to and how different
architectural components affect performance within this experimental setup —
they do not establish causal mechanisms.

### 6.1 Class Activation Maps — EqDenseNet-$C_8$

**Method:** Classic CAM is applied at the output of `group_pool`, immediately before global average pooling and the final linear classifier. This choice is architectural: EqDenseNet-$C_8$ satisfies the GAP + linear-head requirement for CAM directly, whereas Grad-CAM is less reliable here because of `escnn` in-place normalization layers. For each image, the CAM is computed for the **predicted** class.

**What is shown:** One correct and one incorrect example for each class. Green border denotes a correct prediction and red border an incorrect one. Columns show the input image, the CAM, and the CAM overlay.

**Main observations:** In the correct cases, activation generally overlaps the Einstein ring region rather than the background, indicating that the model is using lens structure rather than trivial image-wide statistics. The Vortex example appears more spatially asymmetric, while the No Substructure and Sphere cases are broader and more ring-distributed. In the misclassified examples, the model still often attends to the ring, but the spatial pattern alone does not cleanly explain the class decision.

**Caution:** These maps are coarse. They are generated from low-resolution feature maps and then upsampled to $150\times150$, so they should be read as approximate spatial emphasis, not pixel-level localization. With only six examples, they support qualitative inspection only and should not be treated as evidence of a general mechanistic rule.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Classic CAM: EqDenseNet-C8 — three classes, correct and wrong cases
#
# Classic CAM (Zhou et al., 2016) applied to the GroupPooling layer —
# the equivariant→standard tensor conversion immediately before GAP.
# Backpropagation through escnn is unreliable due to in-place BatchNorm;
# Classic CAM avoids this entirely using only linear weights + feature maps.
# EqDenseNet-C8 ends with GroupPooling → AdaptiveAvgPool2d(1) → Flatten →
# Linear(3) — exactly the GAP + Linear topology CAM requires.
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import cv2
import os

CLASS_NAMES = ['No Substructure', 'Sphere', 'Vortex']


# ── Remove stale hooks from any previous runs ─────────────────────────────────

def remove_all_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()
        module._backward_hooks.clear()
        module._forward_pre_hooks.clear()
        if hasattr(module, '_full_backward_hooks'):
            module._full_backward_hooks.clear()

remove_all_hooks(eq_densenet)
print('All existing hooks removed.')


# ── Classic CAM for EqDenseNet-C8 ────────────────────────────────────────────

class EqDenseNetCAM:
    """
    Classic CAM (Zhou et al., 2016) for EqDenseNet-C8.

    Hook target: group_pool output — last equivariant→standard conversion,
    immediately before AdaptiveAvgPool2d. No backpropagation required.

    classifier[2] is the Linear layer (after AdaptiveAvgPool + Flatten).
    Its weights (n_classes, n_features) serve as channel importance scores.
    """
    def __init__(self, model):
        self.model        = model
        self.feature_maps = None
        self._hooks       = []

        self._hooks.append(
            model.group_pool.register_forward_hook(self._save_features)
        )

    def _save_features(self, module, input, output):
        if hasattr(output, 'tensor'):
            self.feature_maps = output.tensor.detach()   # (1, C, H, W)
        else:
            self.feature_maps = output.detach()

    def generate(self, img_tensor, class_idx):
        """
        img_tensor : (1, 1, H, W) on device
        class_idx  : target class index
        Returns    : CAM heatmap (H, W) numpy array, normalised to [0, 1]
        """
        self.model.eval()
        with torch.no_grad():
            _ = self.model(img_tensor)

        # Linear weights: (n_classes, n_features)
        weights = self.model.classifier[2].weight.detach()   # (3, C)
        w       = weights[class_idx]                          # (C,)

        fmaps = self.feature_maps.squeeze(0)                  # (C, H, W)
        cam   = torch.einsum('c,chw->hw', w, fmaps).cpu().numpy()
        cam   = np.maximum(cam, 0)                            # ReLU

        cam = cv2.resize(cam, (150, 150))
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


# ── Overlay helper ────────────────────────────────────────────────────────────

def overlay_cam(img_np, cam, alpha=0.45):
    img_rgb = np.stack([img_np] * 3, axis=-1)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    overlay = alpha * heatmap + (1 - alpha) * img_rgb
    return np.clip(overlay, 0, 1)


# ── Image selection ───────────────────────────────────────────────────────────

def find_examples(model, val_ds, device, n_per_case=1):
    """
    For each class find n_per_case correctly and incorrectly classified images.
    Returns {cls_int: {'correct': [(idx, img, label, pred)], 'wrong': [...]}}
    """
    model.eval()
    examples = {c: {'correct': [], 'wrong': []} for c in range(3)}
    needed   = {c: {'correct': n_per_case, 'wrong': n_per_case} for c in range(3)}

    for idx in range(len(val_ds)):
        if all(needed[c][k] == 0
               for c in range(3) for k in ['correct', 'wrong']):
            break

        img, label = val_ds[idx]
        label = label.item() if hasattr(label, 'item') else label

        x = img.unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(x).argmax(1).item()

        key = 'correct' if pred == label else 'wrong'
        if needed[label][key] > 0:
            examples[label][key].append((idx, img, label, pred))
            needed[label][key] -= 1

    return examples


# ── Run ───────────────────────────────────────────────────────────────────────

print('Finding example images...')
examples = find_examples(eq_densenet, val_ds, device, n_per_case=1)
cam_obj  = EqDenseNetCAM(eq_densenet)


# ── Plot: 6 rows (3 classes × correct/wrong) × 3 cols (orig/cam/overlay) ─────

fig, axes = plt.subplots(6, 3, figsize=(10, 20))
fig.patch.set_facecolor('#0f0f0f')
plt.suptitle('EqDenseNet-C8 CAM — Correct vs Wrong per Class',
             fontsize=13, fontweight='bold', color='white', y=1.01)

for col, title in enumerate(['Original', 'Class Activation Map', 'Overlay']):
    axes[0, col].set_title(title, fontsize=10, color='#aaaaaa', pad=8)

row = 0
for cls_idx in range(3):
    for case_key, case_label in [('correct', '✓ Correct'), ('wrong', '✗ Wrong')]:
        if not examples[cls_idx][case_key]:
            for col in range(3):
                axes[row, col].set_visible(False)
            row += 1
            continue

        idx, img, label, pred = examples[cls_idx][case_key][0]
        img_tensor = img.unsqueeze(0).to(device)
        img_np     = img.squeeze().cpu().numpy()

        cam = cam_obj.generate(img_tensor, pred)

        color     = '#2ecc71' if case_key == 'correct' else '#e74c3c'
        row_title = (f'{CLASS_NAMES[cls_idx]}  {case_label}\n'
                     f'True: {CLASS_NAMES[label]}  '
                     f'Pred: {CLASS_NAMES[pred]}')

        axes[row, 0].imshow(img_np, cmap='inferno')
        axes[row, 0].set_ylabel(row_title, fontsize=8, color=color,
                                labelpad=8, rotation=0,
                                ha='right', va='center')
        axes[row, 1].imshow(cam, cmap='jet', vmin=0, vmax=1)
        axes[row, 2].imshow(overlay_cam(img_np, cam))

        for col in range(3):
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
            for spine in axes[row, col].spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(1.5)
            axes[row, col].set_facecolor('#1a1a1a')

        row += 1

plt.tight_layout()

save_path = os.path.join(gsoc_drive_path, 'figures', 'eqdensenet_c8_cam.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'Saved → {save_path}')

cam_obj.remove_hooks()

### 6.2 Ring Concentration: EqDenseNet-$C_8$

**Metric:** Ring concentration is the fraction of CAM mass inside a fixed Einstein-ring annulus. The mask covers $7040/22500 = 31.3\%$ of image pixels, so a spatially uniform CAM would score $0.313$. Scores are computed from the **predicted-class** CAM on $n=50$ images per class.

| Class | Mean $\pm$ Std |
|:--|:--:|
| No Substructure | $0.752 \pm 0.085$ |
| Sphere | $0.536 \pm 0.124$ |
| Vortex | $0.719 \pm 0.102$ |
| Random baseline | $0.313$ |

All three classes are well above the random baseline, so CAM mass is generally concentrated on the ring rather than the background. The main class difference is that **Sphere** has the lowest mean and the largest spread, indicating less consistent ring-focused activation than No Substructure or Vortex.

This is compatible with Sphere being the harder class, but the metric should be interpreted narrowly: it shows **where** CAM mass falls, not **which feature** is being used or whether the CAM fully explains the decision.

In [ ]:
# ── Ring concentration: EqDenseNet-C8 ────────────────────────────────────────
# Classic CAM ring concentration scores, n=50 images per class.
# Ring annulus mask and random baseline defined locally

import numpy as np
import matplotlib.pyplot as plt
import torch
import cv2
import os

CLASS_NAMES = ['No Substructure', 'Sphere', 'Vortex']


# ── Ring annulus mask ─────────────────────────────────────────────────────────

def ring_mask(size=150, inner_frac=0.15, outer_frac=0.65):
    """
    Boolean mask for Einstein ring annulus.
    inner_frac, outer_frac: fractions of image half-width.
    """
    cy, cx = size // 2, size // 2
    Y, X   = np.ogrid[:size, :size]
    dist   = np.sqrt((X - cx)**2 + (Y - cy)**2) / (size // 2)
    return (dist >= inner_frac) & (dist <= outer_frac)

RING             = ring_mask(size=150, inner_frac=0.15, outer_frac=0.65)
random_baseline  = RING.sum() / (150 * 150)
print(f'Ring annulus covers {RING.sum()} / {150*150} pixels '
      f'({100*random_baseline:.1f}%) — random baseline = {random_baseline:.3f}')


# ── Remove stale hooks ────────────────────────────────────────────────────────

def remove_all_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()
        module._backward_hooks.clear()
        module._forward_pre_hooks.clear()
        if hasattr(module, '_full_backward_hooks'):
            module._full_backward_hooks.clear()

remove_all_hooks(eq_densenet)


# ── Classic CAM for EqDenseNet-C8 ────────────────────────────────────────────

class EqDenseNetCAM:
    """
    Classic CAM (Zhou et al., 2016) for EqDenseNet-C8.
    Hook target: group_pool output — last equivariant→standard conversion.
    classifier[2] is the Linear layer after AdaptiveAvgPool + Flatten.
    """
    def __init__(self, model):
        self.model        = model
        self.feature_maps = None
        self._hooks       = []
        self._hooks.append(
            model.group_pool.register_forward_hook(self._save_features)
        )

    def _save_features(self, module, input, output):
        if hasattr(output, 'tensor'):
            self.feature_maps = output.tensor.detach()
        else:
            self.feature_maps = output.detach()

    def generate(self, img_tensor, class_idx):
        self.model.eval()
        with torch.no_grad():
            _ = self.model(img_tensor)

        weights = self.model.classifier[2].weight.detach()   # (3, C)
        w       = weights[class_idx]                          # (C,)
        fmaps   = self.feature_maps.squeeze(0)                # (C, H, W)
        cam     = torch.einsum('c,chw->hw', w, fmaps).cpu().numpy()
        cam     = np.maximum(cam, 0)
        cam     = cv2.resize(cam, (150, 150))
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


# ── Compute ring concentration scores ────────────────────────────────────────

N_COMPARE      = 50
ring_scores_eq = {c: [] for c in range(3)}
cam_ring       = EqDenseNetCAM(eq_densenet)

for target_class in range(3):
    count = 0
    for idx in range(len(val_ds)):
        if count >= N_COMPARE:
            break
        img, label = val_ds[idx]
        label = label.item() if hasattr(label, 'item') else label
        if label != target_class:
            continue

        x       = img.unsqueeze(0).to(device)
        cam_map = cam_ring.generate(x, target_class)

        score = cam_map[RING].sum() / (cam_map.sum() + 1e-8)
        ring_scores_eq[target_class].append(float(score))
        count += 1

cam_ring.remove_hooks()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'\n{"─"*72}')
print(f'{"Ring Concentration Score — EqDenseNet-C8 (mean ± std)":^72}')
print(f'{"Fraction of CAM mass inside Einstein ring annulus":^72}')
print(f'{"─"*72}')
print(f'{"Model":<16} {"No Substructure":>18} {"Sphere":>18} {"Vortex":>18}')
print(f'{"─"*72}')
row = []
for c in range(3):
    v = ring_scores_eq[c]
    row.append(f'{np.mean(v):.3f} ± {np.std(v):.3f}')
print(f'{"EqDenseNet-C8":<16} {row[0]:>18} {row[1]:>18} {row[2]:>18}')
print(f'{"─"*72}')
print(f'{"Random baseline":<16} {random_baseline:.3f}')
print(f'{"─"*72}')

# ── Distribution plot ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig.patch.set_facecolor('#0f0f0f')

for c in range(3):
    vals  = ring_scores_eq[c]
    color = ['#3498db', '#e74c3c', '#2ecc71'][c]
    axes[c].set_facecolor('#1a1a1a')
    axes[c].hist(vals, bins=20, alpha=0.75,
                 color=color, label='EqDenseNet-C8', density=True)
    axes[c].axvline(np.mean(vals), color=color,
                    linestyle='--', lw=2,
                    label=f'Mean: {np.mean(vals):.3f}')
    axes[c].axvline(random_baseline, color='white',
                    linestyle=':', lw=1.5,
                    label=f'Random: {random_baseline:.3f}')
    axes[c].set_title(CLASS_NAMES[c], fontsize=11,
                      fontweight='bold', color='white')
    axes[c].set_xlabel('Ring Concentration Score', fontsize=10,
                       color='#aaaaaa')
    axes[c].tick_params(colors='#aaaaaa')
    axes[c].spines['bottom'].set_color('#444444')
    axes[c].spines['left'].set_color('#444444')
    axes[c].spines['top'].set_visible(False)
    axes[c].spines['right'].set_visible(False)
    leg = axes[c].legend(fontsize=9)
    for text in leg.get_texts():
        text.set_color('#cccccc')

axes[0].set_ylabel('Density', fontsize=10, color='#aaaaaa')

plt.suptitle(
    f'EqDenseNet-C8 Ring Concentration Score  (n={N_COMPARE} per class)\n'
    'Fraction of Class Activation Map mass inside Einstein ring annulus',
    fontsize=11, fontweight='bold', color='white'
)
plt.tight_layout()

save_path = os.path.join(gsoc_drive_path, 'figures',
                         'eqdensenet_c8_ring_concentration.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'Saved → {save_path}')

### 6.3 Interpretability: ResNet-50 vs E-ResNet

This comparison asks whether the symmetry-aware E-ResNet shows a meaningfully different spatial emphasis from a standard residual network on the same correctly classified images.

**Method.**
- **ResNet-50:** Grad-CAM at `layer4[-1]`, the last residual block before global average pooling.
- **E-ResNet:** Classic CAM at `gpool`, the final equivariant-to-standard feature conversion before the classifier. This is used because the model has a direct GAP + linear-head structure, while gradient-based CAM through `escnn` layers is less reliable in practice.

Only images classified correctly by **both** models are shown, so the comparison is between two successful decisions rather than between correct and failed ones. The difference map highlights relative emphasis: red indicates stronger E-ResNet activation, and blue indicates stronger ResNet-50 activation.

These maps should be read as qualitative spatial summaries, not precise localization or proof of mechanism.

In [ ]:
from pytorch_grad_cam import GradCAM as LibGradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from scipy.ndimage import zoom

# ── Clear all stale hooks from previous runs ──────────────────────────────
def clear_all_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()
        module._backward_hooks.clear()
        module._forward_pre_hooks.clear()

clear_all_hooks(resnet50)
clear_all_hooks(e_resnet)
print('All hooks cleared.')

def resize_cam(cam, target_h, target_w):
    return zoom(cam, (target_h / cam.shape[0], target_w / cam.shape[1]), order=1)

# ── Classic CAM for E-ResNet (no backprop) ─────────────────────────────────
class CAMEResNet:
    def __init__(self, model):
        self.model       = model
        self.activations = None
        self._fwd = model.gpool.register_forward_hook(self._save)

    def _save(self, module, input, output):
        out = output.tensor if hasattr(output, 'tensor') else output
        self.activations = out.detach().clone()

    def generate(self, img_tensor, class_idx):
        self.model.eval()
        self.activations = None
        with torch.no_grad():
            _ = self.model(img_tensor)
        fc_weights = self.model.fc[2].weight.data[class_idx]
        acts = self.activations.squeeze(0)
        cam  = F.relu((fc_weights[:, None, None] * acts).sum(0)).cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

    def remove(self):
        self._fwd.remove()

cam_eresnet = CAMEResNet(e_resnet)

# ── Loop over all three classes ────────────────────────────────────────────
for target_class in range(3):

    # Step 1: find jointly correct sample — no GradCAM extractor active yet
    found = False
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        with torch.no_grad():
            res_out  = resnet50(images)
            eres_out = e_resnet(images)
        res_preds  = torch.argmax(res_out,  dim=1)
        eres_preds = torch.argmax(eres_out, dim=1)
        correct_mask = (
            (res_preds  == labels) &
            (eres_preds == labels) &
            (labels     == target_class)
        )
        if correct_mask.any():
            idx            = torch.where(correct_mask)[0][0]
            selected_image = images[idx].unsqueeze(0)
            true_label     = labels[idx].item()
            found          = True
            break

    if not found:
        print(f'No jointly-correct image for {CLASS_NAMES[target_class]}')
        continue

    # Step 2: instantiate GradCAM extractor NOW, after search is complete
    resnet_cam_extractor = LibGradCAM(
        model=resnet50,
        target_layers=[resnet50.layer4[-1]]
    )

    # Step 3: generate CAMs
    targets   = [ClassifierOutputTarget(true_label)]
    cam_rn    = resnet_cam_extractor(
                    input_tensor=selected_image,
                    targets=targets
                )[0]
    cam_ern   = cam_eresnet.generate(selected_image, true_label)

    # Step 4: clean up extractor immediately after use
    resnet_cam_extractor.__exit__(None, None, None)

    raw_img    = selected_image.squeeze().cpu().numpy()
    h, w       = raw_img.shape
    cam_rn_up  = resize_cam(cam_rn,  h, w)
    cam_ern_up = resize_cam(cam_ern, h, w)
    diff       = cam_ern_up - cam_rn_up

    # Step 5: plot
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(
        f'Spatial Attention — {CLASS_NAMES[true_label]}\n'
        'ResNet-50 (Grad-CAM) vs E-ResNet (CAM)',
        fontsize=12, fontweight='bold'
    )

    axes[0].imshow(raw_img, cmap='magma', origin='lower')
    axes[0].set_title('Original')
    axes[0].axis('off')

    axes[1].imshow(raw_img, cmap='gray', origin='lower')
    axes[1].imshow(cam_rn_up, cmap='jet', origin='lower', alpha=0.5)
    axes[1].set_title('ResNet-50  Grad-CAM')
    axes[1].axis('off')

    axes[2].imshow(raw_img, cmap='gray', origin='lower')
    axes[2].imshow(cam_ern_up, cmap='jet', origin='lower', alpha=0.5)
    axes[2].set_title('E-ResNet  CAM')
    axes[2].axis('off')

    axes[3].imshow(raw_img, cmap='gray', origin='lower')
    axes[3].imshow(diff, cmap='RdBu_r', origin='lower',
                   alpha=0.6, vmin=-1, vmax=1)
    axes[3].set_title('Difference\n(red = E-ResNet higher)')
    axes[3].axis('off')

    plt.tight_layout()
    plt.savefig(
        os.path.join(gsoc_drive_path, 'figures',
                     f'gradcam_{CLASS_NAMES[true_label].replace(" ", "_")}.png'),
        dpi=150, bbox_inches='tight'
    )
    plt.show()

    with torch.no_grad():
        p_rn  = torch.softmax(resnet50(selected_image),  dim=1).cpu().numpy()[0]
        p_ern = torch.softmax(e_resnet(selected_image),  dim=1).cpu().numpy()[0]
    print(f'{CLASS_NAMES[true_label]}:')
    print('  Softmax probability output of ResNet-50 : ' +
          '  '.join(f'{n}: {p:.3f}' for n, p in zip(CLASS_NAMES, p_rn)))
    print('  Softmax probability output of E-ResNet  : ' +
          '  '.join(f'{n}: {p:.3f}' for n, p in zip(CLASS_NAMES, p_ern)))

cam_eresnet.remove()
print('\nDone.')

#### 6.3.1 Results

All three examples are classified correctly by both models, but the confidence levels are not identical. ResNet-50 is highly confident on all three images, whereas E-ResNet is noticeably less certain on the Sphere example.

**No Substructure.** ResNet-50 places substantial activation in the interior region enclosed by the ring, while E-ResNet is more ring-aligned and spatially distributed along the arc structure. The difference map is consistent with this interior-versus-arc contrast.

**Sphere.** Both models attend to the left side of the image near the bright compact structure, but E-ResNet appears more spatially localized. ResNet-50 is more confident on this example, so the two maps should not be interpreted as equally strong evidence for the same decision rule.

**Vortex.** ResNet-50 again shows broader activation extending into the interior, whereas E-ResNet is more concentrated near the ring region associated with the asymmetric bright structure.

**Summary.** In these examples, E-ResNet appears more ring-focused and spatially compact, while ResNet-50 shows broader interior activation. This is a qualitative pattern in a small number of cases, not a general proof of mechanistic difference.

### 6.4 Disagreement Analysis — When the Models Diverge

Section 6.3 considered images that both models classify correctly. A complementary view is to examine cases where the models disagree, since these are more informative about their differing failure modes.

Each validation image is assigned to one of four categories:

| Case | ResNet-50 | E-ResNet | Interpretation |
|---|---|---|---|
| 1 | Correct | Correct | Both models succeed |
| 2 | Correct | Wrong | ResNet-50 succeeds, E-ResNet fails |
| 3 | Wrong | Correct | E-ResNet succeeds, ResNet-50 fails |
| 4 | Wrong | Wrong | Both models fail |

For each case and class, representative examples are selected from the largest confidence disagreements on the true class. These examples are useful for qualitative comparison, but they should be interpreted as illustrative rather than statistically representative of the full validation set.

In [ ]:
# ── Cell 6.1.2: Find disagreement cases — ResNet-50 vs E-ResNet ──────────────
# Categorise every val image into one of four cases:
#   Case 1: Both correct
#   Case 2: ResNet-50 correct, E-ResNet wrong
#   Case 3: E-ResNet correct, ResNet-50 wrong
#   Case 4: Both wrong

cases = {1: [], 2: [], 3: [], 4: []}

resnet50.eval()
e_resnet.eval()

for idx in range(len(val_ds)):
    img, label = val_ds[idx]
    x = img.unsqueeze(0).to(device)
    with torch.no_grad():
        rn_pred  = resnet50(x).argmax(1).item()
        ern_pred = e_resnet(x).argmax(1).item()

    rn_correct  = (rn_pred  == label)
    ern_correct = (ern_pred == label)

    if rn_correct and ern_correct:
        cases[1].append((idx, label, rn_pred, ern_pred))
    elif rn_correct and not ern_correct:
        cases[2].append((idx, label, rn_pred, ern_pred))
    elif not rn_correct and ern_correct:
        cases[3].append((idx, label, rn_pred, ern_pred))
    else:
        cases[4].append((idx, label, rn_pred, ern_pred))

print('Disagreement case counts across 7,500 val images:')
print(f'  Case 1 — Both correct             : {len(cases[1]):,}')
print(f'  Case 2 — ResNet-50 ✓  E-ResNet ✗  : {len(cases[2]):,}')
print(f'  Case 3 — E-ResNet ✓   ResNet-50 ✗  : {len(cases[3]):,}')
print(f'  Case 4 — Both wrong               : {len(cases[4]):,}')
print(f'\n  Total: {sum(len(v) for v in cases.values()):,}')

# Per-class breakdown for Cases 2, 3, 4
for case_id in [2, 3, 4]:
    print(f'\nCase {case_id} breakdown by true class:')
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        n = sum(1 for _, label, _, _ in cases[case_id] if label == cls_idx)
        print(f'  {cls_name:20s}: {n}')

In [ ]:
# ── Cell: Find disagreement cases — DenseNet-121 vs EqDenseNet-C8 ────────────
# Categorise every val image into one of four cases:
#   Case 1: Both correct
#   Case 2: DenseNet-121 correct, EqDenseNet-C8 wrong
#   Case 3: EqDenseNet-C8 correct, DenseNet-121 wrong
#   Case 4: Both wrong

cases = {1: [], 2: [], 3: [], 4: []}

densenet.eval()
eq_densenet.eval()

for idx in range(len(val_ds)):
    img, label = val_ds[idx]
    x = img.unsqueeze(0).to(device)
    with torch.no_grad():
        dn_pred  = densenet(x).argmax(1).item()
        eqdn_pred = eq_densenet(x).argmax(1).item()

    dn_correct   = (dn_pred  == label)
    eqdn_correct = (eqdn_pred == label)

    if dn_correct and eqdn_correct:
        cases[1].append((idx, label, dn_pred, eqdn_pred))
    elif dn_correct and not eqdn_correct:
        cases[2].append((idx, label, dn_pred, eqdn_pred))
    elif not dn_correct and eqdn_correct:
        cases[3].append((idx, label, dn_pred, eqdn_pred))
    else:
        cases[4].append((idx, label, dn_pred, eqdn_pred))

print('Disagreement case counts across 7,500 val images:')
print(f'  Case 1 — Both correct                  : {len(cases[1]):,}')
print(f'  Case 2 — DenseNet-121 ✓  EqDenseNet ✗  : {len(cases[2]):,}')
print(f'  Case 3 — EqDenseNet ✓   DenseNet-121 ✗  : {len(cases[3]):,}')
print(f'  Case 4 — Both wrong                    : {len(cases[4]):,}')
print(f'\n  Total: {sum(len(v) for v in cases.values()):,}')

# Per-class breakdown for Cases 2, 3, 4
for case_id in [2, 3, 4]:
    print(f'\nCase {case_id} breakdown by true class:')
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        n = sum(1 for _, label, _, _ in cases[case_id] if label == cls_idx)
        print(f'  {cls_name:20s}: {n}')

The disagreement distribution is highly imbalanced. Most validation images fall in **Case 1**: both models are correct on $7198/7500 = 96.0\%$ of samples. The disagreement and joint-failure cases are relatively rare, but still informative: $68$ images in Case 2, $103$ in Case 3, and $131$ in Case 4.

The class breakdown shows that **Sphere** is overrepresented in all failure-related categories. In particular:
- Case 2: $34/68 = 50.0\%$
- Case 3: $55/103 = 53.4\%$
- Case 4: $104/131 = 79.4\%$

The strongest concentration is in **Case 4**, where nearly four-fifths of joint failures are Sphere images. This indicates that Sphere is the main source of residual difficulty for both models, and motivates the dedicated class-specific analysis in Section 7. This pattern is descriptive rather than causal, but it is consistent with Sphere being the hardest class in the benchmark.

In [ ]:
# ── Select representatives for Cases 2, 3, 4 ─────────────────────
# For each of Cases 2, 3, 4 find one representative image per true class.
# Preference: select the image where the confidence gap between the two
# models is largest — these are the most illustrative disagreements.

import torch.nn.functional as F

def get_confidence_gap(idx, model_a, model_b):
    """
    Returns |conf_a(true) - conf_b(true)| for the true class.
    Higher = more dramatic disagreement.
    """
    img, label = val_ds[idx]
    x = img.unsqueeze(0).to(device)
    with torch.no_grad():
        pa = F.softmax(model_a(x), dim=1)[0, label].item()
        pb = F.softmax(model_b(x), dim=1)[0, label].item()
    return abs(pa - pb), label

# Build representatives dict: {case_id: {class_idx: (idx, label, rn_pred, ern_pred)}}
representatives = {2: {}, 3: {}, 4: {}}

for case_id in [2, 3, 4]:
    for cls_idx in range(3):
        candidates = [(idx, l, rp, ep) for idx, l, rp, ep in cases[case_id]
                      if l == cls_idx]
        if not candidates:
            print(f'  Case {case_id}, {CLASS_NAMES[cls_idx]}: no examples found')
            continue
        # Sort by confidence gap — pick the most dramatic disagreement
        candidates_scored = []
        for item in candidates:
            gap, _ = get_confidence_gap(item[0], resnet50, e_resnet)
            candidates_scored.append((gap, item))
        candidates_scored.sort(reverse=True)
        best = candidates_scored[0][1]
        representatives[case_id][cls_idx] = best
        print(f'  Case {case_id} | {CLASS_NAMES[cls_idx]:20s} '
              f'| val idx {best[0]:5d} '
              f'| true={CLASS_NAMES[best[1]]} '
              f'| RN50={CLASS_NAMES[best[2]]} '
              f'| ERN={CLASS_NAMES[best[3]]}')

In [ ]:
# ──Grad-CAM visualisation for Cases 2, 3, 4 ─────────────────────
# For each case and each true class: show original + ResNet-50 Grad-CAM +
# E-ResNet CAM + difference map.
# Confidence scores printed below each figure.

clear_all_hooks(resnet50)
clear_all_hooks(e_resnet)

cam_eresnet_dis = CAMEResNet(e_resnet)

case_labels = {
    2: 'Case 2 — ResNet-50 ✓  |  E-ResNet ✗',
    3: 'Case 3 — E-ResNet ✓   |  ResNet-50 ✗',
    4: 'Case 4 — Both Wrong',
}

for case_id in [2, 3, 4]:
    if not representatives[case_id]:
        print(f'Case {case_id}: no representatives found, skipping.')
        continue

    n_cols_fig = len(representatives[case_id])
    fig, axes  = plt.subplots(
        n_cols_fig, 4,
        figsize=(18, 4.5 * n_cols_fig)
    )
    # Handle single-row case
    if n_cols_fig == 1:
        axes = axes[np.newaxis, :]

    for row, (cls_idx, (idx, true_label, rn_pred, ern_pred)) in \
            enumerate(sorted(representatives[case_id].items())):

        img, _ = val_ds[idx]
        x      = img.unsqueeze(0).to(device)

        # Grad-CAM for ResNet-50
        resnet_cam_ex = LibGradCAM(
            model=resnet50,
            target_layers=[resnet50.layer4[-1]]
        )
        targets  = [ClassifierOutputTarget(true_label)]
        cam_rn   = resnet_cam_ex(input_tensor=x, targets=targets)[0]
        resnet_cam_ex.__exit__(None, None, None)

        # CAM for E-ResNet
        cam_ern = cam_eresnet_dis.generate(x, true_label)

        raw_img    = img.squeeze().cpu().numpy()
        h, w       = raw_img.shape
        cam_rn_up  = resize_cam(cam_rn,  h, w)
        cam_ern_up = resize_cam(cam_ern, h, w)
        diff       = cam_ern_up - cam_rn_up

        # Confidence scores
        with torch.no_grad():
            p_rn  = F.softmax(resnet50(x),  dim=1).cpu().numpy()[0]
            p_ern = F.softmax(e_resnet(x),  dim=1).cpu().numpy()[0]

        # Row label
        axes[row, 0].set_ylabel(
            f'True: {CLASS_NAMES[true_label]}\n'
            f'RN50→{CLASS_NAMES[rn_pred]}\n'
            f'ERN→{CLASS_NAMES[ern_pred]}',
            fontsize=9, fontweight='bold', rotation=90, labelpad=8
        )

        # Col 0 — original
        axes[row, 0].imshow(raw_img, cmap='magma', origin='lower')
        axes[row, 0].set_title('Original', fontsize=10)
        axes[row, 0].axis('off')

        # Col 1 — ResNet-50 Grad-CAM
        axes[row, 1].imshow(raw_img, cmap='gray', origin='lower')
        axes[row, 1].imshow(cam_rn_up, cmap='jet', origin='lower', alpha=0.5)
        axes[row, 1].set_title(
            f'ResNet-50  Grad-CAM\n'
            f'pred={CLASS_NAMES[rn_pred]}  '
            f'conf={p_rn[true_label]:.3f}',
            fontsize=9
        )
        axes[row, 1].axis('off')

        # Col 2 — E-ResNet CAM
        axes[row, 2].imshow(raw_img, cmap='gray', origin='lower')
        axes[row, 2].imshow(cam_ern_up, cmap='jet', origin='lower', alpha=0.5)
        axes[row, 2].set_title(
            f'E-ResNet  CAM\n'
            f'pred={CLASS_NAMES[ern_pred]}  '
            f'conf={p_ern[true_label]:.3f}',
            fontsize=9
        )
        axes[row, 2].axis('off')

        # Col 3 — Difference map
        axes[row, 3].imshow(raw_img, cmap='gray', origin='lower')
        axes[row, 3].imshow(
            diff, cmap='RdBu_r', origin='lower',
            alpha=0.6, vmin=-1, vmax=1
        )
        axes[row, 3].set_title(
            'Difference\n(red=E-ResNet higher)',
            fontsize=9
        )
        axes[row, 3].axis('off')

    fig.suptitle(
        f'{case_labels[case_id]}\n'
        f'Grad-CAM on true class label — one row per true class',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(gsoc_drive_path, 'figures',
                     f'fig6_1_case{case_id}_gradcam.png'),
        dpi=150, bbox_inches='tight'
    )
    plt.show()
    print(f'Case {case_id} figure saved.')

cam_eresnet_dis.remove()
clear_all_hooks(resnet50)
clear_all_hooks(e_resnet)
print('\nAll hooks cleared.')

####6.4.1 Case 2 — ResNet-50 Correct, E-ResNet Wrong

These are images where ResNet-50 is correct and E-ResNet is incorrect.
The selected representatives are the most extreme disagreements by
confidence gap — cases where ResNet-50 is highly confident and correct
while E-ResNet is confidently wrong.

#### 6.4.1 Case 2 — ResNet-50 Correct, E-ResNet Wrong

In these examples, ResNet-50 is correct and E-ResNet is incorrect, so the comparison is useful for identifying failure modes that are more prominent in E-ResNet on this split.

- **No Substructure:** ResNet-50 shows broad ring-centered activation and predicts the correct class with high confidence. E-ResNet also attends to the ring, but with a more fragmented spatial pattern and an incorrect class prediction.
- **Sphere:** ResNet-50 places strong activation near the compact bright structure and is confidently correct. E-ResNet shows weaker, more dispersed activation and assigns lower confidence to the true class.
- **Vortex:** ResNet-50 again produces the correct decision, while E-ResNet concentrates more locally and predicts the wrong class.

Across these three examples, E-ResNet tends to show more spatially fragmented or locally concentrated activation than ResNet-50. That pattern may be relevant to its errors in these cases, but the sample is too small for a stronger mechanistic claim. These examples should therefore be read as illustrative failure cases rather than proof of a general property of equivariant models.

####6.4.2 Case 3 — E-ResNet Correct, ResNet-50 Wrong

These are images where E-ResNet is correct and ResNet-50 is incorrect.
The selected representatives are the most extreme disagreements by
confidence gap — cases where E-ResNet is highly confident and correct
while ResNet-50 is confidently wrong.

#### 6.4.2 Case 3 — E-ResNet Correct, ResNet-50 Wrong

These examples show the opposite pattern: E-ResNet is correct and ResNet-50 is incorrect. They are therefore useful for identifying cases in which the equivariant model may have an advantage on this validation split.

- **No Substructure:** ResNet-50 assigns the wrong class with low confidence and shows broad, weakly focused activation. E-ResNet is correctly confident and places more structured attention along the ring.
- **Sphere:** This is the clearest disagreement. E-ResNet is confidently correct, while ResNet-50 predicts the wrong class. E-ResNet shows more localized activation near compact arc features, whereas ResNet-50 is broader and less specific.
- **Vortex:** E-ResNet again produces the correct class with concentrated activation near the asymmetric ring region, while ResNet-50 remains diffuse and incorrect.

Across these examples, E-ResNet appears more spatially selective and more ring-aligned, while ResNet-50 is broader and less class-specific. This is consistent with the equivariant model being more effective on some perturbation-dominated cases, but the examples remain qualitative. They do not by themselves establish the mechanism behind the difference.

####6.4.3 Case 4 — Both Wrong

These are images where neither architecture produces a correct prediction.
118 of the 152 both-wrong images are Sphere, consistent with the
pattern of Sphere being the most difficult class across all architectures.

#### 6.4.3 Case 4 — Both Wrong

These examples are the most difficult to interpret, because both models fail. They are useful mainly for showing what the attention maps look like when neither architecture reaches the correct decision.

- **No Substructure:** Both models produce a false positive. E-ResNet remains relatively ring-focused, while ResNet-50 is broader and more diffuse.
- **Sphere:** Both models miss the true class. E-ResNet shows localized activation on parts of the arc, whereas ResNet-50 is less spatially specific, but neither response is sufficient for correct classification.
- **Vortex:** Both models confuse the image with another substructure class. E-ResNet is again more concentrated near the ring, while ResNet-50 is broader.

Across these examples, E-ResNet remains more spatially compact and ring-aligned than ResNet-50, even when both are wrong. That does **not** imply that the remaining errors are purely due to low signal-to-noise or that the model is using the correct feature for the correct reason. It only shows that more focused spatial attention is not, by itself, sufficient for correct classification. The question of whether these failures are associated with lower signal strength is addressed separately in Section 7.

#### 6.4.4 Disagreement Summary

Three patterns emerge from the disagreement analysis.

**1. The two models have overlapping but non-identical error sets.**  
Most validation images are classified correctly by both models ($7198/7500 = 96.0\%$), but there are still meaningful disagreement regions: 68 images in Case 2, 103 in Case 3, and 131 in Case 4. This indicates that the models are not failing on exactly the same subset of images.

**2. Sphere is overrepresented in the disagreement cases.**  
Sphere accounts for $34/68 = 50.0\%$ of Case 2 and $55/103 = 53.4\%$ of Case 3. This is consistent with Sphere being the class on which the two architectures differ most in practice.

**3. Joint failures are dominated by Sphere.**  
In Case 4, $104/131 = 79.4\%$ of the images are Sphere. This does not by itself establish why those images are difficult, but it shows that the residual failure set is concentrated heavily in the hardest class. The relationship to image quality and signal strength is examined separately in Section 7.

### <a id="65-interpretability-vit-vs-densenet"></a>6.5 Interpretability: ViT vs DenseNet

This section compares two strong but architecturally different models:

- **DenseNet-121** — convolutional, with dense feature reuse  
- **ViT-Base** — transformer-based, with patch self-attention

The visualizations are **not directly equivalent**. DenseNet is analyzed with **Grad-CAM**, which is class-discriminative and highlights regions that influence a specific prediction. ViT is analyzed with **attention rollout**, which is not class-specific and instead summarizes how attention propagates from the CLS token through the image patches.

Because these methods answer different questions, the comparison is qualitative rather than one-to-one. To keep the interpretation grounded, a simple ring-concentration metric is also reported as a shared quantitative reference.

This section contains five figures:

| Figure | Content |
|---|---|
| 6.2a | Spatial attention grid: original / DenseNet Grad-CAM / ViT rollout |
| 6.2b | DenseNet resolution diagnostic |
| 6.2c | ViT ring concentration by class |
| 6.2d | ViT attention consistency across images |
| 6.2e | Ring concentration comparison: DenseNet vs ViT |

#### 6.5.1 Setup: ViT Attention Patching and Attention Rollout

Torchvision's ViT does not return attention weights by default, so the attention modules are patched to retain them during the forward pass. These stored weights are then used to compute **attention rollout**, which combines attention across layers while accounting for residual connections, yielding a single spatial map over input patches.

Two implementation details are important. First, the identity matrix used in the rollout must be created on the same device as the attention tensors to avoid CPU/GPU mismatch errors. Second, the final attention map must be moved to CPU before conversion to NumPy.

The result is a class-agnostic patch-level saliency map that summarizes how information from the image contributes to the final CLS-token representation.

In [ ]:
# ViT attention patching and AttentionRollout ───────────────────────

def patch_vit_attention(model):
    for module in model.modules():
        if isinstance(module, nn.MultiheadAttention):
            module.saved_attn_weights = None
            original_forward = module.forward

            def make_patched(orig, mod):
                def patched_forward(query, key, value, **kwargs):
                    kwargs['need_weights']         = True
                    kwargs['average_attn_weights'] = False
                    out, attn = orig(query, key, value, **kwargs)
                    mod.saved_attn_weights = attn.detach()
                    return out, attn
                return patched_forward

            module.forward = make_patched(original_forward, module)
    print(f'patch_vit_attention: patched {sum(1 for m in model.modules() if isinstance(m, nn.MultiheadAttention))} MultiheadAttention modules.')


class AttentionRollout:
    def __init__(self, model, head_fusion='mean', discard_ratio=0.9):
        self.model         = model
        self.head_fusion   = head_fusion
        self.discard_ratio = discard_ratio

    def __call__(self, x):
        self.model.eval()
        with torch.no_grad():
            _ = self.model(x)

        attn_weights = []
        for module in self.model.modules():
            if (isinstance(module, nn.MultiheadAttention)
                    and module.saved_attn_weights is not None):
                attn_weights.append(module.saved_attn_weights)

        if not attn_weights:
            raise RuntimeError(
                'No attention weights found. '
                'Call patch_vit_attention before AttentionRollout.'
            )

        device = attn_weights[0].device
        result = torch.eye(attn_weights[0].shape[-1], device=device)

        for attn in attn_weights:
            if self.head_fusion == 'mean':
                attn_fused = attn[0].mean(dim=0)
            elif self.head_fusion == 'max':
                attn_fused = attn[0].max(dim=0).values
            else:
                attn_fused = attn[0].min(dim=0).values

            flat       = attn_fused.view(-1)
            threshold  = flat.quantile(self.discard_ratio)
            attn_fused = attn_fused * (attn_fused > threshold).float()

            attn_fused = attn_fused + torch.eye(
                attn_fused.shape[-1], device=device
            )
            attn_fused = attn_fused / (
                attn_fused.sum(dim=-1, keepdim=True) + 1e-8
            )
            result = torch.matmul(attn_fused, result)

        mask        = result[0, 1:].cpu()
        num_patches = mask.shape[0]
        size        = int(num_patches ** 0.5)
        return mask.reshape(size, size).numpy()


print('AttentionRollout ready.')

####6.5.2 Initialisation

All hooks from previous sections are cleared before initialising new ones.
ViT is patched for attention weight capture and a `rollout` instance is
created. DenseNet Grad-CAM is pointed at `denseblock4` — the final dense
block before the transition layer, which has the largest spatial resolution
still available before the classifier head. A sanity check confirms the
rollout produces a valid output map.

In [ ]:
# ──Initialise ViT rollout + DenseNet Grad-CAM ───────────────────────
import cv2
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# Clear all hooks from any previous runs
clear_all_hooks(vit_model)
clear_all_hooks(densenet)

# Patch ViT and instantiate rollout
patch_vit_attention(vit_model)
vit_model.eval()
rollout = AttentionRollout(vit_model, head_fusion='mean', discard_ratio=0.9)

# Instantiate DenseNet Grad-CAM
densenet.eval()
densenet_cam = GradCAM(
    model=densenet,
    target_layers=[densenet.features.denseblock4]
)

# Sanity check rollout
_img, _ = val_ds[0]
_x      = _img.unsqueeze(0).to(device)
_map    = rollout(_x)
print(f'Rollout sanity check: output shape {_map.shape}, '
      f'range [{_map.min():.3f}, {_map.max():.3f}]')
del _img, _x, _map

print('All tools initialised.')

#### 6.5.3 Image Selection

One image per class is selected from the validation set subject to the
constraint that both ViT and DenseNet predict the correct class. This
joint-correctness criterion ensures that all heatmaps shown in Figures
6.2a and 6.2b reflect genuine learned features rather than the attention
patterns of a confused model, which would be uninformative for
interpretability purposes.

In [ ]:
# ──Select one jointly-correct image per class ───────────────────────
# Both ViT and DenseNet must predict correctly.
# Ensures heatmaps reflect genuine learned features, not confusion artefacts.

sample_images = {}

for target_class in range(3):
    for idx in range(len(val_ds)):
        img, label = val_ds[idx]
        if label != target_class:
            continue
        x = img.unsqueeze(0).to(device)
        with torch.no_grad():
            vit_pred = vit_model(x).argmax(1).item()
            dn_pred  = densenet(x).argmax(1).item()
        if vit_pred == target_class and dn_pred == target_class:
            sample_images[target_class] = (img, idx)
            break

print('Selected images (both models correct):')
for c, (img, idx) in sample_images.items():
    print(f'  {CLASS_NAMES[c]:20s}  val idx {idx}')

####6.5.4 Spatial Attention Grid

Three representative images — one per class, jointly correct for both models
— are shown alongside their DenseNet Grad-CAM and ViT Attention Rollout maps.
The two attention columns are intentionally not placed side-by-side for
comparison; they are presented together only for visual convenience. Their
interpretation must remain separate.

In [ ]:
# ── Spatial attention grid ─────────────────────────────
# 3×3 grid: Original / DenseNet Grad-CAM / ViT Attention Rollout
# One row per class. No difference map — methods are not comparable quantities.

fig, axes = plt.subplots(3, 3, figsize=(14, 14))

col_titles = [
    'Original Image',
    'DenseNet-121  Grad-CAM\n(class-discriminative gradient signal)',
    'ViT-Base  Attention Rollout\n(class-agnostic patch salience)'
]

for c, (img, idx) in sample_images.items():
    img_np  = img.squeeze().cpu().numpy()
    img_01  = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    img_rgb = np.stack([img_01] * 3, axis=-1).astype(np.float32)
    x       = img.unsqueeze(0).to(device)

    # Col 0 — original
    axes[c, 0].imshow(img_np, cmap='inferno')
    axes[c, 0].set_ylabel(CLASS_NAMES[c], fontsize=12,
                           fontweight='bold', rotation=90, labelpad=10)
    axes[c, 0].set_xticks([]); axes[c, 0].set_yticks([])

    # Col 1 — DenseNet Grad-CAM
    cam_map = densenet_cam(input_tensor=x, targets=None)[0]
    cam_map = (cam_map - cam_map.min()) / (cam_map.max() - cam_map.min() + 1e-8)
    dn_overlay = show_cam_on_image(img_rgb, cam_map, use_rgb=True)
    axes[c, 1].imshow(dn_overlay)
    axes[c, 1].set_xticks([]); axes[c, 1].set_yticks([])

    # Col 2 — ViT Attention Rollout
    attn_map = rollout(x)
    attn_150 = cv2.resize(attn_map, (150, 150),
                          interpolation=cv2.INTER_LINEAR)
    attn_150 = (attn_150 - attn_150.min()) / (
        attn_150.max() - attn_150.min() + 1e-8)
    axes[c, 2].imshow(img_np, cmap='gray')
    axes[c, 2].imshow(attn_150, cmap='jet', alpha=0.55)
    axes[c, 2].set_xticks([]); axes[c, 2].set_yticks([])

for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=10, fontweight='bold', pad=8)

plt.suptitle(
    'Figure 6.2a — Spatial Attention: DenseNet-121 vs ViT-Base\n'
    'Grad-CAM and Attention Rollout measure different quantities '
    'and cannot be compared directly',
    fontsize=11, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures', 'fig6_2a_attention_grid.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Figure 6.2a saved.')

Figure 6.2a should be read mainly as a **method comparison**, not a model ranking. DenseNet Grad-CAM is very coarse in all three rows, whereas ViT attention rollout is visibly sharper and more structured. That visual contrast is expected: DenseNet's final convolutional features are extremely low-resolution before upsampling, while ViT operates on a fixed patch grid and therefore retains a finer spatial layout.

The key point is interpretive caution. The sharper ViT maps do **not** imply better class reasoning, and the diffuse DenseNet maps do **not** imply that DenseNet ignores the ring. The two visualization methods measure different quantities, and the spatial resolution of the underlying features is very different. This figure therefore supports only a limited conclusion: ViT rollout yields finer-looking spatial maps, while DenseNet Grad-CAM is strongly resolution-limited. The question of whether either model is more ring-focused is addressed separately with quantitative analysis.

#### 6.5.5 DenseNet Resolution Diagnostic

The coarse appearance of DenseNet Grad-CAM in Figure 6.2a is primarily a resolution effect. By the final DenseNet block used for Grad-CAM, the spatial feature map is extremely small relative to the $150\times150$ input. After weighting and upsampling, the result is inevitably smooth and blob-like.

This does **not** mean DenseNet is ignoring the Einstein ring. It means the visualization is being produced from a highly compressed spatial representation. The figure makes this explicit by showing the native low-resolution map, the upsampled version, and the final overlay.

In [ ]:
# ── Figure 6.2b — DenseNet resolution diagnostic ─────────────────────
# Shows raw denseblock4 feature map at native resolution vs upsampled.
# Purpose: make explicit that the Grad-CAM blob is a resolution artefact.

raw_features = {}

def capture_features(module, input, output):
    raw_features['denseblock4'] = output.detach().cpu()

hook_handle = densenet.features.denseblock4.register_forward_hook(
    capture_features
)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

diag_col_titles = [
    'Original (150×150)',
    'Raw denseblock4 output\n(mean across channels)',
    'Bicubic upsampled\nto 150×150',
    'Grad-CAM overlay\n(weighted activation)'
]

for c, (img, idx) in sample_images.items():
    img_np  = img.squeeze().cpu().numpy()
    img_01  = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    img_rgb = np.stack([img_01] * 3, axis=-1).astype(np.float32)
    x       = img.unsqueeze(0).to(device)

    with torch.no_grad():
        _ = densenet(x)

    raw      = raw_features['denseblock4']
    raw_mean = raw[0].mean(dim=0).numpy()
    h, w     = raw_mean.shape

    raw_up = cv2.resize(raw_mean, (150, 150), interpolation=cv2.INTER_CUBIC)
    raw_up = (raw_up - raw_up.min()) / (raw_up.max() - raw_up.min() + 1e-8)

    cam_map = densenet_cam(input_tensor=x, targets=None)[0]
    cam_map = (cam_map - cam_map.min()) / (cam_map.max() - cam_map.min() + 1e-8)
    overlay = show_cam_on_image(img_rgb, cam_map, use_rgb=True)

    # Col 0 — original
    axes[c, 0].imshow(img_np, cmap='inferno')
    axes[c, 0].set_ylabel(CLASS_NAMES[c], fontsize=11,
                           fontweight='bold', rotation=90, labelpad=10)
    axes[c, 0].set_xticks([]); axes[c, 0].set_yticks([])

    # Col 1 — raw feature map at native resolution with pixel grid
    axes[c, 1].imshow(raw_mean, cmap='viridis', interpolation='nearest')
    axes[c, 1].set_xticks([]); axes[c, 1].set_yticks([])
    if c == 0:
        axes[c, 1].set_title(
            f'Native resolution: {h}×{w} pixels',
            fontsize=8, color='gray'
        )
    for i in range(h + 1):
        axes[c, 1].axhline(i - 0.5, color='white', lw=0.5, alpha=0.5)
    for j in range(w + 1):
        axes[c, 1].axvline(j - 0.5, color='white', lw=0.5, alpha=0.5)

    # Col 2 — bicubic upsampled
    axes[c, 2].imshow(raw_up, cmap='viridis')
    axes[c, 2].set_xticks([]); axes[c, 2].set_yticks([])

    # Col 3 — Grad-CAM overlay
    axes[c, 3].imshow(overlay)
    axes[c, 3].set_xticks([]); axes[c, 3].set_yticks([])

for j, title in enumerate(diag_col_titles):
    axes[0, j].set_title(title, fontsize=10, fontweight='bold', pad=8)

hook_handle.remove()

plt.suptitle(
    'Figure 6.2b — DenseNet-121 Resolution Diagnostic\n'
    'The blob appearance of Grad-CAM is a mathematical consequence of '
    'spatial compression at denseblock4\nnot evidence that DenseNet ignores ring structure',
    fontsize=11, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures', 'fig6_2b_densenet_resolution.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Figure 6.2b saved.')

The native DenseNet maps are extremely low resolution, with only a small number of spatial cells carrying the final class-discriminative signal. Different images still produce different activation patterns at this stage, so the representation is not uninformative; the limitation is spatial precision, not absence of signal.

After bicubic upsampling, these low-resolution maps inevitably become smooth blobs. That blur is therefore a consequence of spatial compression, not evidence that DenseNet ignores ring structure.

**Interpretation:** Grad-CAM at this depth is useful only as a coarse regional diagnostic. It is not reliable for fine localization on the original $150\times150$ image. For DenseNet, quantitative summaries such as ring-concentration are more informative than visual sharpness alone.

#### 6.5.6 ViT Ring Concentration and Attention Consistency

To move beyond a few hand-picked examples, two simple aggregate summaries are used.

**Figure 6.2c** reports the **ring concentration score**: the fraction of total attention-rollout mass inside a fixed annulus centered on the image. The mask covers $19.0\%$ of pixels, so a spatially uniform attention map would score $0.190$. Scores are computed on $n=50$ images per class.

**Figure 6.2d** reports **attention consistency** across images of the same class using the pixelwise standard deviation of rollout maps over $n=20$ images per class. Lower standard deviation indicates a more stable spatial attention pattern; higher standard deviation indicates greater image-to-image variation.

These metrics are descriptive. They summarize where attention tends to fall and how stable it is across images, but they do not by themselves establish which features drive the final class decision.

In [ ]:
# ──Figures 6.2c and 6.2d ─────────────────────────────────────────────
# 6.2c — ViT ring concentration score by class (n=50 per class)
# 6.2d — ViT attention consistency (mean + std across 20 images per class)

def ring_mask(size=150, inner_frac=0.25, outer_frac=0.55):
    cy, cx = size // 2, size // 2
    Y, X   = np.ogrid[:size, :size]
    dist   = np.sqrt((X - cx)**2 + (Y - cy)**2) / (size // 2)
    return (dist >= inner_frac) & (dist <= outer_frac)

ring             = ring_mask()
random_baseline  = ring.mean()
print(f'Ring annulus: {random_baseline*100:.1f}% of pixels '
      f'(random attention baseline = {random_baseline:.3f})\n')

# ── Figure 6.2c — ViT ring concentration ──────────────────────────────────────
N_RING    = 50
vit_ring  = {c: [] for c in range(3)}

for target_class in range(3):
    count = 0
    for idx in range(len(val_ds)):
        if count >= N_RING:
            break
        img, label = val_ds[idx]
        if label != target_class:
            continue
        x        = img.unsqueeze(0).to(device)
        attn_map = rollout(x)
        attn_150 = cv2.resize(attn_map, (150, 150))
        attn_150 = (attn_150 - attn_150.min()) / (
            attn_150.max() - attn_150.min() + 1e-8)
        vit_ring[target_class].append(
            float(attn_150[ring].sum() / (attn_150.sum() + 1e-8))
        )
        count += 1
    print(f'  {CLASS_NAMES[target_class]:20s} ring score: '
          f'mean={np.mean(vit_ring[target_class]):.3f} '
          f'± {np.std(vit_ring[target_class]):.3f}')

colors_cls = ['#3498db', '#e74c3c', '#2ecc71']
fig, axes  = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for c in range(3):
    vals = vit_ring[c]
    axes[c].hist(vals, bins=20, color=colors_cls[c],
                 alpha=0.75, density=True, label=CLASS_NAMES[c])
    axes[c].axvline(np.mean(vals), color=colors_cls[c],
                    linestyle='--', lw=2.5,
                    label=f'Mean: {np.mean(vals):.3f}')
    axes[c].axvline(random_baseline, color='black',
                    linestyle=':', lw=2,
                    label=f'Random baseline: {random_baseline:.3f}')
    axes[c].set_title(CLASS_NAMES[c], fontsize=12, fontweight='bold')
    axes[c].set_xlabel('Ring Concentration Score', fontsize=10)
    axes[c].legend(fontsize=8)

axes[0].set_ylabel('Density', fontsize=10)
plt.suptitle(
    f'Figure 6.2c — ViT-Base Ring Concentration Score by Class  (n={N_RING} per class)\n'
    'Fraction of attention rollout mass inside Einstein ring annulus\n'
    'Sphere and Vortex show higher ring concentration than No Substructure — physically motivated',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures', 'fig6_2c_vit_ring_concentration.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Figure 6.2c saved.\n')

# ── Figure 6.2d — ViT attention consistency ────────────────────────────────────
N_CONSISTENCY = 20
attn_stacks   = {c: [] for c in range(3)}

for target_class in range(3):
    count = 0
    for idx in range(len(val_ds)):
        if count >= N_CONSISTENCY:
            break
        img, label = val_ds[idx]
        if label != target_class:
            continue
        x        = img.unsqueeze(0).to(device)
        attn_map = rollout(x)
        attn_150 = cv2.resize(attn_map, (150, 150))
        attn_150 = (attn_150 - attn_150.min()) / (
            attn_150.max() - attn_150.min() + 1e-8)
        attn_stacks[target_class].append(attn_150)
        count += 1

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for c in range(3):
    stack    = np.stack(attn_stacks[c], axis=0)
    mean_map = stack.mean(axis=0)
    std_map  = stack.std(axis=0)

    im0 = axes[0, c].imshow(mean_map, cmap='jet')
    axes[0, c].set_title(CLASS_NAMES[c], fontsize=12, fontweight='bold')
    axes[0, c].set_xticks([]); axes[0, c].set_yticks([])
    plt.colorbar(im0, ax=axes[0, c], shrink=0.8)

    im1 = axes[1, c].imshow(std_map, cmap='hot')
    axes[1, c].set_xticks([]); axes[1, c].set_yticks([])
    axes[1, c].set_xlabel(
        f'Mean std: {std_map.mean():.4f}  (lower = more consistent)',
        fontsize=9
    )
    plt.colorbar(im1, ax=axes[1, c], shrink=0.8)

axes[0, 0].set_ylabel('Mean Attention\n(n=20 images)',
                        fontsize=10, fontweight='bold')
axes[1, 0].set_ylabel('Attention Std Dev\n(consistency)',
                        fontsize=10, fontweight='bold')

plt.suptitle(
    f'Figure 6.2d — ViT-Base Attention Consistency  (n={N_CONSISTENCY} per class)\n'
    'Mean and standard deviation of attention rollout across images\n'
    'Low std = stable spatial strategy  |  High std = inconsistent attention',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures', 'fig6_2d_vit_attention_consistency.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Figure 6.2d saved.')

**Figure 6.2c results.**

| Class | Mean ring concentration |
|---|---:|
| No Substructure | $0.371 \pm 0.032$ |
| Sphere | $0.462 \pm 0.050$ |
| Vortex | $0.468 \pm 0.053$ |
| Random baseline | $0.190$ |

All three classes are well above the random baseline, so ViT attention rollout is generally concentrated on the ring region rather than distributed uniformly across the image. Sphere and Vortex have higher mean ring concentration than No Substructure, which is consistent with stronger ring-focused attention on the substructure classes.

The spread is also larger for Sphere and Vortex than for No Substructure, suggesting greater image-to-image variability in where the model places attention. This is only a descriptive pattern; ring concentration alone does not show which specific feature is driving the decision.

**Figure 6.2d results.**

| Class | Mean attention std |
|---|---:|
| No Substructure | $0.0409$ |
| Sphere | $0.0597$ |
| Vortex | $0.0516$ |

The mean attention maps are broadly similar across classes, with attention concentrated near the central ring region. The main difference appears in the standard-deviation maps: **No Substructure** is the most stable, while **Sphere** and **Vortex** show greater image-to-image variation.

A plausible interpretation is that attention shifts more across substructure images because the relevant perturbation is not fixed at one location on the ring. That said, this metric is descriptive only. It shows variability in spatial attention, not whether that variability helps or hurts classification performance.

####6.5.7 Ring Concentration: DenseNet vs ViT

The final analysis in this section places both models on the same quantitative
axis — ring concentration score — computed over n=50 images per class.
DenseNet scores are derived from Grad-CAM, ViT scores from Attention Rollout.
The critical caveat from Section 6.2.1 applies: because the two methods
measure different quantities, the absolute scores are not directly comparable.
What is informative is the pattern of scores across classes within each model,
and the direction of difference between models.

In [ ]:
# ──Ring concentration comparison — DenseNet vs ViT ──────────────────
# Computes ring concentration scores for both models on n=50 images per class.
# DenseNet uses Grad-CAM; ViT uses Attention Rollout.
# Produces a summary table and distribution comparison plot.

N_COMPARE   = 50
ring_scores = {
    'DenseNet': {c: [] for c in range(3)},
    'ViT':      {c: [] for c in range(3)}
}

for target_class in range(3):
    count = 0
    for idx in range(len(val_ds)):
        if count >= N_COMPARE:
            break
        img, label = val_ds[idx]
        if label != target_class:
            continue
        x = img.unsqueeze(0).to(device)

        # DenseNet Grad-CAM ring score
        cam_map = densenet_cam(input_tensor=x, targets=None)[0]
        cam_map = (cam_map - cam_map.min()) / (
            cam_map.max() - cam_map.min() + 1e-8)
        ring_scores['DenseNet'][target_class].append(
            float(cam_map[ring].sum() / (cam_map.sum() + 1e-8))
        )

        # ViT Attention Rollout ring score
        attn_map = rollout(x)
        attn_150 = cv2.resize(attn_map, (150, 150))
        attn_150 = (attn_150 - attn_150.min()) / (
            attn_150.max() - attn_150.min() + 1e-8)
        ring_scores['ViT'][target_class].append(
            float(attn_150[ring].sum() / (attn_150.sum() + 1e-8))
        )

        count += 1

# ── Summary table ──────────────────────────────────────────────────────────────
print(f'\n{"─"*72}')
print(f'{"Ring Concentration Score (mean ± std)":^72}')
print(f'{"Fraction of attention mass inside Einstein ring annulus":^72}')
print(f'{"─"*72}')
print(f'{"Model":<12} {"No Substructure":>22} {"Sphere":>22} {"Vortex":>22}')
print(f'{"─"*72}')
for model_name in ['DenseNet', 'ViT']:
    row = []
    for c in range(3):
        v = ring_scores[model_name][c]
        row.append(f'{np.mean(v):.3f} ± {np.std(v):.3f}')
    print(f'{model_name:<12} {row[0]:>22} {row[1]:>22} {row[2]:>22}')
print(f'{"─"*72}')
print(f'{"Random baseline":<12} {random_baseline:.3f}')
print(f'{"─"*72}')

# ── Distribution comparison plot ───────────────────────────────────────────────
colors_m = {'DenseNet': '#2980b9', 'ViT': '#e67e22'}
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for c in range(3):
    for model_name, color in colors_m.items():
        vals = ring_scores[model_name][c]
        axes[c].hist(vals, bins=20, alpha=0.6,
                     color=color, label=model_name, density=True)
        axes[c].axvline(np.mean(vals), color=color,
                        linestyle='--', lw=2)
    axes[c].axvline(random_baseline, color='black',
                    linestyle=':', lw=1.5,
                    label=f'Random ({random_baseline:.3f})')
    axes[c].set_title(CLASS_NAMES[c], fontsize=11, fontweight='bold')
    axes[c].set_xlabel('Ring Concentration Score', fontsize=10)
    axes[c].legend(fontsize=9)

axes[0].set_ylabel('Density', fontsize=10)
plt.suptitle(
    f'Figure 6.2e — Ring Concentration: DenseNet-121 vs ViT-Base  (n={N_COMPARE} per class)\n'
    'ViT consistently concentrates more attention on the Einstein ring than DenseNet\n'
    'Note: the two methods measure different quantities — see Section 6.2.1',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures',
                 'fig6_2e_ring_concentration_comparison.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

# ── Cleanup ───────────────────────────────────────────────────────────────────
clear_all_hooks(densenet)
clear_all_hooks(vit_model)
print('\nAll hooks cleared. Section 6.2 complete.')

**Results.**

| Model | No Substructure | Sphere | Vortex |
|---|---:|---:|---:|
| DenseNet | $0.282 \pm 0.019$ | $0.288 \pm 0.020$ | $0.285 \pm 0.014$ |
| ViT | $0.371 \pm 0.032$ | $0.462 \pm 0.050$ | $0.468 \pm 0.053$ |
| Random baseline | $0.190$ | $0.190$ | $0.190$ |

Both models are above the random baseline, so both place more attention on the ring than a spatially uniform map would. The difference is in **class sensitivity**. DenseNet is nearly flat across classes, whereas ViT shows higher ring concentration for Sphere and Vortex than for No Substructure.

This suggests that ViT attention is more class-dependent at the level of coarse ring localization, while DenseNet's Grad-CAM remains relatively similar across classes. That pattern should be interpreted cautiously, however, because the two visualization methods are not equivalent and DenseNet Grad-CAM is strongly limited by spatial compression.

**ViT shows stronger and more class-varying ring concentration than DenseNet under their respective interpretability measures.** This does not by itself explain the performance difference between the models.

#### 6.5.8 Summary

| Finding | Evidence |
|---|---|
| DenseNet Grad-CAM is strongly resolution-limited | Figure 6.2b: final spatial map is extremely coarse before upsampling |
| Both models concentrate attention above the random ring baseline | DenseNet $\approx 0.28$, ViT $\approx 0.37$–$0.47$, baseline $0.19$ |
| ViT shows stronger class-dependent ring concentration | No Substructure $0.371$ vs Sphere/Vortex $0.462$–$0.468$ |
| DenseNet ring concentration is nearly flat across classes | DenseNet $0.282$–$0.288$ across all three classes |
| ViT attention is less stable on substructure classes | Higher attention std for Sphere and Vortex than for No Substructure |
| Higher ring concentration does not guarantee higher classification performance | ViT is more ring-concentrated, but DenseNet has higher macro AUC |

Taken together, these results suggest that coarse ring focus and classification performance are not the same thing. ViT shows stronger ring-centered attention under rollout, whereas DenseNet performs better despite much coarser Grad-CAM maps. This comparison is informative, but not causal: the models use different architectures, different interpretability tools, and different internal spatial resolutions.

### 6.6 Predictive Uncertainty — Deep Ensemble ($M=8$)

Predictive uncertainty is included to distinguish **confident, stable predictions** from cases that remain ambiguous even when overall accuracy is high. A correct prediction is not necessarily a reliable one: some images lie close to the decision boundary and are sensitive to model choice.

To estimate this uncertainty, we use a **deep ensemble**. Multiple trained models are evaluated on the same image, and their predicted probability vectors are averaged. If the models agree, the prediction is more likely to be robust; if they disagree, the image is more likely to be difficult, ambiguous, or weakly supported by the learned features. In this section, ensemble disagreement is summarized through predictive entropy.

Monte Carlo dropout is not applicable here because the trained models do not contain active dropout at inference. Uncertainty is therefore estimated with a **deep ensemble**, formed by averaging the predicted probability vectors from eight independently trained architectures.

The uncertainty score is the **predictive entropy** of the ensemble mean:
$$
H = -\sum_c \bar{p}_c \log \bar{p}_c,
\qquad
\bar{p}_c = \frac{1}{M}\sum_{m=1}^{M} p_c^{(m)} .
$$

For a three-class problem, the maximum entropy is $\log 3 \approx 1.099$.

**Ensemble members ($M=8$):** ResNet-18, ResNet-50, DenseNet-121, EfficientNet-B3, ViT-Base, E-ResNet, EqDenseNet-C8, and Equivariant-$D_4$.

AlexNet and VGG-16 are excluded because their performance is substantially below the rest of the benchmark, so their inclusion would mainly add low-quality votes rather than useful disagreement. The goal here is not to maximize ensemble accuracy, but to use cross-model disagreement as a practical proxy for predictive uncertainty.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 8 — Deep Ensemble Uncertainty (7 models including EqDenseNet-C8)
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import torch
import torch.nn.functional as F
import os

CLASS_NAMES = ['No Substructure', 'Sphere', 'Vortex']

# ── Step 0: Collect EqDenseNet-C8 probs/preds if not already in ARCH_METRICS ─

if 'EqDenseNet-C8' not in ARCH_METRICS:
    print('Computing EqDenseNet-C8 predictions...')
    from torch.utils.data import DataLoader
    _loader = DataLoader(val_ds, batch_size=256,
                         shuffle=False, num_workers=2, pin_memory=True)
    eq_densenet.eval()
    _all_probs, _all_preds, _all_labels = [], [], []
    with torch.no_grad():
        for imgs, lbls in _loader:
            imgs   = imgs.to(device)
            logits = eq_densenet(imgs)
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            _all_probs.append(probs)
            _all_preds.append(probs.argmax(1))
            _all_labels.append(lbls.numpy())
    ARCH_METRICS['EqDenseNet-C8'] = {
        'probs':  np.concatenate(_all_probs,  axis=0),
        'preds':  np.concatenate(_all_preds,  axis=0),
        'labels': np.concatenate(_all_labels, axis=0),
    }
    print(f'  EqDenseNet-C8 accuracy: '
          f'{(ARCH_METRICS["EqDenseNet-C8"]["preds"] == ARCH_METRICS["EqDenseNet-C8"]["labels"]).mean():.4f}')
else:
    print('EqDenseNet-C8 already in ARCH_METRICS.')


# ── Step 1: Stack probability arrays [M, N, C] ────────────────────────────────

ENSEMBLE_MODELS = ['ResNet-18', 'ResNet-50', 'DenseNet-121',
                   'EfficientNet-B3', 'ViT-Base', 'E-ResNet',
                   'EqDenseNet-C8', 'Equivariant-D4']

N_CLASSES   = 3
N_VAL       = 7500
MAX_ENTROPY = np.log(N_CLASSES)

prob_stack = np.stack(
    [ARCH_METRICS[m]['probs'] for m in ENSEMBLE_MODELS], axis=0
)                                                              # [7, 7500, 3]
labels_arr = np.array(ARCH_METRICS['ResNet-18']['labels'])

print(f'Probability stack shape: {prob_stack.shape}')
print(f'Ensemble members: {ENSEMBLE_MODELS}')

# ── Step 2: Ensemble mean and predictive entropy ──────────────────────────────

mean_probs    = prob_stack.mean(axis=0)
entropy       = -np.sum(mean_probs * np.log(mean_probs + 1e-9), axis=1)
ensemble_pred = mean_probs.argmax(axis=1)

print(f'\nEntropy range: [{entropy.min():.4f}, {entropy.max():.4f}]')
print(f'Max possible : {MAX_ENTROPY:.4f}')
print(f'\nEnsemble accuracy: {(ensemble_pred == labels_arr).mean():.4f}')

# ── Step 3: Per-class entropy statistics ──────────────────────────────────────

print('\nPer-class entropy statistics:')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask    = labels_arr == cls_idx
    ent_cls = entropy[mask]
    print(f'  {cls_name:20s}: mean={ent_cls.mean():.4f}  '
          f'std={ent_cls.std():.4f}  '
          f'median={np.median(ent_cls):.4f}  '
          f'p95={np.percentile(ent_cls, 95):.4f}')

# ── Step 4: Statistical tests ─────────────────────────────────────────────────

sphere_ent = entropy[labels_arr == 1]
no_sub_ent = entropy[labels_arr == 0]
vortex_ent = entropy[labels_arr == 2]

u1, p1 = stats.mannwhitneyu(sphere_ent, no_sub_ent, alternative='greater')
u2, p2 = stats.mannwhitneyu(sphere_ent, vortex_ent, alternative='greater')

print(f'\nMann-Whitney U (Sphere entropy > No Substructure): p={p1:.6f}  '
      f'{"✅" if p1 < 0.05 else "❌"}')
print(f'Mann-Whitney U (Sphere entropy > Vortex):          p={p2:.6f}  '
      f'{"✅" if p2 < 0.05 else "❌"}')

# ── Step 5: Per-class entropy distribution plot ───────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['steelblue', 'crimson', 'seagreen']

# Violin
parts = axes[0].violinplot(
    [no_sub_ent, sphere_ent, vortex_ent],
    positions=[0, 1, 2], showmedians=True, showextrema=True
)
for pc, color in zip(parts['bodies'], colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
axes[0].set_xticks([0, 1, 2])
axes[0].set_xticklabels(CLASS_NAMES, fontsize=10)
axes[0].set_ylabel('Predictive entropy', fontsize=11)
axes[0].set_title('Entropy Distribution by True Class\n'
                  f'(Sphere vs No Sub: p={p1:.2e}  '
                  f'Sphere vs Vortex: p={p2:.2e})', fontsize=10)
axes[0].axhline(MAX_ENTROPY, color='black', linestyle='--',
                lw=1, label=f'Max entropy ({MAX_ENTROPY:.3f})')
axes[0].legend(fontsize=9)

# Histogram
bins = np.linspace(0, MAX_ENTROPY, 50)
for cls_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    axes[1].hist(entropy[labels_arr == cls_idx], bins=bins,
                 alpha=0.6, color=color, density=True, label=cls_name)
axes[1].set_xlabel('Predictive entropy', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title('Entropy Histogram by True Class', fontsize=11)
axes[1].legend(fontsize=10)

# Confidence vs uncertainty scatter
max_prob = mean_probs.max(axis=1)
for cls_idx, (cls_name, color) in enumerate(zip(CLASS_NAMES, colors)):
    mask = labels_arr == cls_idx
    axes[2].scatter(max_prob[mask], entropy[mask],
                    alpha=0.15, s=4, color=color, label=cls_name)
axes[2].set_xlabel('Ensemble max probability (confidence)', fontsize=11)
axes[2].set_ylabel('Predictive entropy', fontsize=11)
axes[2].set_title('Confidence vs Uncertainty\n(each point = one val image)',
                  fontsize=11)
axes[2].legend(fontsize=10, markerscale=4)

plt.suptitle(
    f'Deep Ensemble Predictive Uncertainty (M={len(ENSEMBLE_MODELS)} models)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(gsoc_drive_path, 'figures',
                         'ensemble_uncertainty.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Step 6: Model disagreement analysis ───────────────────────────────────────

model_preds = np.stack(
    [ARCH_METRICS[m]['preds'] for m in ENSEMBLE_MODELS], axis=0
)
disagreement = np.mean(
    model_preds != ensemble_pred[np.newaxis, :], axis=0
)

print('\nPer-class model disagreement (fraction of ensemble members '
      'not matching ensemble prediction):')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    mask = labels_arr == cls_idx
    print(f'  {cls_name:20s}: mean={disagreement[mask].mean():.4f}  '
          f'max={disagreement[mask].max():.4f}')

# ── Step 7: High-uncertainty Sphere image grid ────────────────────────────────

sphere_indices  = np.where(labels_arr == 1)[0]
sphere_entropy  = entropy[sphere_indices]
top12_local_idx = np.argsort(sphere_entropy)[::-1][:12]
top12_val_idx   = sphere_indices[top12_local_idx]

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
axes = axes.flatten()

for i, val_idx in enumerate(top12_val_idx):
    img = np.load(val_ds.filepaths[val_idx]).astype(np.float32).squeeze()
    axes[i].imshow(img, cmap='magma', origin='lower')
    axes[i].axis('off')
    ent_val  = entropy[val_idx]
    pred_val = ensemble_pred[val_idx]
    axes[i].set_title(
        f'H={ent_val:.3f}\npred: {CLASS_NAMES[pred_val][:6]}',
        fontsize=8
    )

fig.suptitle(
    'Top 12 Highest-Entropy Sphere Images\n'
    '(images the ensemble is most uncertain about)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(gsoc_drive_path, 'figures',
                         'sphere_high_entropy_grid.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Step 8: Universally-misclassified Sphere images ───────────────────────────

sphere_val_indices = np.where(labels_arr == 1)[0]
wrong_as_no_sub    = []

for val_idx in sphere_val_indices:
    all_predicted_no_sub = all(
        ARCH_METRICS[m]['preds'][val_idx] == 0
        for m in ENSEMBLE_MODELS
    )
    if all_predicted_no_sub:
        wrong_as_no_sub.append(val_idx)

print(f'\nSphere images predicted as No Substructure by all '
      f'{len(ENSEMBLE_MODELS)} ensemble members: {len(wrong_as_no_sub)}')

overlap = set(top12_val_idx.tolist()).intersection(set(wrong_as_no_sub))
print(f'Overlap between top-12 high-entropy Sphere images '
      f'and universally-misclassified: {len(overlap)}/12')

#### 6.6.1 Entropy Distribution Results

| Class | Mean | Std | Median | p95 |
|:--|--:|--:|--:|--:|
| No Substructure | 0.2401 | 0.1882 | 0.1740 | 0.6308 |
| Sphere | 0.2491 | **0.2911** | 0.1117 | **0.8447** |
| Vortex | 0.1798 | 0.2537 | 0.0448 | 0.7557 |

Ensemble accuracy on the validation set is **96.88%**.

All three entropy distributions are strongly right-skewed: most images have low predictive entropy, with a smaller tail of highly uncertain cases. The main class-level difference is that **Sphere** has the heaviest tail and the largest variance, while **Vortex** is the most consistently low-entropy class.

The rank-based comparisons are mixed. Sphere entropy is **not** larger than No Substructure in a one-sided Mann–Whitney test, despite a slightly higher mean; this reflects a distributional difference rather than a simple location shift. By contrast, Sphere entropy is clearly higher than Vortex.

| Class | Mean disagreement | Max disagreement |
|:--|--:|--:|
| No Substructure | 0.0104 | 0.6250 |
| Sphere | **0.0688** | 0.7500 |
| Vortex | 0.0410 | 0.7500 |

Model disagreement is also highest for **Sphere**, which is consistent with that class being the least stable across architectures.

A further limitation of entropy-based triage is visible here: **54 Sphere images** are misclassified as No Substructure by **all 8 ensemble members**, and none of the top-12 highest-entropy Sphere images belong to that set. This means high uncertainty and unanimous confident error are two different failure modes; entropy captures the former but not the latter.

#### 6.6.2 High-Entropy Sphere Images

The 12 highest-entropy Sphere images lie very close to the maximum possible entropy, so they represent cases of near-maximal ensemble disagreement. The predicted classes are distributed across all three labels, indicating that no single interpretation dominates across models.

Qualitatively, these images tend to look ambiguous: the arcs are often faint, partial, or morphologically mixed. That observation is only descriptive, but it is consistent with why these cases produce high entropy.

Importantly, these images are **not** the same as the unanimous Sphere$\rightarrow$No Substructure failures. The overlap is **0/12**. This shows that high entropy and confident error are distinct failure modes: entropy identifies disagreement, not error itself.

#### 6.6.3 Two Distinct Sphere Failure Populations

The zero overlap between the highest-entropy Sphere images and the 54 unanimously misclassified Sphere images suggests that Sphere errors do not form a single homogeneous group. Instead, under the ensemble-entropy view, they separate into two practically different categories:

| Failure group | Entropy | $n$ | Description | Operational implication |
|:--|:--|:--:|:--|:--|
| Silent failure | Low ($H \to 0$) | 54 | All 8 models predict No Substructure with little or no disagreement. | Not detected by entropy-based triage. |
| High-entropy ambiguity | High ($H \approx 1.09$) | $\sim 12$ | Ensemble predictions are split across classes. | Naturally flagged for review by entropy-based triage. |

The more challenging group operationally is the **silent-failure** set: the ensemble is wrong, but does not express uncertainty. By contrast, the **high-entropy** cases are at least self-identified as ambiguous.

This distinction matters because the two groups likely require different responses. High-entropy cases point to unresolved class ambiguity, whereas silent failures are more consistent with weak or poorly recoverable signal. The later image-quality analysis is relevant for interpreting that second group, but this section alone should be read as a descriptive partition of the failure set rather than a full causal explanation.

A single Sphere recall value collapses these two behaviors into one number. The entropy analysis helps separate them, which is more useful for diagnosing what kind of improvement would actually be needed.

#### 6.6.4 Practical Implication for Survey Pipelines

The main practical value of ensemble entropy is **triage**, not guaranteed error detection. High-entropy cases can be flagged as ambiguous and routed to secondary review, while low-entropy cases remain candidates for automated acceptance.

However, the earlier analysis also shows a clear limitation: **low entropy does not imply correctness**. In particular, the silent-failure Sphere cases would pass through any entropy-based filter despite unanimous error. Entropy is therefore useful for identifying some uncertain predictions, but it is not sufficient as a standalone reliability criterion.

In a survey setting, the most defensible use of this signal would be as one component of a broader decision pipeline, combined with other checks such as class-specific error analysis, image-quality indicators, or follow-up review rules. The silent-failure population points to a separate problem that uncertainty alone does not solve.

### 6.7 Ablation Study — E-ResNet Component Analysis

This ablation examines two design choices in the equivariant model: **residual connections** and **training-time augmentation**.

| Run | Architecture | Augmentation |
|---|---|---|
| A | Plain equivariant CNN | None |
| B | Plain equivariant CNN | $D_4$ augmentation |
| C | E-ResNet | None |
| D | E-ResNet | $D_4$ augmentation |

All four runs use the same corrected architecture setup and the same training recipe, so the comparison is internally consistent. The goal is not to make a strong causal claim from a single four-run study, but to see which changes are associated with better optimization and validation performance.

The ablation asks three narrower questions:

- **Do residual connections improve optimization and final performance within the equivariant setting?**
- **Is any residual benefit especially visible on the Sphere class, which is the hardest class in the benchmark?**
- **How much does explicit augmentation add once $D_4$ equivariance is already built into the model?**

These are the questions the results section addresses; the present subsection only defines the comparison.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 6.4 — E-ResNet Ablation Study (Architecture with: padding=3)
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.metrics import auc

# ── EPlainBlock ─────────────────────────────────────────────────────
class EPlainBlock(nn.Module):
    """
    Equivariant block WITHOUT residual connection.
    Identical channel structure to EResBlock — only the skip connection
    is removed. Isolates the contribution of residual connections from
    the equivariant inductive bias.
    """
    def __init__(self, in_type, out_type, stride=1):
        super().__init__()
        self.conv1 = enn.R2Conv(in_type,  out_type, kernel_size=3,
                                padding=1, stride=stride, bias=False)
        self.bn1   = enn.InnerBatchNorm(out_type)
        self.relu1 = enn.ReLU(out_type, inplace=True)
        self.conv2 = enn.R2Conv(out_type, out_type, kernel_size=3,
                                padding=1, bias=False)
        self.bn2   = enn.InnerBatchNorm(out_type)
        self.relu2 = enn.ReLU(out_type, inplace=True)

    def forward(self, x):
        out = self.relu1(self.bn1(self.conv1(x)))
        return self.relu2(self.bn2(self.conv2(out)))


# ── EquivariantPlainCNN ─────────────────────────────────────────────
class EquivariantPlainCNN(nn.Module):
    """
    D4-equivariant plain CNN — identical to EquivariantResNet but with
    residual connections removed.

    FIX: stem padding changed from 2 → 3 to ensure even spatial dimensions
    at every strided operation:
        150×150 → 76×76 (stem, stride=2)  ← even ✓
        76×76   → 38×38 (layer1, stride=2) ← even ✓
        38×38   → 19×19 (layer2, stride=2) ← AdaptiveAvgPool handles this ✓

    Without this fix, the stem outputs 75×75 (odd). A stride-2 convolution
    on an odd-dimensional feature map samples even-indexed coordinates.
    After 90° rotation, previously even coordinates become odd, breaking
    the D4 invariance guarantee at the pixel level.
    """
    def __init__(self, n_classes=3):
        super().__init__()
        act = gspaces.flipRot2dOnR2(N=4)

        self.in_type = enn.FieldType(act, 1  * [act.trivial_repr])
        t16          = enn.FieldType(act, 16 * [act.regular_repr])
        t32          = enn.FieldType(act, 32 * [act.regular_repr])
        t64          = enn.FieldType(act, 64 * [act.regular_repr])

        self.stem   = enn.SequentialModule(
            enn.R2Conv(self.in_type, t16,
                       kernel_size=5,
                       padding=3,        # ← FIXED: was 2, now 3
                       stride=2, bias=False),
            enn.InnerBatchNorm(t16),
            enn.ReLU(t16, inplace=True)
        )
        self.layer1 = EPlainBlock(t16, t32, stride=2)
        self.layer2 = EPlainBlock(t32, t64, stride=2)
        self.gpool  = enn.GroupPooling(t64)
        self.fc     = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = enn.GeometricTensor(x, self.in_type)
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.gpool(x)
        return self.fc(x.tensor)

# ── EResBlock — required by EquivariantResNet ─────────────────────────────────
class EResBlock(nn.Module):
    """
    D4-equivariant residual block with skip connection.
    The identity shortcut allows gradients to bypass the convolutional
    transformations, providing a gradient highway through equivariant layers.
    """
    def __init__(self, in_type, out_type, stride=1):
        super().__init__()
        self.conv1      = enn.R2Conv(in_type,  out_type, kernel_size=3,
                                     padding=1, stride=stride, bias=False)
        self.bn1        = enn.InnerBatchNorm(out_type)
        self.relu1      = enn.ReLU(out_type, inplace=True)
        self.conv2      = enn.R2Conv(out_type, out_type, kernel_size=3,
                                     padding=1, bias=False)
        self.bn2        = enn.InnerBatchNorm(out_type)
        self.relu2      = enn.ReLU(out_type, inplace=True)
        self.downsample = None
        if stride != 1 or in_type != out_type:
            self.downsample = enn.SequentialModule(
                enn.R2Conv(in_type, out_type, kernel_size=1,
                           stride=stride, bias=False),
                enn.InnerBatchNorm(out_type)
            )

    def forward(self, x):
        identity = x
        out      = self.relu1(self.bn1(self.conv1(x)))
        out      = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu2(out + identity)

# ──EquivariantResNet ───────────────────────────────────────────────
class EquivariantResNet(nn.Module):
    """
    D4-equivariant residual network.

    FIX: stem padding changed from 2 → 3 (same reason as EquivariantPlainCNN).
    """
    def __init__(self, n_classes=3):
        super().__init__()
        act = gspaces.flipRot2dOnR2(N=4)

        self.in_type = enn.FieldType(act, 1 * [act.trivial_repr])
        t16          = enn.FieldType(act, 16 * [act.regular_repr])
        t32          = enn.FieldType(act, 32 * [act.regular_repr])
        t64          = enn.FieldType(act, 64 * [act.regular_repr])

        self.stem = enn.SequentialModule(
            enn.R2Conv(self.in_type, t16,
                       kernel_size=5,
                       padding=3,        # ← FIXED: was 2, now 3
                       stride=2, bias=False),
            enn.InnerBatchNorm(t16),
            enn.ReLU(t16, inplace=True)
        )
        self.layer1 = EResBlock(t16, t32, stride=2)
        self.layer2 = EResBlock(t32, t64, stride=2)
        self.gpool  = enn.GroupPooling(t64)
        self.fc     = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        x = enn.GeometricTensor(x, self.in_type)
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.gpool(x)
        return self.fc(x.tensor)


# ── Sanity check — verify even spatial dimensions before any training ─────────
print('Spatial dimension trace (corrected architecture):')
_m = EquivariantPlainCNN().to(device)
_x = torch.zeros(1, 1, 150, 150).to(device)
with torch.no_grad():
    _g  = enn.GeometricTensor(_x, _m.in_type)
    _s  = _m.stem(_g)
    h1, w1 = _s.tensor.shape[2], _s.tensor.shape[3]
    _l1 = _m.layer1(_s)
    h2, w2 = _l1.tensor.shape[2], _l1.tensor.shape[3]
    _l2 = _m.layer2(_l1)
    h3, w3 = _l2.tensor.shape[2], _l2.tensor.shape[3]

print(f'  Input:          150×150')
print(f'  After stem:     {h1}×{w1}  '
      f'{"✓ even" if h1 % 2 == 0 else "✗ odd — DO NOT TRAIN"}')
print(f'  After layer1:   {h2}×{w2}  '
      f'{"✓ even" if h2 % 2 == 0 else "✗ odd — DO NOT TRAIN"}')
print(f'  After layer2:   {h3}×{w3}  '
      f'(AdaptiveAvgPool handles any size)')
del _m, _x, _g, _s, _l1, _l2

assert h1 % 2 == 0 and h2 % 2 == 0, \
    f'Odd dimensions detected: stem={h1}, layer1={h2}. Fix padding before training.'
print('\nArchitecture verified. Safe to proceed with training.\n')


# ── No-augmentation train loader ──────────────────────────────────────────────
train_ds_noaug     = LensingDataset(train_path, augment=False)
train_loader_noaug = DataLoader(
    train_ds_noaug, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=True
)
print(f'No-aug train loader: {len(train_ds_noaug):,} samples')


# ── Per-class recall helper ───────────────────────────────────────────────────
def per_class_recall(preds, labels, n_classes=3):
    recalls = []
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() == 0:
            recalls.append(0.0)
        else:
            recalls.append(float((preds[mask] == c).mean()))
    return recalls


# ── Training function ─────────────────────────────────────────────────────────
def load_or_train_ablation(model, model_name, train_ldr, epochs=60, lr=1e-3):
    save_path = os.path.join(gsoc_drive_path, f'{model_name}_best.pth')

    if os.path.exists(save_path):
        print(f'\n[SKIP] {model_name} — weights found, loading...')
        model = model.to(device)
        model.load_state_dict(torch.load(save_path, map_location=device))
        labels_v, probs_v, preds_v = evaluate_on_loader(model, val_loader)
        print_per_class_report(labels_v, preds_v, model_name)
        fig_dir = os.path.join(gsoc_drive_path, 'figures')
        metrics = plot_results(model_name, labels_v, probs_v, preds_v,
                               save_dir=fig_dir)
        test_acc      = (preds_v == labels_v).mean()
        sphere_recall = (preds_v[labels_v == 1] == 1).sum() / \
                        (labels_v == 1).sum()
        history_path  = os.path.join(gsoc_drive_path,
                                     f'{model_name}_history.npy')
        history = np.load(history_path, allow_pickle=True).item() \
                  if os.path.exists(history_path) else {}
        ARCH_METRICS[model_name] = {
            **metrics,
            'test_accuracy':  test_acc,
            'sphere_recall':  float(sphere_recall),
            'total_params_M': sum(p.numel() for p in model.parameters()) / 1e6,
            'pretrained':     False,
            'labels':  labels_v, 'probs': probs_v, 'preds': preds_v,
            'history': history,
        }
        print(f'  Test Accuracy : {test_acc:.4f}')
        print(f'  Macro AUC     : {metrics["macro_auc"]:.4f}')
        print(f'  Sphere Recall : {sphere_recall:.4f}')
        return model

    print(f'\n{"="*60}')
    print(f'  {model_name}')
    n_total     = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters()
                      if p.requires_grad)
    print(f'  Parameters: {n_total/1e6:.2f}M total  |  '
          f'{n_trainable/1e6:.2f}M trainable')
    print(f'  LR: {lr}  |  Max epochs: {epochs}  |  Patience: 15')
    print(f'{"="*60}')

    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs
    )

    history = {
        'train_loss': [], 'val_loss': [], 'val_acc': [],
        'recall_no_sub': [], 'recall_sphere': [], 'recall_vortex': []
    }

    best_val_loss = float('inf')
    patience_ctr  = 0
    patience      = 15

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for imgs, lbls in train_ldr:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        model.eval()
        val_loss     = 0.0
        all_preds_e  = []
        all_labels_e = []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                logits      = model(imgs)
                val_loss   += criterion(logits, lbls).item()
                all_preds_e.extend(logits.argmax(1).cpu().numpy())
                all_labels_e.extend(lbls.cpu().numpy())

        all_preds_e  = np.array(all_preds_e)
        all_labels_e = np.array(all_labels_e)
        avg_train    = train_loss / len(train_ldr)
        avg_val      = val_loss   / len(val_loader)
        val_acc      = (all_preds_e == all_labels_e).mean()
        recalls      = per_class_recall(all_preds_e, all_labels_e)

        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)
        history['val_acc'].append(val_acc)
        history['recall_no_sub'].append(recalls[0])
        history['recall_sphere'].append(recalls[1])
        history['recall_vortex'].append(recalls[2])

        print(f'Epoch {epoch:3d} | Train: {avg_train:.4f} | '
              f'Val: {avg_val:.4f} | Acc: {val_acc:.4f} | '
              f'Sph: {recalls[1]:.4f}')

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            patience_ctr  = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'\nEarly stopping at epoch {epoch}.')
                break

    history_path = os.path.join(gsoc_drive_path,
                                f'{model_name}_history.npy')
    np.save(history_path, history)
    print(f'\nBest weights → {save_path}')

    model.load_state_dict(torch.load(save_path, map_location=device))
    labels_v, probs_v, preds_v = evaluate_on_loader(model, val_loader)
    print_per_class_report(labels_v, preds_v, model_name)
    fig_dir = os.path.join(gsoc_drive_path, 'figures')
    metrics  = plot_results(model_name, labels_v, probs_v, preds_v,
                            save_dir=fig_dir)
    test_acc      = (preds_v == labels_v).mean()
    sphere_recall = (preds_v[labels_v == 1] == 1).sum() / \
                    (labels_v == 1).sum()
    ARCH_METRICS[model_name] = {
        **metrics,
        'test_accuracy':  test_acc,
        'sphere_recall':  float(sphere_recall),
        'total_params_M': sum(p.numel() for p in model.parameters()) / 1e6,
        'pretrained':     False,
        'labels':  labels_v, 'probs': probs_v, 'preds': preds_v,
        'history': history,
    }
    print(f'  Test Accuracy : {test_acc:.4f}')
    print(f'  Macro AUC     : {metrics["macro_auc"]:.4f}')
    print(f'  Sphere Recall : {sphere_recall:.4f}')
    return model


# ── Run A: Plain CNN, no augmentation ─────────────────────────────────────────
plain_noaug = EquivariantPlainCNN()
plain_noaug = load_or_train_ablation(
    plain_noaug, 'EPlainCNN-NoAug', train_loader_noaug, epochs=60, lr=1e-3
)

# ── Run B: Plain CNN, with augmentation ───────────────────────────────────────
plain_aug = EquivariantPlainCNN()
plain_aug = load_or_train_ablation(
    plain_aug, 'EPlainCNN-Aug', train_loader, epochs=60, lr=1e-3
)

# ── Run C: E-ResNet, no augmentation ──────────────────────────────────────────
eresnet_noaug = EquivariantResNet()
eresnet_noaug = load_or_train_ablation(
    eresnet_noaug, 'E-ResNet-NoAug', train_loader_noaug, epochs=60, lr=1e-3
)

# ── Run D: E-ResNet with augmentation — retrain with corrected architecture ───
# The previously saved E-ResNet weights used padding=2 (buggy architecture).
# Run D must be retrained with padding=3 to be consistent with Runs A, B, C.
e_resnet = EquivariantResNet()
e_resnet = load_or_train_ablation(
    e_resnet, 'E-ResNet', train_loader, epochs=60, lr=1e-3
)

print(f'\n── Ablation complete ─────────────────────────────────────────────')
for key in ['EPlainCNN-NoAug', 'EPlainCNN-Aug', 'E-ResNet-NoAug', 'E-ResNet']:
    if key in ARCH_METRICS:
        print(f'  {key:<20} AUC={ARCH_METRICS[key]["macro_auc"]:.4f}  '
              f'Sphere Recall={ARCH_METRICS[key]["sphere_recall"]:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 6.4 — Ablation Results Table and Convergence Plots
# All four runs use corrected architecture (stem padding=3, even dimensions)
# ─────────────────────────────────────────────────────────────────────────────

ablation_runs = [
    ('A', 'EPlainCNN-NoAug', 'Plain CNN', 'None'),
    ('B', 'EPlainCNN-Aug',   'Plain CNN', 'D₄ augmentation'),
    ('C', 'E-ResNet-NoAug',  'E-ResNet',  'None'),
    ('D', 'E-ResNet',        'E-ResNet',  'D₄ augmentation'),
]

# ── Summary table ─────────────────────────────────────────────────────────────
print(f'\n{"─"*90}')
print(f'{"Ablation Study — E-ResNet Component Analysis (corrected architecture)":^90}')
print(f'{"─"*90}')
print(f'{"Run":<5} {"Architecture":<18} {"Augmentation":<20} '
      f'{"AUC":>8} {"Accuracy":>10} '
      f'{"No Sub Rec":>12} {"Sphere Rec":>12} {"Vortex Rec":>12}')
print(f'{"─"*90}')

for run_id, key, arch, aug in ablation_runs:
    if key in ARCH_METRICS:
        m       = ARCH_METRICS[key]
        recalls = per_class_recall(
            np.array(m['preds']), np.array(m['labels'])
        )
        print(f'{run_id:<5} {arch:<18} {aug:<20} '
              f'{m["macro_auc"]:>8.4f} '
              f'{m["test_accuracy"]:>10.4f} '
              f'{recalls[0]:>12.4f} '
              f'{recalls[1]:>12.4f} '
              f'{recalls[2]:>12.4f}')
    else:
        print(f'{run_id:<5} {arch:<18} {aug:<20}  not yet run')

print(f'{"─"*90}')

# ── Effect sizes ──────────────────────────────────────────────────────────────
keys_present = all(k in ARCH_METRICS for k in
                   ['EPlainCNN-NoAug', 'EPlainCNN-Aug',
                    'E-ResNet-NoAug',  'E-ResNet'])

if keys_present:
    def _auc(k): return ARCH_METRICS[k]['macro_auc']
    def _sph(k):
        m = ARCH_METRICS[k]
        return per_class_recall(
            np.array(m['preds']), np.array(m['labels'])
        )[1]

    print(f'\nEffect sizes on Macro AUC:')
    print(f'  Residual connections (no aug) : '
          f'{_auc("E-ResNet-NoAug") - _auc("EPlainCNN-NoAug"):+.4f}  (C − A)')
    print(f'  Residual connections (aug)    : '
          f'{_auc("E-ResNet") - _auc("EPlainCNN-Aug"):+.4f}  (D − B)')
    print(f'  Augmentation (plain CNN)      : '
          f'{_auc("EPlainCNN-Aug") - _auc("EPlainCNN-NoAug"):+.4f}  (B − A)')
    print(f'  Augmentation (E-ResNet)       : '
          f'{_auc("E-ResNet") - _auc("E-ResNet-NoAug"):+.4f}  (D − C)')
    print(f'  Interaction (residual × aug)  : '
          f'{(_auc("E-ResNet") - _auc("EPlainCNN-Aug")) - (_auc("E-ResNet-NoAug") - _auc("EPlainCNN-NoAug")):+.4f}')

    print(f'\nEffect sizes on Sphere Recall:')
    print(f'  Residual connections (no aug) : '
          f'{_sph("E-ResNet-NoAug") - _sph("EPlainCNN-NoAug"):+.4f}  (C − A)')
    print(f'  Residual connections (aug)    : '
          f'{_sph("E-ResNet") - _sph("EPlainCNN-Aug"):+.4f}  (D − B)')
    print(f'  Augmentation (plain CNN)      : '
          f'{_sph("EPlainCNN-Aug") - _sph("EPlainCNN-NoAug"):+.4f}  (B − A)')
    print(f'  Augmentation (E-ResNet)       : '
          f'{_sph("E-ResNet") - _sph("E-ResNet-NoAug"):+.4f}  (D − C)')

# ── Convergence curves ────────────────────────────────────────────────────────
# All four runs now have full epoch history since Run D was retrained.
# No special-case horizontal reference lines needed.

colors_abl = ['#e74c3c', '#e67e22', '#2980b9', '#27ae60']
styles     = ['--',      '-.',      '--',      '-'     ]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (run_id, key, arch, aug) in enumerate(ablation_runs):
    if key not in ARCH_METRICS:
        print(f'  Warning: {key} not in ARCH_METRICS — skipping.')
        continue

    h     = ARCH_METRICS[key].get('history', {})
    c     = colors_abl[i]
    ls    = styles[i]
    label = f'Run {run_id}: {arch} / {aug}'

    if not h.get('train_loss'):
        # Fallback — history missing, show final value as reference line
        print(f'  Warning: no epoch history for {key} — showing final value only.')
        axes[1].axhline(
            y=float(ARCH_METRICS[key]['test_accuracy']),
            color=c, linestyle=':', lw=1.5,
            label=f'{label} (final only)'
        )
        axes[2].axhline(
            y=float(ARCH_METRICS[key]['sphere_recall']),
            color=c, linestyle=':', lw=1.5,
            label=f'{label} (final only)'
        )
        continue

    epochs_ran = range(1, len(h['train_loss']) + 1)

    # Panel 1 — training loss
    axes[0].plot(epochs_ran, h['train_loss'],
                 color=c, linestyle=ls, lw=2, label=label)

    # Panel 2 — validation accuracy
    axes[1].plot(epochs_ran, h['val_acc'],
                 color=c, linestyle=ls, lw=2, label=label)

    # Panel 3 — Sphere recall
    if h.get('recall_sphere'):
        axes[2].plot(epochs_ran, h['recall_sphere'],
                     color=c, linestyle=ls, lw=2, label=label)

# Reference lines
axes[1].axhline(0.90, color='black', linestyle=':', lw=1.5,
                label='90% accuracy threshold')
axes[2].axhline(0.90, color='black', linestyle=':', lw=1.5,
                label='90% recall threshold')

# Labels and titles
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Training Loss')
axes[0].set_title(
    'Training Loss Convergence',
    fontsize=10, fontweight='bold'
)
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Validation Accuracy')
axes[1].set_title(
    'Validation Accuracy',
    fontsize=10, fontweight='bold'
)
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Sphere Recall')
axes[2].set_title(
    'Sphere Class Recall\n(most sensitive metric for subhalo detection)',
    fontsize=10, fontweight='bold'
)
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.suptitle(
    'Figure 6.4 — Ablation Study: Convergence Curves\n'
    'Residual connections vs Plain CNN  ×  Augmentation vs None\n'
    'All four runs: corrected architecture (stem padding=3, even spatial dimensions)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures', 'ablation_convergence.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()
print('Figure 6.4 saved.')

#### 6.7.1 Hypothesis Assessment

The ablation supports one clear result and two weaker ones.

**1. Augmentation has the largest and most consistent effect.**  
Across both architectures, $D_4$ augmentation improves macro AUC, accuracy, and Sphere recall:

| Effect | $\Delta$ Macro AUC | $\Delta$ Sphere Recall |
|:--|--:|--:|
| Plain CNN: Aug − NoAug | +0.0090 | +0.0336 |
| E-ResNet: Aug − NoAug | +0.0073 | +0.0196 |

This is the dominant pattern in the table. The gain is visible not only for Sphere, but also for **No Substructure**, where recall increases substantially in both architectures, indicating fewer false positives on smooth lenses.

**2. Residual connections provide only a small benefit without augmentation.**  
Without augmentation, E-ResNet is slightly better than the plain equivariant CNN:
- Macro AUC: $0.9879$ vs $0.9866$
- Sphere recall: $0.9028$ vs $0.8932$

The direction is consistent with a modest residual benefit, but the magnitude is small.

**3. With augmentation, the residual benefit disappears.**  
Under augmentation, the plain equivariant CNN slightly exceeds E-ResNet on both macro AUC and Sphere recall:
- Macro AUC: $0.9956$ vs $0.9952$
- Sphere recall: $0.9268$ vs $0.9224$

This does not justify a strong claim against residual connections in general, but it does show that, in this shallow equivariant setting, residual connections are not the main source of performance.

**Summary.**  
Within this ablation, the strongest and most reliable effect is **augmentation**. Residual connections help slightly in the no-augmentation setting, but do not provide an additional gain once augmentation is already present.

A narrower practical takeaway is that **equivariance alone is not enough**: even in a symmetry-aware architecture, explicit augmentation still provides a substantial benefit.

### 6.8 Rotation Invariance Verification

This section tests **output invariance**, not internal feature equivariance. The question is simple: if an input image is rotated, how much does the model's output probability vector change?

Two stages are separated:

- **Stage 1: untrained models** — tests the symmetry built into the architecture before learning.
- **Stage 2: trained models** — tests how much of that stability remains after optimization.

For the equivariant models, transformed inputs are generated with `escnn`'s own group action via `in_type.transform`, which is the correct test for this architecture. For ResNet-50, standard tensor rotations are used. The metric is the $L_2$ distance between the original and transformed output probability vectors; smaller values mean greater invariance.

#### 6.8.1 Equivariance Verification Test

The invariance test is purely an inference-time diagnostic. For each selected validation image, the model is evaluated on the original input and on transformed versions of the same image.

- **Equivariant models:** all 8 $D_4$ group elements are tested using `in_type.transform`.
- **ResNet-50:** $90^\circ$, $180^\circ$, and $270^\circ$ rotations are tested using `torch.rot90`.

For each transformation, we compute the $L_2$ distance between the original and transformed output probability vectors. A perfectly invariant classifier would give distance $0$ for every transformed copy of the same image.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 6.5 — Rotation Invariance Verification (CORRECTED)
#
# Key corrections vs original:
#   1. in_type.transform(x, g_el) replaces torch.rot90 for equivariant models
#      torch.rot90 does not match escnn's internal group action convention —
#      verified in Test V development: produced false failures and false passes
#   2. All 8 D₄ group elements tested (4 rotations + 4 reflections)
#      Original only tested 3 rotations, missing the full reflection subgroup
#   3. ResNet-50 still uses torch.rot90 — correct for standard CNNs with
#      no escnn group structure
#   4. L2 computed on logits directly (before softmax) for cleaner signal
#
# Terminology:
#   Equivariance: f(g·x) = g·f(x) — internal feature maps transform
#                 consistently with input rotation.
#   Invariance:   f(g·x) = f(x)   — classifier output is unchanged
#                 under rotation. Achieved after GroupPooling.
# ─────────────────────────────────────────────────────────────────────────────
import torch.nn.functional as F


def get_probs(model, x):
    """Forward pass → softmax probabilities. x: (1,1,H,W) on device."""
    model.eval()
    with torch.no_grad():
        logits = model(x)
    return F.softmax(logits, dim=1).squeeze(0).cpu()


def run_invariance_test_equivariant(model, model_in_type,
                                    sample_indices, val_ds):
    """
    Correct invariance test for escnn equivariant models.
    Uses in_type.transform(x, g_el) — escnn's own group action.
    Tests all 8 D₄ elements.

    Returns: {g_el_str: [l2_distances across samples]}
    """
    results = {}
    g_elements = list(model_in_type.gspace.fibergroup.elements)

    for g_el in g_elements:
        results[str(g_el)] = []

    for idx in sample_indices:
        img, _ = val_ds[idx]
        x = img.unsqueeze(0).to(device)

        model.eval()
        with torch.no_grad():
            p_orig = F.softmax(model(x), dim=1).squeeze(0).cpu()

        for g_el in g_elements:
            x_tfm = model_in_type.transform(x.cpu(), g_el).to(device)
            with torch.no_grad():
                p_tfm = F.softmax(model(x_tfm), dim=1).squeeze(0).cpu()
            l2 = torch.norm(p_orig - p_tfm).item()
            results[str(g_el)].append(l2)

    return results


def run_invariance_test_standard(model, sample_indices, val_ds):
    """
    Standard invariance test for non-escnn models (ResNet-50 etc.).
    Uses torch.rot90 — correct for standard CNNs with no group structure.
    Tests 4 rotations only (reflections not applicable).

    Returns: {'90°': [...], '180°': [...], '270°': [...]}
    """
    results = {'90°': [], '180°': [], '270°': []}

    for idx in sample_indices:
        img, _ = val_ds[idx]
        x = img.unsqueeze(0).to(device)

        model.eval()
        with torch.no_grad():
            p_orig = F.softmax(model(x), dim=1).squeeze(0).cpu()

        for k, rot_label in zip([1, 2, 3], ['90°', '180°', '270°']):
            x_rot = torch.rot90(x, k=k, dims=[2, 3])
            with torch.no_grad():
                p_rot = F.softmax(model(x_rot), dim=1).squeeze(0).cpu()
            l2 = torch.norm(p_orig - p_rot).item()
            results[rot_label].append(l2)

    return results


def print_equivariant_invariance_table(results, model_name, title):
    """Print D₄ invariance table for escnn model."""
    print(f'\n{"─"*60}')
    print(f'{title:^60}')
    print(f'{"─"*60}')
    print(f'{"Group element":<35} {"Mean L2":>10} {"Max L2":>10}')
    print(f'{"─"*60}')
    all_means = []
    for g_str, vals in results.items():
        mean_l2 = np.mean(vals)
        max_l2  = np.max(vals)
        all_means.append(mean_l2)
        flag = '✓' if mean_l2 < 1e-3 else '⚠'
        print(f'  {g_str:<33} {mean_l2:>10.5f} {max_l2:>10.5f}  {flag}')
    print(f'{"─"*60}')
    print(f'  {"Overall mean L2":<33} {np.mean(all_means):>10.5f}')
    print(f'{"─"*60}')
    return np.mean(all_means)


def print_standard_invariance_table(results, model_name, title):
    """Print rotation invariance table for standard CNN."""
    print(f'\n{"─"*60}')
    print(f'{title:^60}')
    print(f'{"─"*60}')
    print(f'{"Rotation":<20} {"Mean L2":>10} {"Max L2":>10}')
    print(f'{"─"*60}')
    all_means = []
    for rot_label, vals in results.items():
        mean_l2 = np.mean(vals)
        max_l2  = np.max(vals)
        all_means.append(mean_l2)
        print(f'  {rot_label:<18} {mean_l2:>10.5f} {max_l2:>10.5f}')
    print(f'{"─"*60}')
    print(f'  {"Overall mean L2":<18} {np.mean(all_means):>10.5f}')
    print(f'{"─"*60}')
    return np.mean(all_means)


# ── Select 100 validation images (stratified ~33 per class) ───────────────────
torch.manual_seed(42)
n_per_class    = 34
sample_indices = []
for c in range(3):
    class_indices = [i for i in range(len(val_ds)) if val_ds[i][1] == c]
    selected      = torch.randperm(len(class_indices))[:n_per_class].tolist()
    sample_indices.extend([class_indices[j] for j in selected])
sample_indices = sample_indices[:100]
print(f'Sampled {len(sample_indices)} validation images')


# ═══════════════════════════════════════════════════════════════════════════
# STAGE 1 — Untrained models: theoretical invariance from architecture alone
# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('STAGE 1 — Theoretical invariance (untrained models)')
print('='*60)

untrained_eresnet  = EquivariantResNet(n_classes=3).to(device)
untrained_plaincnn = EquivariantPlainCNN(n_classes=3).to(device)

untrained_resnet50 = models.resnet50(weights=None)
untrained_resnet50.conv1 = nn.Conv2d(
    1, 64, kernel_size=7, stride=2, padding=3, bias=False
)
untrained_resnet50.fc = nn.Linear(
    untrained_resnet50.fc.in_features, 3
)
untrained_resnet50 = untrained_resnet50.to(device)

# Equivariant models — use in_type.transform
results_eresnet_untrained = run_invariance_test_equivariant(
    untrained_eresnet, untrained_eresnet.in_type,
    sample_indices, val_ds
)
mean_eresnet_untrained = print_equivariant_invariance_table(
    results_eresnet_untrained,
    'E-ResNet (untrained)',
    'E-ResNet D₄ — Untrained — All 8 D₄ Elements'
)

results_plaincnn_untrained = run_invariance_test_equivariant(
    untrained_plaincnn, untrained_plaincnn.in_type,
    sample_indices, val_ds
)
mean_plaincnn_untrained = print_equivariant_invariance_table(
    results_plaincnn_untrained,
    'EPlainCNN (untrained)',
    'EPlainCNN D₄ — Untrained — All 8 D₄ Elements'
)

# Standard CNN — use torch.rot90
results_resnet50_untrained = run_invariance_test_standard(
    untrained_resnet50, sample_indices, val_ds
)
mean_resnet50_untrained = print_standard_invariance_table(
    results_resnet50_untrained,
    'ResNet-50 (untrained)',
    'ResNet-50 — Untrained — Rotations Only'
)


# ═══════════════════════════════════════════════════════════════════════════
# STAGE 2 — Trained models: empirical invariance after training
# ═══════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('STAGE 2 — Empirical invariance (trained models)')
print('='*60)

# Equivariant models — use in_type.transform
results_eresnet_trained = run_invariance_test_equivariant(
    e_resnet, e_resnet.in_type,
    sample_indices, val_ds
)
mean_eresnet_trained = print_equivariant_invariance_table(
    results_eresnet_trained,
    'E-ResNet (trained)',
    'E-ResNet D₄ — Trained — All 8 D₄ Elements'
)

results_plaincnn_trained = run_invariance_test_equivariant(
    plain_noaug, plain_noaug.in_type,
    sample_indices, val_ds
)
mean_plaincnn_trained = print_equivariant_invariance_table(
    results_plaincnn_trained,
    'EPlainCNN (trained, no aug)',
    'EPlainCNN D₄ — Trained (no aug) — All 8 D₄ Elements'
)

# Standard CNN — use torch.rot90
results_resnet50_trained = run_invariance_test_standard(
    resnet50, sample_indices, val_ds
)
mean_resnet50_trained = print_standard_invariance_table(
    results_resnet50_trained,
    'ResNet-50 (trained, aug)',
    'ResNet-50 — Trained (aug) — Rotations Only'
)


# ═══════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════════════════
print(f'\n{"─"*72}')
print(f'{"Theoretical vs Empirical Invariance Summary":^72}')
print(f'{"─"*72}')
print(f'{"Model":<25} {"Theoretical":>14} {"Empirical":>14} '
      f'{"Change":>12} {"Note"}')
print(f'{"─"*72}')

summary_pairs = [
    ('E-ResNet D₄',
     mean_eresnet_untrained,
     mean_eresnet_trained,
     'escnn in_type.transform, 8 D₄ elements'),
    ('EPlainCNN D₄',
     mean_plaincnn_untrained,
     mean_plaincnn_trained,
     'escnn in_type.transform, 8 D₄ elements'),
    ('ResNet-50',
     mean_resnet50_untrained,
     mean_resnet50_trained,
     'torch.rot90, 3 rotations only'),
]

for label, theoretical, empirical, note in summary_pairs:
    change    = empirical - theoretical
    direction = '↑ worse' if change > 0.001 else '↓ better' \
                if change < -0.001 else '≈ stable'
    print(f'{label:<25} {theoretical:>14.5f} {empirical:>14.5f} '
          f'{change:>+10.5f}  {direction}')
    print(f'  {"":25} {note}')

print(f'{"─"*72}')
print("""
Notes:
  Equivariant models: verified with escnn in_type.transform — the correct
  method matching escnn's internal group action convention. torch.rot90
  does not match escnn's convention and produces misleading results for
  equivariant models (confirmed during Test V development).

  ResNet-50: torch.rot90 is correct for standard CNNs. Non-zero L2 is
  expected — ResNet-50 has no architectural symmetry constraint.

  Theoretical L2 ≈ 0 for equivariant models confirms the architectural
  invariance guarantee before any training. Empirical L2 after training
  reflects any residual sensitivity introduced by data-driven learning.
""")

# ── Cleanup ───────────────────────────────────────────────────────────────────
del untrained_eresnet, untrained_plaincnn, untrained_resnet50
print('Untrained model objects deleted from memory.')

#### 6.8.2 Stage 1 — Theoretical Invariance (Untrained Models)

Before training, both equivariant architectures show near-zero output variation under the tested group transformations:

| Model | Mean $L_2$ |
|:--|--:|
| EPlainCNN (untrained) | **0.00005** |
| E-ResNet (untrained) | **0.00009** |
| ResNet-50 (untrained) | 0.06480 |

For the equivariant models, the per-transformation mean $L_2$ values remain on the order of $10^{-4}$ or below across all tested $D_4$ elements. By contrast, the untrained ResNet-50 shows much larger rotation sensitivity.

This is the expected architectural pattern: before any learning takes place, the equivariant models are already nearly invariant at the classifier output under the tested transformations, whereas the standard CNN is not. The small numerical differences between EPlainCNN and E-ResNet at this stage are negligible in practical terms and should not be over-interpreted.

Stage 1 therefore provides strong evidence that the symmetry-aware architectures are behaving as intended at initialization. Stage 2 asks how much of that output stability remains after training.

#### 6.8.3 Stage 2 — Empirical Invariance After Training

After training, the equivariant models remain more output-stable under transformation than ResNet-50, although their invariance is no longer numerically near-perfect.

| Model | Training | Mean $L_2$ | Max $L_2$ |
|:--|:--:|--:|--:|
| E-ResNet $D_4$ | with aug | **0.01817** | 0.75205 |
| EPlainCNN $D_4$ | no aug | 0.03768 | 1.02230 |
| ResNet-50 | with aug | 0.05814 | 1.26498 |

Under this test, E-ResNet has the lowest mean output variation, followed by EPlainCNN, then ResNet-50. So the trained equivariant models remain more rotation-stable than the standard CNN, even though exact numerical invariance is weakened by training.

A useful comparison is that **EPlainCNN without augmentation** is still more stable than **ResNet-50 with augmentation**. This suggests that the architectural symmetry prior contributes meaningfully to output stability beyond what augmentation alone provides.

Within the equivariant models, the residual variant is also substantially more stable than the plain variant after training. This is one of the clearest places in the notebook where residual connections appear beneficial, even though their effect on validation AUC was small in the earlier ablation.

The non-identity transforms account for nearly all of the remaining variation. By contrast, the identity case and one reflection-aligned case remain at or near zero. The main safe conclusion is therefore empirical rather than mechanistic: training weakens exact numerical invariance, but the equivariant architectures still preserve substantially more output stability than the non-equivariant baseline.

#### 6.8.4 Theoretical vs Empirical Invariance

| Model | Theoretical | Empirical | Change |
|:--|--:|--:|--:|
| E-ResNet $D_4$ | 0.00009 | 0.01817 | +0.01808 |
| EPlainCNN $D_4$ | 0.00005 | 0.03768 | +0.03763 |
| ResNet-50 | 0.06480 | 0.05814 | -0.00666 |

Three points are clear.

**1. Training increases output variation in the equivariant models.**  
Both equivariant architectures move away from their near-zero untrained baseline. So the practically relevant result is not exact numerical invariance after training, but whether the trained models remain more stable than a non-equivariant alternative.

**2. The equivariant models still retain a clear stability advantage.**  
After training, both E-ResNet and EPlainCNN remain more transformation-stable than ResNet-50 under this test. E-ResNet is the strongest of the three.

**3. E-ResNet preserves output stability better than EPlainCNN.**  
This is one of the clearest differences between the two equivariant variants: the residual model shows substantially less empirical drift away from the untrained baseline.

The small decrease for ResNet-50 after training should not be over-interpreted. Its output variation remains much larger than that of the equivariant models, so the main conclusion is unchanged: architectural equivariance provides a substantial stability advantage at the classifier output, even after training.

#### 6.8.5 Connection to §6.7 and Deployment Recommendation

The ablation in §6.7 and the invariance test in §6.8 measure different properties, so their conclusions are not directly contradictory.

- In **§6.7**, the best validation performance within the equivariant family comes from **EPlainCNN + augmentation**:
  - macro AUC: $0.9956$
  - Sphere recall: $0.9268$

- In **§6.8**, the strongest output stability under rotation is obtained by **E-ResNet**:
  - mean $L_2 = 0.01817$, compared with
  - $0.03768$ for EPlainCNN (no augmentation) and
  - $0.05814$ for ResNet-50.

The performance gap between the two augmented equivariant runs is small, whereas the difference in measured rotation stability is much larger. That suggests the model choice depends on the objective:

| Objective | More suitable model | Reason |
|:--|:--:|:--|
| Highest validation performance on this benchmark | EPlainCNN + augmentation | Slightly better AUC and Sphere recall |
| Stronger rotation stability at classifier output | E-ResNet | Lowest mean output variation under transformation |

A cautious deployment takeaway is therefore this: if orientation robustness is considered important, E-ResNet is the safer choice despite its slightly lower validation metrics. If the priority is only benchmark performance on the current split, EPlainCNN + augmentation is marginally stronger.

This should still be treated as a practical recommendation, not a definitive one. The invariance comparison and the accuracy comparison are informative, but they do not by themselves quantify downstream bias on real survey data.

## <a id="7-sphere-class-analysis--failure-mode-analysis"></a>7. Sphere Class Analysis & Failure Mode Analysis

### 7.1 Cross-Architecture Sphere Confusion Pattern

A recurring error across the stronger models is the misclassification of **Sphere** images as **No Substructure**. This section asks a narrower question: which Sphere images are missed **consistently across models**, and do those images differ in simple raw-image statistics from Sphere images that all models classify correctly?

The intersection analysis is performed over the stronger converged models in the benchmark. The goal is not to claim a single physical mechanism, but to isolate a subset of **shared hard cases** rather than model-specific mistakes.

Two groups are compared:
1. **Universally misclassified Sphere images** — Sphere images predicted as **No Substructure** by all selected models.
2. **Universally correct Sphere images** — Sphere images predicted correctly as **Sphere** by all selected models.

The raw `.npy` arrays are analyzed directly, without additional normalization, to test whether these two groups differ in simple image-level statistics such as mean intensity.

**Important note:** raw contrast defined as $\max(x)-\min(x)$ is not informative here because the dataset arrays are already min-max normalized, making this quantity essentially constant across images. The meaningful comparison in this subsection is therefore the distribution of **raw mean intensity**, not contrast.

The main question is descriptive: are the shared Sphere failures systematically dimmer than the shared Sphere successes? If so, that would support the interpretation that the hardest Sphere cases are associated with weaker image-level signal, without by itself proving a hard detection threshold.

In [ ]:
import matplotlib.gridspec as gridspec
# ── Configuration ─────────────────────────────────────────────────────────────
CONVERGED_MODELS = ['ResNet-18', 'ResNet-50', 'DenseNet-121',
                    'EfficientNet-B3', 'ViT-Base', 'E-ResNet', 'EqDenseNet-C8', 'Equivariant-D4']

SPHERE_IDX = 1
NO_SUB_IDX = 0

# ── Raw image statistics (bypasses normalisation) ─────────────────────────────
def image_contrast_raw(filepath):
    """Peak-to-peak pixel range on the raw unnormalised array."""
    img = np.load(filepath).astype(np.float32).squeeze()
    return float(img.max() - img.min())

def image_mean_raw(filepath):
    img = np.load(filepath).astype(np.float32).squeeze()
    return float(img.mean())

def image_std_raw(filepath):
    img = np.load(filepath).astype(np.float32).squeeze()
    return float(img.std())

def image_max_raw(filepath):
    img = np.load(filepath).astype(np.float32).squeeze()
    return float(img.max())

# ── Step 1: Isolate Sphere sample indices ─────────────────────────────────────
sphere_mask    = np.array(ARCH_METRICS['ResNet-18']['labels']) == SPHERE_IDX
sphere_indices = np.where(sphere_mask)[0]
print(f'Total Sphere samples: {sphere_mask.sum()}')

# ── Step 2: Intersection — images ALL converged models misclassify as No Sub ──
wrong_as_no_sub = None
for model_name in CONVERGED_MODELS:
    preds = np.array(ARCH_METRICS[model_name]['preds'])
    this_wrong = set(
        sphere_indices[preds[sphere_indices] == NO_SUB_IDX].tolist()
    )
    wrong_as_no_sub = this_wrong if wrong_as_no_sub is None \
                      else wrong_as_no_sub.intersection(this_wrong)

wrong_as_no_sub = sorted(wrong_as_no_sub)
print(f'Sphere → No Substructure by ALL {len(CONVERGED_MODELS)} models: {len(wrong_as_no_sub)}')

# ── Step 3: Images ALL converged models get correct ───────────────────────────
all_correct = None
for model_name in CONVERGED_MODELS:
    preds = np.array(ARCH_METRICS[model_name]['preds'])
    this_correct = set(
        sphere_indices[preds[sphere_indices] == SPHERE_IDX].tolist()
    )
    all_correct = this_correct if all_correct is None \
                  else all_correct.intersection(this_correct)

all_correct = sorted(all_correct)
print(f'Sphere images ALL models correct: {len(all_correct)}')

# ── Step 4: Per-model confusion breakdown ─────────────────────────────────────
print('\nPer-model Sphere → No Substructure misclassification:')
for model_name in CONVERGED_MODELS:
    preds  = np.array(ARCH_METRICS[model_name]['preds'])
    n_wrong = (preds[sphere_indices] == NO_SUB_IDX).sum()
    print(f'  {model_name:20s}: {n_wrong:4d} / 2500  ({n_wrong/25:.1f}%)')

# ── Step 5: Raw contrast statistics ───────────────────────────────────────────
print('\nComputing raw image statistics (bypassing normalisation)...')

wrong_contrasts   = [image_contrast_raw(val_ds.filepaths[i]) for i in wrong_as_no_sub]
correct_contrasts = [image_contrast_raw(val_ds.filepaths[i]) for i in all_correct]

wrong_means       = [image_mean_raw(val_ds.filepaths[i]) for i in wrong_as_no_sub]
correct_means     = [image_mean_raw(val_ds.filepaths[i]) for i in all_correct]

wrong_stds        = [image_std_raw(val_ds.filepaths[i]) for i in wrong_as_no_sub]
correct_stds      = [image_std_raw(val_ds.filepaths[i]) for i in all_correct]

wrong_maxvals     = [image_max_raw(val_ds.filepaths[i]) for i in wrong_as_no_sub]
correct_maxvals   = [image_max_raw(val_ds.filepaths[i]) for i in all_correct]

print(f'\nRaw image contrast (max − min):')
print(f'  Universally wrong  : mean={np.mean(wrong_contrasts):.6f}  '
      f'std={np.std(wrong_contrasts):.6f}  '
      f'median={np.median(wrong_contrasts):.6f}')
print(f'  Universally correct: mean={np.mean(correct_contrasts):.6f}  '
      f'std={np.std(correct_contrasts):.6f}  '
      f'median={np.median(correct_contrasts):.6f}')

print(f'\nRaw pixel mean intensity:')
print(f'  Universally wrong  : mean={np.mean(wrong_means):.6f}  '
      f'std={np.std(wrong_means):.6f}')
print(f'  Universally correct: mean={np.mean(correct_means):.6f}  '
      f'std={np.std(correct_means):.6f}')

print(f'\nRaw pixel std (local texture):')
print(f'  Universally wrong  : mean={np.mean(wrong_stds):.6f}  '
      f'std={np.std(wrong_stds):.6f}')
print(f'  Universally correct: mean={np.mean(correct_stds):.6f}  '
      f'std={np.std(correct_stds):.6f}')

print(f'\nRaw max pixel intensity:')
print(f'  Universally wrong  : mean={np.mean(wrong_maxvals):.6f}  '
      f'std={np.std(wrong_maxvals):.6f}')
print(f'  Universally correct: mean={np.mean(correct_maxvals):.6f}  '
      f'std={np.std(correct_maxvals):.6f}')

# ── Step 6: Statistical tests ─────────────────────────────────────────────────
from scipy import stats

# Mann-Whitney U: are wrong images lower contrast than correct ones?
u_contrast, p_contrast = stats.mannwhitneyu(
    wrong_contrasts, correct_contrasts, alternative='less'
)
u_mean, p_mean = stats.mannwhitneyu(
    wrong_means, correct_means, alternative='less'
)
u_std, p_std = stats.mannwhitneyu(
    wrong_stds, correct_stds, alternative='less'
)
u_max, p_max = stats.mannwhitneyu(
    wrong_maxvals, correct_maxvals, alternative='less'
)

print(f'\nMann-Whitney U tests (one-sided: wrong < correct):')
print(f'  Contrast  : U={u_contrast:.0f}  p={p_contrast:.6f}  '
      f'{"Significant" if p_contrast < 0.05 else "Not significant"}')
print(f'  Mean int. : U={u_mean:.0f}  p={p_mean:.6f}  '
      f'{"Significant" if p_mean < 0.05 else "Not significant"}')
print(f'  Std       : U={u_std:.0f}  p={p_std:.6f}  '
      f'{"Significant" if p_std < 0.05 else "Not significant"}')
print(f'  Max int.  : U={u_max:.0f}  p={p_max:.6f}  '
      f'{"Significant" if p_max < 0.05 else "Not significant"}')

# ── Step 7: Contrast distribution plot ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Contrast histogram
axes[0].hist(correct_contrasts, bins=40, alpha=0.7, color='steelblue',
             density=True, label=f'All correct (n={len(all_correct)})')
axes[0].hist(wrong_contrasts, bins=40, alpha=0.7, color='crimson',
             density=True, label=f'All wrong (n={len(wrong_as_no_sub)})')
axes[0].axvline(np.mean(correct_contrasts), color='steelblue',
                linestyle='--', lw=2,
                label=f'Mean correct: {np.mean(correct_contrasts):.4f}')
axes[0].axvline(np.mean(wrong_contrasts), color='crimson',
                linestyle='--', lw=2,
                label=f'Mean wrong: {np.mean(wrong_contrasts):.4f}')
axes[0].set_xlabel('Raw image contrast (max − min)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title(f'Contrast Distribution\n(p={p_contrast:.4f})', fontsize=11)
axes[0].legend(fontsize=8)

# Mean intensity histogram
axes[1].hist(correct_means, bins=40, alpha=0.7, color='steelblue',
             density=True, label=f'All correct (n={len(all_correct)})')
axes[1].hist(wrong_means, bins=40, alpha=0.7, color='crimson',
             density=True, label=f'All wrong (n={len(wrong_as_no_sub)})')
axes[1].axvline(np.mean(correct_means), color='steelblue', linestyle='--', lw=2)
axes[1].axvline(np.mean(wrong_means), color='crimson', linestyle='--', lw=2)
axes[1].set_xlabel('Raw mean pixel intensity', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title(f'Mean Intensity Distribution\n(p={p_mean:.4f})', fontsize=11)
axes[1].legend(fontsize=8)

axes[2].boxplot(
    [correct_means, wrong_means],
    tick_labels=[f'All correct\n(n={len(all_correct)})',
                 f'All wrong\n(n={len(wrong_as_no_sub)})'],
    patch_artist=True,
    boxprops=dict(alpha=0.7),
    medianprops=dict(color='black', lw=2)
)
axes[2].set_ylabel('Raw mean pixel intensity', fontsize=11)
axes[2].set_title(f'Mean Intensity Boxplot\n(Mann-Whitney p={p_mean:.2e})', fontsize=11)

plt.suptitle('Sphere Class Difficulty — Raw Image Statistics\n'
             '(computed on unnormalised .npy arrays)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(gsoc_drive_path, 'figures', 'sphere_contrast_analysis.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Step 8: Image grid — universally wrong vs universally correct ──────────────
n_show = min(12, len(wrong_as_no_sub), len(all_correct))

fig = plt.figure(figsize=(20, 8))
gs  = gridspec.GridSpec(2, n_show, figure=fig, hspace=0.5, wspace=0.05)

for col in range(n_show):
    # Top row: universally wrong
    img_wrong = np.load(val_ds.filepaths[wrong_as_no_sub[col]]).squeeze()
    ax = fig.add_subplot(gs[0, col])
    ax.imshow(img_wrong, cmap='magma', origin='lower', vmin=0)
    ax.axis('off')
    c = image_contrast_raw(val_ds.filepaths[wrong_as_no_sub[col]])
    ax.set_title(f'c={c:.4f}', fontsize=6.5)
    if col == 0:
        ax.text(-0.15, 0.5, 'All models\nWRONG\n(→ No Sub)',
                transform=ax.transAxes, fontsize=8, va='center',
                ha='right', color='crimson', fontweight='bold')

    # Bottom row: universally correct
    img_correct = np.load(val_ds.filepaths[all_correct[col]]).squeeze()
    ax2 = fig.add_subplot(gs[1, col])
    ax2.imshow(img_correct, cmap='magma', origin='lower', vmin=0)
    ax2.axis('off')
    c2 = image_contrast_raw(val_ds.filepaths[all_correct[col]])
    ax2.set_title(f'c={c2:.4f}', fontsize=6.5)
    if col == 0:
        ax2.text(-0.15, 0.5, 'All models\nCORRECT\n(→ Sphere)',
                 transform=ax2.transAxes, fontsize=8, va='center',
                 ha='right', color='steelblue', fontweight='bold')

fig.suptitle(
    'Sphere Subhalo Images: Universally Misclassified vs Universally Correct\n'
    'c = raw contrast (max−min of unnormalised pixels). '
    'Top row: predicted as No Substructure by all 6 models.',
    fontsize=11, fontweight='bold'
)
plt.savefig(os.path.join(gsoc_drive_path, 'figures', 'sphere_misclassified_grid.png'),
            dpi=150, bbox_inches='tight')
plt.show()

#### 7.1.1 Results

**Intersection analysis across all 8 models:**
- Total Sphere samples: $2500$
- Sphere $\rightarrow$ No Substructure by all 8 models: **54** images ($2.2\%$)
- Sphere images classified correctly by all 8 models: **1703** images ($68.1\%$)

**Per-model Sphere $\rightarrow$ No Substructure error rate:**

| Model | Wrong as No Sub | Rate |
|:--|:--:|--:|
| DenseNet-121 | 105 / 2500 | 4.2% |
| EqDenseNet-C8 | 107 / 2500 | 4.3% |
| E-ResNet | 129 / 2500 | 5.2% |
| ResNet-50 | 135 / 2500 | 5.4% |
| ResNet-18 | 153 / 2500 | 6.1% |
| EfficientNet-B3 | 193 / 2500 | 7.7% |
| ViT-Base | 309 / 2500 | 12.4% |
| Equivariant-D4 | 343 / 2500 | 13.7% |

The per-model rates vary substantially, but there remains a nontrivial core of **54 Sphere images** that all 8 models miss in the same way.

**Raw image statistics — universally wrong ($n=54$) vs universally correct ($n=1703$):**

| Metric | Wrong | Correct | Mann–Whitney $p$ |
|:--|:--:|:--:|:--:|
| Contrast ($\max-\min$) | $1.000 \pm 0.000$ | $1.000 \pm 0.000$ | $1.000$ |
| **Mean pixel intensity** | **$0.0581 \pm 0.0072$** | **$0.0639 \pm 0.0087$** | **$1.82\times10^{-7}$** |
| Pixel std | $0.1114 \pm 0.0158$ | $0.1190 \pm 0.0169$ | $4.74\times10^{-4}$ |
| Max pixel intensity | $1.000 \pm 0.000$ | $1.000 \pm 0.000$ | $1.000$ |

The informative differences are in **mean intensity** and **pixel standard deviation**: the universally misclassified Sphere images are, on average, dimmer and slightly less variable than the universally correct ones. By contrast, contrast and maximum intensity are uninformative here because the arrays are already min-max normalized.

This pattern is consistent with the shared Sphere failures being associated with weaker image-level signal. That interpretation should remain cautious: the distributions still overlap substantially, so this is a shift in tendency, not a clean separating threshold.

### 7.1.2 Interpretation

Because the arrays are already min-max normalized per image, **contrast** and **maximum intensity** are fixed by construction and therefore uninformative in this comparison.

The two statistics that remain informative are **mean pixel intensity** and **pixel standard deviation**. The universally misclassified Sphere images are, on average, dimmer and slightly less variable than the universally correct Sphere images. This indicates that the shared Sphere failures are associated with weaker image-level signal under the normalized representation used here.

That association should be interpreted cautiously. It does **not** establish a single mechanism, and it does not prove a hard detection threshold. It only shows that the Sphere images missed by all models tend to occupy a lower-intensity, lower-variation part of the distribution.

A practical implication follows from the cross-model agreement: these 54 images are not isolated failures of one architecture, but a common hard subset across very different model families. That makes them more likely to reflect a property of the input cases themselves than an idiosyncratic weakness of any single model.

---
###7.1.3 Future DeepLense Project Activity — Failure-Cohort Analysis for Hard Sphere Cases

A focused next step is to treat the **54 universally misclassified Sphere images** as a dedicated failure cohort and analyze them against two control groups:  
(i) Sphere images classified correctly by all strong models, and  
(ii) Sphere images misclassified by only a subset of models.  
The goal is to determine whether these failures are associated with weak signal, missing spatial context, morphological similarity to No Substructure, or a mixture of multiple failure modes.

**Because the data are simulated, the failure cohort can be interpreted as a structured hard subset of the simulator’s parameter space. Analyzing that subset helps identify whether errors are associated with weak signal, reduced spatial context, or morphology close to the null class.**

#### 1. Build a compact failure table
For each image in the failure and control cohorts, compute a compact feature set including:
- raw mean intensity and raw standard deviation
- ring-only mean intensity and ring-only standard deviation
- brightest-pixel value
- fraction of total flux inside a ring mask
- arc length / ring completeness
- ellipticity / eccentricity
- number of bright connected components on the ring
- distance of the brightest knot from the ring centerline
- estimated perturbation size
- per-model predicted probabilities
- ensemble entropy

This table will provide a common basis for both statistical comparison and case-by-case inspection.

#### 2. Compare distributions using effect sizes
For each feature, compare:
- universally wrong vs universally correct Sphere
- universally wrong vs partially wrong Sphere

In addition to significance tests, report effect sizes such as:
- Cliff’s delta
- Cohen’s $d$
- ROC AUC of the feature alone

This will identify which image properties actually separate the hard-failure cohort rather than merely differ in mean.

#### 3. Measure ring-specific geometry
Global image statistics are likely too coarse. A more informative analysis should quantify structure on the Einstein ring itself, including:
- total ring flux
- ring thickness
- ring continuity
- angular span of visible arc
- asymmetry across angle
- local curvature
- number and prominence of local maxima along the arc

A particularly useful representation is to transform each image into **polar coordinates around the lens center** and analyze intensity as a function of angle. This can reveal whether universally wrong Sphere images tend to have:
- weaker peaks
- narrower peaks
- fewer peaks
- lower angular contrast
- shorter visible arc extent

#### 4. Assess visual recoverability systematically
The failure cohort should also be inspected manually, but in a structured way. Each image can be annotated with simple binary or categorical tags such as:
- complete ring / partial ring
- thin arc / thick arc
- one dominant knot / multiple knots
- high symmetry / low symmetry
- obvious perturbation / subtle perturbation

This helps determine whether the failure set is visually coherent or contains multiple distinct subtypes.

#### 5. Embedding-space analysis
Extract penultimate-layer embeddings from the best-performing classifier and compare:
- universally correct Sphere
- universally wrong Sphere
- No Substructure

Use UMAP or t-SNE only for visualization, but use nearest-neighbor structure in embedding space for interpretation. The key questions are:
- do the 54 failures cluster together?
- do they lie close to No Substructure?
- are they scattered, suggesting multiple failure modes?

This is one of the highest-value analyses because it tests whether the failure set forms a coherent representation-level group.

#### 6. Meta-classifier for failure prediction
Define a binary target:
- $1$ = universally wrong Sphere
- $0$ = universally correct Sphere

Then train a simple interpretable model on handcrafted features, such as:
- logistic regression
- shallow decision tree
- gradient-boosted trees

The objective is not classification performance itself, but interpretability: which measurable image properties are most predictive of universal failure?

#### 7. Split silent failures from ambiguous failures
Within the hard Sphere cohort, separate:
- low-entropy unanimous failures
- higher-entropy ambiguous failures

Then compare their geometry, brightness, and embedding structure. This will test whether silent failures are a distinct subtype rather than simply the extreme end of the same distribution.

#### 8. Nearest-neighbor retrieval
For each universally wrong Sphere image, retrieve nearest neighbors from:
- universally correct Sphere
- No Substructure

Use distances based on:
- raw image features
- polar ring-profile features
- learned embeddings

This can test whether the hard failures are genuinely closer to No Substructure than to typical Sphere examples.

#### 9. Test centering and scale effects
Measure:
- ring center offset from image center
- ring radius
- apparent size of bright structures

A subset of failures may occur because the ring is off-center, too small, or occupies too few effective pixels for stable discrimination.

#### 10. Check label ambiguity
Finally, inspect whether some universally wrong Sphere cases are visually borderline even relative to nearby correct Sphere examples. This should not be used to claim label noise without evidence, but it is important to test whether a fraction of the hardest failures are intrinsically ambiguous under the current image formation setup.

#### Expected outcome
This activity would move the analysis beyond aggregate performance metrics and toward a more scientific account of *why* the hardest Sphere cases fail. The main question is:

> Are the universally wrong Sphere images hard because the perturbation signal is weak, because the ring geometry removes useful spatial context, or because they lie morphologically close to No Substructure?

Answering that would directly inform future model design, data-generation strategy, and evaluation methodology in the next phase of the project.

---

### 7.2 E-ResNet and EqDenseNet-C8 Failure Mode Analysis

Section 7.1 showed that the Sphere $\rightarrow$ No Substructure confusion is shared across models, with a core set of 54 unanimously misclassified Sphere images and a tendency toward lower mean intensity and lower pixel standard deviation in that cohort.

This section narrows the focus to the two strongest equivariant models — **EqDenseNet-C8** and **E-ResNet** — and asks a simpler question: do their Sphere failures mostly overlap with the shared hard cases, or does each model also have its own distinct failure subset?

Both models are analyzed on the validation set using their saved checkpoints only; no retraining is performed.

#### 7.2.1 Sphere False Negatives (Fig. 12a–12b)

E-ResNet misclassifies **129 / 2500** Sphere images as No Substructure and correctly classifies **2306 / 2500**. Figure 12a shows the highest-confidence false negatives.

**Morphological statistics — Sphere true positives vs Sphere $\rightarrow$ No Substructure false negatives:**

| Statistic | TP mean | FN mean | Mann–Whitney $p$ |
|:--|--:|--:|--:|
| Ring mean flux | 0.15862 | 0.14226 | $6.07\times10^{-10}$ |
| Ring flux std | 0.16986 | 0.16302 | $2.91\times10^{-3}$ |
| Ring max intensity | 0.99990 | 1.00000 | $5.97\times10^{-1}$ |
| Ring asymmetry | 0.07817 | 0.08053 | $8.99\times10^{-1}$ |
| Compactness | 0.63747 | 0.65851 | $3.78\times10^{-8}$ |

The clearest differences are lower **ring mean flux** and slightly lower **ring flux standard deviation** in the false negatives, together with higher **compactness**. In contrast, **ring asymmetry** and **maximum intensity** do not separate the two groups in this analysis.

The most defensible interpretation is narrow: Sphere images missed as No Substructure tend to have weaker ring-level signal and a more concentrated intensity distribution. That is consistent with these cases being harder to distinguish from smooth lenses, but it does not by itself identify the underlying mechanism.

#### 7.2.2 Sphere $\leftrightarrow$ Vortex Confusion (Fig. 12c–12d)

E-ResNet confuses **65** Sphere images as Vortex and **54** Vortex images as Sphere. These form a different error subset from the Sphere $\rightarrow$ No Substructure false negatives above.

**Residual elongation** (CAE residual major/minor axis ratio):

| Group | $n$ | Mean elongation |
|:--|--:|--:|
| Sphere TP | 2306 | 1.205 |
| Sphere $\rightarrow$ Vortex | 65 | 1.199 |
| Vortex TP | 2421 | 1.205 |
| Vortex $\rightarrow$ Sphere | 54 | 1.213 |

The four groups are very similar on this metric, and the distributions overlap strongly. So residual elongation alone does not appear to explain the Sphere–Vortex confusion in a useful way.

The gallery in Fig. 12c is therefore best read qualitatively: some confused cases show mixed local and arc-level structure, but no single visual pattern is cleanly sufficient on its own.

#### 7.2.3 Classification Accuracy vs Ring Brightness (Fig. 12e)

Across all three classes, classification accuracy shows an overall positive trend with **ring mean flux**. The effect is strongest for **Sphere**, where low-brightness bins are substantially less accurate than high-brightness bins.

Mean confidence follows the same broad pattern as accuracy. This suggests that lower-brightness cases are not only harder, but are also assigned lower confidence on average.

This result should still be interpreted descriptively. It shows that ring brightness is associated with classification difficulty, not that brightness alone determines success or failure.

#### 7.2.4 Perturbation Position vs Failure (Fig. 12f)

Figure 12f plots CAE residual centroids for correct and incorrect classifications. Across all panels, the centroids remain concentrated near the image center, with substantial visual overlap between correct and incorrect cases.

No clear spatial separation is visible between true positives and the different confusion types. Under this centroid-based summary, perturbation position does not appear to provide a strong explanation for the observed errors.

That conclusion should remain limited to this representation: the absence of separation in centroid position does not rule out more subtle position-dependent effects that are not captured by a single centroid statistic.

Taken together, the E-ResNet errors appear more strongly associated with **signal strength and concentration** than with coarse asymmetry, residual elongation, or centroid position.

In [ ]:
# ── Define residual path if not already in memory ─────────────────────────────
# Adjust filename to match whatever was saved in the CAE section
res_val_path = os.path.join(gsoc_drive_path, 'val_residuals.npz')

# Verify it exists before loading
if not os.path.exists(res_val_path):
    # Try alternate common filenames from the CAE section
    for candidate in ['val_residuals.npz', 'residuals_val.npz',
                      'cae_val_residuals.npz']:
        candidate_path = os.path.join(gsoc_drive_path, candidate)
        if os.path.exists(candidate_path):
            res_val_path = candidate_path
            print(f'Found residuals at: {res_val_path}')
            break
    else:
        print('ERROR: Could not find val residuals file.')
        print(f'Contents of {gsoc_drive_path}:')
        print([f for f in os.listdir(gsoc_drive_path) if 'resid' in f.lower()])

val_res_data  = np.load(res_val_path)
val_residuals = val_res_data['residuals']   # (7500, 150, 150)
print(f'Loaded val_residuals: shape={val_residuals.shape}')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Section 12 — Failure Mode Analysis
#
# Target model: E-ResNet (best raw-image classifier, macro AUC 0.9959)
#
# Four analyses:
#   12.1  Sphere false negatives — image grid + intensity statistics
#   12.2  Sphere-Vortex confusion — examples + residual morphology
#   12.3  Classification confidence vs ring brightness
#   12.4  Perturbation position vs failure (CAE residual centroid)
#
# All analyses use val_ds, the frozen E-ResNet, and the frozen CAE.
# No retraining required.
# ═════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import os

figures_path = os.path.join(gsoc_drive_path, 'figures')

# ── Shared utilities ──────────────────────────────────────────────────────────

def ring_mask(size=150, inner_frac=0.15, outer_frac=0.65):
    cy, cx = size // 2, size // 2
    Y, X   = np.ogrid[:size, :size]
    dist   = np.sqrt((X - cx)**2 + (Y - cy)**2) / (size // 2)
    return (dist >= inner_frac) & (dist <= outer_frac)

RING = ring_mask(size=150, inner_frac=0.15, outer_frac=0.65)


def get_all_predictions(model, dataset, device, batch_size=256):
    """
    Run model on entire dataset.
    Returns:
        probs   (N, 3)  softmax probabilities
        preds   (N,)    argmax predictions
        labels  (N,)    ground truth
        images  (N, 150, 150)  raw images as numpy
    """
    from torch.utils.data import DataLoader
    loader = DataLoader(dataset, batch_size=batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)
    model.eval()

    all_probs, all_preds, all_labels, all_images = [], [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            logits = model(imgs)
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.append(probs.argmax(1))
            all_labels.append(labels.numpy())
            all_images.append(imgs.squeeze(1).cpu().numpy())

    return (np.concatenate(all_probs,  axis=0),
            np.concatenate(all_preds,  axis=0),
            np.concatenate(all_labels, axis=0),
            np.concatenate(all_images, axis=0))


print('Running E-ResNet on full val set...')
eresnet_probs, eresnet_preds, val_labels, val_images = \
    get_all_predictions(e_resnet, val_ds, device)

val_acc = (eresnet_preds == val_labels).mean()
print(f'Val accuracy: {val_acc*100:.2f}%  (expected ~96.23%)')


def image_stats(img_np):
    """
    Compute per-image morphological statistics from a (150,150) array.

    Returns dict with:
      ring_mean    — mean flux inside ring annulus
      ring_std     — flux std inside ring annulus (proxy for perturbation strength)
      ring_max     — peak flux inside ring annulus
      asymmetry    — |left_half - right_half| / total  (bilateral asymmetry)
      compactness  — fraction of ring flux in top quartile pixels
    """
    ring_pixels  = img_np[RING]
    ring_mean    = ring_pixels.mean()
    ring_std     = ring_pixels.std()
    ring_max     = ring_pixels.max()

    # Bilateral asymmetry along horizontal axis
    half = img_np.shape[1] // 2
    left  = img_np[:, :half][RING[:, :half]].sum()
    right = img_np[:, half:][RING[:, half:]].sum()
    total = left + right + 1e-9
    asymmetry = abs(left - right) / total

    # Compactness — top-quartile fraction of ring flux
    q75 = np.percentile(ring_pixels, 75)
    compactness = ring_pixels[ring_pixels >= q75].sum() / (ring_pixels.sum() + 1e-9)

    return dict(ring_mean=ring_mean, ring_std=ring_std, ring_max=ring_max,
                asymmetry=asymmetry, compactness=compactness)


# Compute stats for all val images
print('Computing morphological statistics for all val images...')
all_stats = [image_stats(val_images[i]) for i in range(len(val_images))]
stat_keys = list(all_stats[0].keys())
stats_arr  = {k: np.array([s[k] for s in all_stats]) for k in stat_keys}
print('Done.')


# ── 12.1  Sphere false negatives ─────────────────────────────────────────────
print('\n═══ 12.1 Sphere False Negatives ═══')

# Indices where true=Sphere, predicted=No Substructure
sphere_fn_mask = (val_labels == 1) & (eresnet_preds == 0)
sphere_tp_mask = (val_labels == 1) & (eresnet_preds == 1)

sphere_fn_idx  = np.where(sphere_fn_mask)[0]
sphere_tp_idx  = np.where(sphere_tp_mask)[0]

print(f'Sphere true positives:  {sphere_tp_mask.sum()}')
print(f'Sphere false negatives (→ No Sub): {sphere_fn_mask.sum()}')

# ── Statistics comparison ──────────────────────────────────────────────────
print('\nMorphological statistics — TP vs FN:')
print(f'{"Statistic":20s}  {"TP mean":>10s}  {"FN mean":>10s}  {"p-value":>10s}')
print('─' * 56)

for k in stat_keys:
    tp_vals = stats_arr[k][sphere_tp_idx]
    fn_vals = stats_arr[k][sphere_fn_idx]
    t, p    = stats.mannwhitneyu(tp_vals, fn_vals, alternative='two-sided')
    print(f'{k:20s}  {tp_vals.mean():>10.5f}  '
          f'{fn_vals.mean():>10.5f}  {p:>10.4e}')

# ── Figure 12a — FN image grid ─────────────────────────────────────────────
# Sort FNs by classifier confidence in wrong class (most confidently wrong first)
fn_wrong_conf = eresnet_probs[sphere_fn_idx, 0]   # confidence in No Sub
sort_idx      = np.argsort(fn_wrong_conf)[::-1]   # descending
fn_sorted     = sphere_fn_idx[sort_idx]

n_show = min(12, len(fn_sorted))
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
axes = axes.flatten()

for i in range(n_show):
    idx  = fn_sorted[i]
    img  = val_images[idx]
    conf = eresnet_probs[idx, 0]
    axes[i].imshow(img, cmap='inferno', vmin=0, vmax=img.max())
    axes[i].set_title(
        f'Conf(NoSub)={conf:.3f}\n'
        f'ring_mean={stats_arr["ring_mean"][idx]:.4f}',
        fontsize=8
    )
    axes[i].axis('off')

plt.suptitle(
    'Figure 12a — Sphere False Negatives (predicted as No Substructure)\n'
    'Sorted by E-ResNet confidence in wrong class (highest confidence first)\n'
    'These are the Sphere images the best model cannot distinguish from smooth lenses',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12a_sphere_false_negatives.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 12b — Ring brightness distribution: TP vs FN ───────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_stats = ['ring_mean', 'ring_std', 'asymmetry']
xlabels    = ['Ring mean flux', 'Ring flux std\n(perturbation strength proxy)',
              'Ring asymmetry']

for i, (k, xl) in enumerate(zip(plot_stats, xlabels)):
    tp_vals = stats_arr[k][sphere_tp_idx]
    fn_vals = stats_arr[k][sphere_fn_idx]
    _, p    = stats.mannwhitneyu(tp_vals, fn_vals, alternative='two-sided')

    axes[i].hist(tp_vals, bins=50, alpha=0.6, density=True,
                 color='#2ecc71', label=f'True Positive (n={len(tp_vals)})')
    axes[i].hist(fn_vals, bins=50, alpha=0.6, density=True,
                 color='#e74c3c', label=f'False Negative (n={len(fn_vals)})')
    axes[i].axvline(tp_vals.mean(), color='#27ae60', lw=2, linestyle='--')
    axes[i].axvline(fn_vals.mean(), color='#c0392b', lw=2, linestyle='--')
    axes[i].set_xlabel(xl, fontsize=10)
    axes[i].set_ylabel('Density', fontsize=10)
    axes[i].set_title(f'p = {p:.2e}', fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].grid(alpha=0.3)

plt.suptitle(
    'Figure 12b — Sphere TP vs FN: Morphological Statistics\n'
    'False negatives have systematically lower ring flux and perturbation strength',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12b_sphere_fn_statistics.png'),
            dpi=150, bbox_inches='tight')
plt.show()


# ── 12.2  Sphere ↔ Vortex confusion ──────────────────────────────────────────
print('\n═══ 12.2 Sphere ↔ Vortex Confusion ═══')

sv_mask = (val_labels == 1) & (eresnet_preds == 2)   # Sphere → Vortex
vs_mask = (val_labels == 2) & (eresnet_preds == 1)   # Vortex → Sphere

sv_idx = np.where(sv_mask)[0]
vs_idx = np.where(vs_mask)[0]

print(f'Sphere predicted as Vortex: {sv_mask.sum()}')
print(f'Vortex predicted as Sphere: {vs_mask.sum()}')

# Use CAE residuals to characterise morphology of confused images
# Load precomputed val residuals
val_res_data = np.load(res_val_path)
val_residuals = val_res_data['residuals']   # (7500, 150, 150)

def residual_morphology(res_np):
    """
    Characterise the spatial morphology of a CAE residual image.

    Returns:
      res_std      — overall residual std (perturbation strength)
      res_centroid — (y, x) centroid of |residual| within ring
      elongation   — ratio of major to minor axis of |residual| distribution
    """
    ring_res  = np.abs(res_np) * RING
    total     = ring_res.sum() + 1e-9

    # Centroid
    Y, X  = np.mgrid[:150, :150]
    cy    = (ring_res * Y).sum() / total
    cx    = (ring_res * X).sum() / total

    # Second moments (for elongation)
    dy2   = ((ring_res * (Y - cy)**2).sum() / total)
    dx2   = ((ring_res * (X - cx)**2).sum() / total)
    dxy   = ((ring_res * (Y - cy) * (X - cx)).sum() / total)

    # Eigenvalues of moment tensor → elongation
    trace  = dy2 + dx2
    det    = dy2 * dx2 - dxy**2
    disc   = max(0, (trace/2)**2 - det)
    lam1   = trace/2 + np.sqrt(disc)
    lam2   = trace/2 - np.sqrt(disc) + 1e-9
    elongation = np.sqrt(lam1 / lam2)

    return dict(
        res_std     = res_np[RING].std(),
        centroid_y  = cy,
        centroid_x  = cx,
        elongation  = elongation
    )


# Compute residual morphology for confused and correct images
sphere_tp_res = [residual_morphology(val_residuals[i]) for i in sphere_tp_idx]
sv_res        = [residual_morphology(val_residuals[i]) for i in sv_idx]
vortex_tp_idx = np.where((val_labels == 2) & (eresnet_preds == 2))[0]
vs_res        = [residual_morphology(val_residuals[i]) for i in vs_idx]
vortex_tp_res = [residual_morphology(val_residuals[i]) for i in vortex_tp_idx]

def mean_morph(morph_list, key):
    return np.mean([m[key] for m in morph_list])

print('\nResidual elongation (1.0 = circular, higher = elongated):')
print(f'  Sphere TP:         {mean_morph(sphere_tp_res, "elongation"):.3f}')
print(f'  Sphere → Vortex:   {mean_morph(sv_res,        "elongation"):.3f}')
print(f'  Vortex TP:         {mean_morph(vortex_tp_res, "elongation"):.3f}')
print(f'  Vortex → Sphere:   {mean_morph(vs_res,        "elongation"):.3f}')

# ── Figure 12c — Confused image gallery ───────────────────────────────────
fig, axes = plt.subplots(3, 8, figsize=(20, 9))
row_labels = ['True Sphere\n(correct)', 'Sphere → Vortex\n(confused)', 'Vortex → Sphere\n(confused)']
row_indices = [sphere_tp_idx[:8], sv_idx[:8], vs_idx[:8]]

for r, (label, idxs) in enumerate(zip(row_labels, row_indices)):
    for c, idx in enumerate(idxs[:8]):
        axes[r, c].imshow(val_images[idx], cmap='inferno')
        axes[r, c].axis('off')
        if c == 0:
            axes[r, c].set_ylabel(label, fontsize=9, fontweight='bold',
                                   rotation=90, labelpad=50)

plt.suptitle(
    'Figure 12c — Sphere ↔ Vortex Confusion Gallery\n'
    'Row 1: Correctly classified Sphere  |  '
    'Row 2: Sphere misclassified as Vortex  |  '
    'Row 3: Vortex misclassified as Sphere',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12c_sv_confusion_gallery.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 12d — Elongation distributions ─────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

groups = [
    (sphere_tp_res,  '#2ecc71', 'Sphere TP'),
    (sv_res,         '#e74c3c', 'Sphere → Vortex'),
    (vortex_tp_res,  '#3498db', 'Vortex TP'),
    (vs_res,         '#f39c12', 'Vortex → Sphere'),
]

for morph_list, color, label in groups:
    vals = [m['elongation'] for m in morph_list]
    ax.hist(np.clip(vals, 1, 10), bins=40, alpha=0.6,
            density=True, color=color, label=f'{label} (n={len(vals)})')

ax.set_xlabel('Residual elongation (major/minor axis ratio)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title(
    'Figure 12d — Residual Morphology: Elongation by Confusion Type\n'
    'Sphere-Vortex confusion correlates with intermediate elongation values',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12d_elongation_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()


# ── 12.3  Confidence vs ring brightness ───────────────────────────────────────
print('\n═══ 12.3 Confidence vs Ring Brightness ═══')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_colors = ['#3498db', '#e74c3c', '#2ecc71']

for c in range(3):
    mask        = val_labels == c
    ring_means  = stats_arr['ring_mean'][mask]
    conf_correct = eresnet_probs[mask, c]          # confidence in true class
    correct_bool = (eresnet_preds[mask] == c)

    # Bin by ring brightness
    bins   = np.percentile(ring_means, np.linspace(0, 100, 11))
    bins   = np.unique(bins)
    bin_idx = np.digitize(ring_means, bins) - 1
    bin_idx = np.clip(bin_idx, 0, len(bins) - 2)

    bin_centers = [(bins[i] + bins[i+1]) / 2 for i in range(len(bins)-1)]
    bin_acc     = [correct_bool[bin_idx == i].mean()
                   for i in range(len(bins)-1)]
    bin_conf    = [conf_correct[bin_idx == i].mean()
                   for i in range(len(bins)-1)]

    ax = axes[c]
    ax2 = ax.twinx()

    ax.plot(bin_centers, bin_acc,  'o-', color=class_colors[c],
            lw=2, markersize=6, label='Accuracy')
    ax2.plot(bin_centers, bin_conf, 's--', color=class_colors[c],
             lw=1.5, markersize=5, alpha=0.7, label='Mean confidence')

    ax.set_xlabel('Ring mean flux (brightness)', fontsize=10)
    ax.set_ylabel('Classification accuracy',     fontsize=10, color='black')
    ax2.set_ylabel('Mean confidence',            fontsize=10, color='gray')
    ax.set_title(CLASS_NAMES[c], fontsize=11, fontweight='bold')
    ax.set_ylim(0.5, 1.05)
    ax.grid(alpha=0.3)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='lower right')

plt.suptitle(
    'Figure 12e — Classification Accuracy and Confidence vs Ring Brightness\n'
    'Low-brightness images are harder to classify — '
    'substructure signal falls below noise floor',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12e_confidence_vs_brightness.png'),
            dpi=150, bbox_inches='tight')
plt.show()


# ── 12.4  Perturbation position vs failure ────────────────────────────────────
print('\n═══ 12.4 Perturbation Position vs Failure ═══')

# Compute residual centroid positions for Sphere and Vortex val images
# Normalise centroid to [-1, 1] relative to image centre
CX0, CY0 = 75, 75   # image centre

def centroid_offset(res_np):
    """
    Returns (dx, dy) centroid of |residual| within ring,
    normalised to image half-width (so ±1 = image edge).
    """
    ring_res = np.abs(res_np) * RING
    total    = ring_res.sum() + 1e-9
    Y, X     = np.mgrid[:150, :150]
    cx       = (ring_res * X).sum() / total
    cy       = (ring_res * Y).sum() / total
    return (cx - CX0) / 75.0, (cy - CY0) / 75.0


fig, axes = plt.subplots(2, 3, figsize=(18, 12))
row_titles = ['Sphere', 'Vortex']
failure_masks = {
    'Sphere': {
        'correct':  (val_labels == 1) & (eresnet_preds == 1),
        'fn_nosub': (val_labels == 1) & (eresnet_preds == 0),
        'fn_vortex':(val_labels == 1) & (eresnet_preds == 2),
    },
    'Vortex': {
        'correct':  (val_labels == 2) & (eresnet_preds == 2),
        'fn_nosub': (val_labels == 2) & (eresnet_preds == 0),
        'fn_sphere':(val_labels == 2) & (eresnet_preds == 1),
    }
}
plot_colors = {'correct': '#2ecc71', 'fn_nosub': '#e74c3c',
               'fn_vortex': '#f39c12', 'fn_sphere': '#f39c12'}
plot_labels = {'correct': 'Correct', 'fn_nosub': '→ No Sub',
               'fn_vortex': '→ Vortex', 'fn_sphere': '→ Sphere'}

for r, cls_name in enumerate(['Sphere', 'Vortex']):
    masks = failure_masks[cls_name]
    for c_idx, (key, mask) in enumerate(masks.items()):
        idxs = np.where(mask)[0]
        # subsample for speed
        idxs = idxs[::max(1, len(idxs)//300)]
        offsets = [centroid_offset(val_residuals[i]) for i in idxs]
        if not offsets:
            axes[r, c_idx].axis('off')
            continue
        dx = [o[0] for o in offsets]
        dy = [o[1] for o in offsets]

        ax = axes[r, c_idx]
        ax.scatter(dx, dy, alpha=0.4, s=15,
                   color=plot_colors[key])
        # Draw ring outline for reference
        theta = np.linspace(0, 2*np.pi, 100)
        for frac in [0.15, 0.65]:
            ax.plot(np.cos(theta)*frac, np.sin(theta)*frac,
                    'k--', lw=0.8, alpha=0.4)
        ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
        ax.set_aspect('equal')
        ax.axhline(0, color='gray', lw=0.5)
        ax.axvline(0, color='gray', lw=0.5)
        ax.set_title(f'{cls_name} — {plot_labels[key]}\n(n={len(dx)})',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Centroid x offset', fontsize=9)
        ax.set_ylabel('Centroid y offset', fontsize=9)
        ax.grid(alpha=0.2)

plt.suptitle(
    'Figure 12f — CAE Residual Centroid Position by Classification Outcome\n'
    'Each point is the spatial position of peak substructure signal for one image\n'
    'Non-uniform distribution indicates position-dependent failure modes',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12f_perturbation_position.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('\nSection 12 complete. Figures saved:')
for fname in ['fig12a_sphere_false_negatives.png',
              'fig12b_sphere_fn_statistics.png',
              'fig12c_sv_confusion_gallery.png',
              'fig12d_elongation_distribution.png',
              'fig12e_confidence_vs_brightness.png',
              'fig12f_perturbation_position.png']:
    path = os.path.join(figures_path, fname)
    exists = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {exists}  {fname}')

###7.3 Sphere Class Analysis on E-DenseNet-C8

### 7.3.1 EqDenseNet-C8 Failure Mode Analysis

EqDenseNet-C8 (macro AUC $0.9973$, validation accuracy $97.35\%$) misclassifies **107 / 2500** Sphere images as No Substructure and correctly classifies **2362 / 2500**. Both figures are better than E-ResNet, consistent with the higher Sphere recall reported earlier.

**Sphere $\rightarrow$ No Substructure false negatives (Fig. 12a–12b).**  
The highest-confidence false negatives have predicted No-Substructure confidence in the range **0.992–0.998**, so the model can still be strongly confident on the images it misses.

| Statistic | TP mean | FN mean | Mann–Whitney $p$ |
|:--|--:|--:|--:|
| Ring mean flux | 0.15830 | 0.14046 | $2.73\times10^{-10}$ |
| Ring flux std | 0.16969 | 0.16253 | $2.23\times10^{-3}$ |
| Ring max intensity | 0.99990 | 1.00000 | $6.34\times10^{-1}$ |
| Ring asymmetry | 0.07859 | 0.08226 | $6.54\times10^{-1}$ |
| Compactness | 0.63773 | 0.66217 | $1.05\times10^{-8}$ |

The main pattern closely matches E-ResNet: false negatives have lower **ring mean flux**, slightly lower **ring flux standard deviation**, and higher **compactness**, while **ring asymmetry** and **maximum intensity** do not separate the groups in this analysis.

**Sphere $\leftrightarrow$ Vortex confusion (Fig. 12c–12d).**  
EqDenseNet-C8 confuses **31** Sphere images as Vortex and **34** Vortex images as Sphere, which is substantially fewer than E-ResNet.

| Group | $n$ | Mean elongation |
|:--|--:|--:|
| Sphere TP | 2362 | 1.205 |
| Sphere $\rightarrow$ Vortex | 31 | 1.200 |
| Vortex TP | 2449 | 1.205 |
| Vortex $\rightarrow$ Sphere | 34 | 1.194 |

As with E-ResNet, residual elongation does not separate the confused groups in a useful way: the means are very similar and the distributions overlap strongly.

**Classification accuracy vs ring brightness (Fig. 12e).**  
The brightness–accuracy pattern is qualitatively similar to E-ResNet, but shifted upward overall. EqDenseNet-C8 performs better across most brightness bins, while the lowest-brightness Sphere regime remains difficult for both architectures.

**Perturbation position vs failure (Fig. 12f).**  
The CAE residual centroids remain tightly clustered near the image center for both correct and incorrect predictions. No clear spatial separation is visible between true positives and the different confusion types under this centroid-based summary.

**Cross-model summary.**

| Metric | E-ResNet | EqDenseNet-C8 |
|:--|--:|--:|
| Validation accuracy | 95.96% | **97.35%** |
| Sphere FN ($\rightarrow$ No Sub) | 129 (5.2%) | **107 (4.3%)** |
| Sphere $\rightarrow$ Vortex | 65 | **31** |
| Vortex $\rightarrow$ Sphere | 54 | **34** |
| FN ring mean flux gap | -10.3% | -11.3% |
| FN ring mean flux $p$ | $6.07\times10^{-10}$ | $2.73\times10^{-10}$ |
| Asymmetry $p$ | 0.899 | 0.654 |
| Clear centroid separation | none visible | none visible |

EqDenseNet-C8 reduces the number of failures relative to E-ResNet, but the *type* of Sphere false negative remains very similar. In both models, the missed Sphere cases are associated with lower ring-level signal and higher compactness, which is consistent with a shared hard subset of the data rather than two completely different failure regimes.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Section 12 — Failure Mode Analysis
#
# Target model: EqDenseNet-C8 (best overall model, macro AUC 0.9974)
#
# Four analyses:
#   12.1  Sphere false negatives — image grid + intensity statistics
#   12.2  Sphere-Vortex confusion — examples + residual morphology
#   12.3  Classification confidence vs ring brightness
#   12.4  Perturbation position vs failure (CAE residual centroid)
#
# All analyses use val_ds, the frozen EqDenseNet-C8, and the frozen CAE.
# No retraining required.
# ═════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy import stats
import os

figures_path = os.path.join(gsoc_drive_path, 'figures')

# ── Shared utilities ──────────────────────────────────────────────────────────

def ring_mask(size=150, inner_frac=0.15, outer_frac=0.65):
    cy, cx = size // 2, size // 2
    Y, X   = np.ogrid[:size, :size]
    dist   = np.sqrt((X - cx)**2 + (Y - cy)**2) / (size // 2)
    return (dist >= inner_frac) & (dist <= outer_frac)

RING = ring_mask(size=150, inner_frac=0.15, outer_frac=0.65)


def get_all_predictions(model, dataset, device, batch_size=256):
    """
    Run model on entire dataset.
    Returns:
        probs   (N, 3)       softmax probabilities
        preds   (N,)         argmax predictions
        labels  (N,)         ground truth
        images  (N, 150, 150) raw images as numpy
    """
    from torch.utils.data import DataLoader
    loader = DataLoader(dataset, batch_size=batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)
    model.eval()

    all_probs, all_preds, all_labels, all_images = [], [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            logits = model(imgs)
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.append(probs.argmax(1))
            all_labels.append(labels.numpy())
            all_images.append(imgs.squeeze(1).cpu().numpy())

    return (np.concatenate(all_probs,  axis=0),
            np.concatenate(all_preds,  axis=0),
            np.concatenate(all_labels, axis=0),
            np.concatenate(all_images, axis=0))


print('Running EqDenseNet-C8 on full val set...')
eqdn_probs, eqdn_preds, val_labels, val_images = \
    get_all_predictions(eq_densenet, val_ds, device)   # ← eq_densenet

val_acc = (eqdn_preds == val_labels).mean()
print(f'Val accuracy: {val_acc*100:.2f}%  (expected ~97.35%)')


def image_stats(img_np):
    """
    Compute per-image morphological statistics from a (150,150) array.

    Returns dict with:
      ring_mean    — mean flux inside ring annulus
      ring_std     — flux std inside ring annulus (proxy for perturbation strength)
      ring_max     — peak flux inside ring annulus
      asymmetry    — |left_half - right_half| / total  (bilateral asymmetry)
      compactness  — fraction of ring flux in top quartile pixels
    """
    ring_pixels  = img_np[RING]
    ring_mean    = ring_pixels.mean()
    ring_std     = ring_pixels.std()
    ring_max     = ring_pixels.max()

    half = img_np.shape[1] // 2
    left  = img_np[:, :half][RING[:, :half]].sum()
    right = img_np[:, half:][RING[:, half:]].sum()
    total = left + right + 1e-9
    asymmetry = abs(left - right) / total

    q75 = np.percentile(ring_pixels, 75)
    compactness = ring_pixels[ring_pixels >= q75].sum() / (ring_pixels.sum() + 1e-9)

    return dict(ring_mean=ring_mean, ring_std=ring_std, ring_max=ring_max,
                asymmetry=asymmetry, compactness=compactness)


print('Computing morphological statistics for all val images...')
all_stats = [image_stats(val_images[i]) for i in range(len(val_images))]
stat_keys = list(all_stats[0].keys())
stats_arr  = {k: np.array([s[k] for s in all_stats]) for k in stat_keys}
print('Done.')


# ── 12.1  Sphere false negatives ──────────────────────────────────────────────
print('\n═══ 12.1 Sphere False Negatives ═══')

sphere_fn_mask = (val_labels == 1) & (eqdn_preds == 0)   # ← eqdn_preds
sphere_tp_mask = (val_labels == 1) & (eqdn_preds == 1)

sphere_fn_idx  = np.where(sphere_fn_mask)[0]
sphere_tp_idx  = np.where(sphere_tp_mask)[0]

print(f'Sphere true positives:             {sphere_tp_mask.sum()}')
print(f'Sphere false negatives (→ No Sub): {sphere_fn_mask.sum()}')

print('\nMorphological statistics — TP vs FN:')
print(f'{"Statistic":20s}  {"TP mean":>10s}  {"FN mean":>10s}  {"p-value":>10s}')
print('─' * 56)

for k in stat_keys:
    tp_vals = stats_arr[k][sphere_tp_idx]
    fn_vals = stats_arr[k][sphere_fn_idx]
    t, p    = stats.mannwhitneyu(tp_vals, fn_vals, alternative='two-sided')
    print(f'{k:20s}  {tp_vals.mean():>10.5f}  '
          f'{fn_vals.mean():>10.5f}  {p:>10.4e}')

# ── Figure 12a — FN image grid ────────────────────────────────────────────────
fn_wrong_conf = eqdn_probs[sphere_fn_idx, 0]             # ← eqdn_probs
sort_idx      = np.argsort(fn_wrong_conf)[::-1]
fn_sorted     = sphere_fn_idx[sort_idx]

n_show = min(12, len(fn_sorted))
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
axes = axes.flatten()

for i in range(n_show):
    idx  = fn_sorted[i]
    img  = val_images[idx]
    conf = eqdn_probs[idx, 0]                            # ← eqdn_probs
    axes[i].imshow(img, cmap='inferno', vmin=0, vmax=img.max())
    axes[i].set_title(
        f'Conf(NoSub)={conf:.3f}\n'
        f'ring_mean={stats_arr["ring_mean"][idx]:.4f}',
        fontsize=8
    )
    axes[i].axis('off')

plt.suptitle(
    'Figure 12a — Sphere False Negatives (predicted as No Substructure)\n'
    'Sorted by EqDenseNet-C8 confidence in wrong class (highest first)\n'
    'These are the Sphere images the best model cannot distinguish from smooth lenses',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12a_sphere_false_negatives.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 12b — Ring brightness distribution: TP vs FN ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_stats = ['ring_mean', 'ring_std', 'asymmetry']
xlabels    = ['Ring mean flux', 'Ring flux std\n(perturbation strength proxy)',
              'Ring asymmetry']

for i, (k, xl) in enumerate(zip(plot_stats, xlabels)):
    tp_vals = stats_arr[k][sphere_tp_idx]
    fn_vals = stats_arr[k][sphere_fn_idx]
    _, p    = stats.mannwhitneyu(tp_vals, fn_vals, alternative='two-sided')

    axes[i].hist(tp_vals, bins=50, alpha=0.6, density=True,
                 color='#2ecc71', label=f'True Positive (n={len(tp_vals)})')
    axes[i].hist(fn_vals, bins=50, alpha=0.6, density=True,
                 color='#e74c3c', label=f'False Negative (n={len(fn_vals)})')
    axes[i].axvline(tp_vals.mean(), color='#27ae60', lw=2, linestyle='--')
    axes[i].axvline(fn_vals.mean(), color='#c0392b', lw=2, linestyle='--')
    axes[i].set_xlabel(xl, fontsize=10)
    axes[i].set_ylabel('Density', fontsize=10)
    axes[i].set_title(f'p = {p:.2e}', fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].grid(alpha=0.3)

plt.suptitle(
    'Figure 12b — Sphere TP vs FN: Morphological Statistics\n'
    'EqDenseNet-C8 false negatives have systematically lower ring flux '
    'and perturbation strength',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12b_sphere_fn_statistics.png'),
            dpi=150, bbox_inches='tight')
plt.show()


# ── 12.2  Sphere ↔ Vortex confusion ──────────────────────────────────────────
print('\n═══ 12.2 Sphere ↔ Vortex Confusion ═══')

sv_mask = (val_labels == 1) & (eqdn_preds == 2)   # ← eqdn_preds
vs_mask = (val_labels == 2) & (eqdn_preds == 1)

sv_idx = np.where(sv_mask)[0]
vs_idx = np.where(vs_mask)[0]

print(f'Sphere predicted as Vortex: {sv_mask.sum()}')
print(f'Vortex predicted as Sphere: {vs_mask.sum()}')

val_res_data  = np.load(res_val_path)
val_residuals = val_res_data['residuals']   # (7500, 150, 150)

def residual_morphology(res_np):
    ring_res  = np.abs(res_np) * RING
    total     = ring_res.sum() + 1e-9
    Y, X  = np.mgrid[:150, :150]
    cy    = (ring_res * Y).sum() / total
    cx    = (ring_res * X).sum() / total
    dy2   = ((ring_res * (Y - cy)**2).sum() / total)
    dx2   = ((ring_res * (X - cx)**2).sum() / total)
    dxy   = ((ring_res * (Y - cy) * (X - cx)).sum() / total)
    trace  = dy2 + dx2
    det    = dy2 * dx2 - dxy**2
    disc   = max(0, (trace/2)**2 - det)
    lam1   = trace/2 + np.sqrt(disc)
    lam2   = trace/2 - np.sqrt(disc) + 1e-9
    elongation = np.sqrt(lam1 / lam2)
    return dict(
        res_std    = res_np[RING].std(),
        centroid_y = cy,
        centroid_x = cx,
        elongation = elongation
    )

sphere_tp_res = [residual_morphology(val_residuals[i]) for i in sphere_tp_idx]
sv_res        = [residual_morphology(val_residuals[i]) for i in sv_idx]
vortex_tp_idx = np.where((val_labels == 2) & (eqdn_preds == 2))[0]   # ← eqdn_preds
vs_res        = [residual_morphology(val_residuals[i]) for i in vs_idx]
vortex_tp_res = [residual_morphology(val_residuals[i]) for i in vortex_tp_idx]

def mean_morph(morph_list, key):
    return np.mean([m[key] for m in morph_list])

print('\nResidual elongation (1.0 = circular, higher = elongated):')
print(f'  Sphere TP:         {mean_morph(sphere_tp_res, "elongation"):.3f}')
print(f'  Sphere → Vortex:   {mean_morph(sv_res,        "elongation"):.3f}')
print(f'  Vortex TP:         {mean_morph(vortex_tp_res, "elongation"):.3f}')
print(f'  Vortex → Sphere:   {mean_morph(vs_res,        "elongation"):.3f}')

# ── Figure 12c — Confused image gallery ───────────────────────────────────────
fig, axes = plt.subplots(3, 8, figsize=(20, 9))
row_labels  = ['True Sphere\n(correct)', 'Sphere → Vortex\n(confused)',
               'Vortex → Sphere\n(confused)']
row_indices = [sphere_tp_idx[:8], sv_idx[:8], vs_idx[:8]]

for r, (label, idxs) in enumerate(zip(row_labels, row_indices)):
    for c, idx in enumerate(idxs[:8]):
        axes[r, c].imshow(val_images[idx], cmap='inferno')
        axes[r, c].axis('off')
        if c == 0:
            axes[r, c].set_ylabel(label, fontsize=9, fontweight='bold',
                                   rotation=90, labelpad=50)

plt.suptitle(
    'Figure 12c — Sphere ↔ Vortex Confusion Gallery  [EqDenseNet-C8]\n'
    'Row 1: Correctly classified Sphere  |  '
    'Row 2: Sphere misclassified as Vortex  |  '
    'Row 3: Vortex misclassified as Sphere',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12c_sv_confusion_gallery.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 12d — Elongation distributions ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

groups = [
    (sphere_tp_res,  '#2ecc71', 'Sphere TP'),
    (sv_res,         '#e74c3c', 'Sphere → Vortex'),
    (vortex_tp_res,  '#3498db', 'Vortex TP'),
    (vs_res,         '#f39c12', 'Vortex → Sphere'),
]

for morph_list, color, label in groups:
    vals = [m['elongation'] for m in morph_list]
    ax.hist(np.clip(vals, 1, 10), bins=40, alpha=0.6,
            density=True, color=color, label=f'{label} (n={len(vals)})')

ax.set_xlabel('Residual elongation (major/minor axis ratio)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title(
    'Figure 12d — Residual Morphology: Elongation by Confusion Type  '
    '[EqDenseNet-C8]\n'
    'Sphere-Vortex confusion correlates with intermediate elongation values',
    fontsize=11, fontweight='bold'
)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12d_elongation_distribution.png'),
            dpi=150, bbox_inches='tight')
plt.show()


# ── 12.3  Confidence vs ring brightness ───────────────────────────────────────
print('\n═══ 12.3 Confidence vs Ring Brightness ═══')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
class_colors = ['#3498db', '#e74c3c', '#2ecc71']

for c in range(3):
    mask         = val_labels == c
    ring_means   = stats_arr['ring_mean'][mask]
    conf_correct = eqdn_probs[mask, c]             # ← eqdn_probs
    correct_bool = (eqdn_preds[mask] == c)         # ← eqdn_preds

    bins    = np.percentile(ring_means, np.linspace(0, 100, 11))
    bins    = np.unique(bins)
    bin_idx = np.digitize(ring_means, bins) - 1
    bin_idx = np.clip(bin_idx, 0, len(bins) - 2)

    bin_centers = [(bins[i] + bins[i+1]) / 2 for i in range(len(bins)-1)]
    bin_acc     = [correct_bool[bin_idx == i].mean()
                   for i in range(len(bins)-1)]
    bin_conf    = [conf_correct[bin_idx == i].mean()
                   for i in range(len(bins)-1)]

    ax  = axes[c]
    ax2 = ax.twinx()

    ax.plot(bin_centers, bin_acc,  'o-', color=class_colors[c],
            lw=2, markersize=6, label='Accuracy')
    ax2.plot(bin_centers, bin_conf, 's--', color=class_colors[c],
             lw=1.5, markersize=5, alpha=0.7, label='Mean confidence')

    ax.set_xlabel('Ring mean flux (brightness)', fontsize=10)
    ax.set_ylabel('Classification accuracy',     fontsize=10, color='black')
    ax2.set_ylabel('Mean confidence',            fontsize=10, color='gray')
    ax.set_title(CLASS_NAMES[c], fontsize=11, fontweight='bold')
    ax.set_ylim(0.5, 1.05)
    ax.grid(alpha=0.3)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='lower right')

plt.suptitle(
    'Figure 12e — Classification Accuracy and Confidence vs Ring Brightness  '
    '[EqDenseNet-C8]\n'
    'Low-brightness images are harder to classify — '
    'substructure signal falls below noise floor',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12e_confidence_vs_brightness.png'),
            dpi=150, bbox_inches='tight')
plt.show()


# ── 12.4  Perturbation position vs failure ────────────────────────────────────
print('\n═══ 12.4 Perturbation Position vs Failure ═══')

CX0, CY0 = 75, 75

def centroid_offset(res_np):
    ring_res = np.abs(res_np) * RING
    total    = ring_res.sum() + 1e-9
    Y, X     = np.mgrid[:150, :150]
    cx       = (ring_res * X).sum() / total
    cy       = (ring_res * Y).sum() / total
    return (cx - CX0) / 75.0, (cy - CY0) / 75.0


fig, axes = plt.subplots(2, 3, figsize=(18, 12))

failure_masks = {
    'Sphere': {
        'correct':   (val_labels == 1) & (eqdn_preds == 1),   # ← eqdn_preds
        'fn_nosub':  (val_labels == 1) & (eqdn_preds == 0),
        'fn_vortex': (val_labels == 1) & (eqdn_preds == 2),
    },
    'Vortex': {
        'correct':   (val_labels == 2) & (eqdn_preds == 2),
        'fn_nosub':  (val_labels == 2) & (eqdn_preds == 0),
        'fn_sphere': (val_labels == 2) & (eqdn_preds == 1),
    }
}
plot_colors = {'correct': '#2ecc71', 'fn_nosub': '#e74c3c',
               'fn_vortex': '#f39c12', 'fn_sphere': '#f39c12'}
plot_labels = {'correct': 'Correct', 'fn_nosub': '→ No Sub',
               'fn_vortex': '→ Vortex', 'fn_sphere': '→ Sphere'}

for r, cls_name in enumerate(['Sphere', 'Vortex']):
    masks = failure_masks[cls_name]
    for c_idx, (key, mask) in enumerate(masks.items()):
        idxs = np.where(mask)[0]
        idxs = idxs[::max(1, len(idxs)//300)]
        offsets = [centroid_offset(val_residuals[i]) for i in idxs]
        if not offsets:
            axes[r, c_idx].axis('off')
            continue
        dx = [o[0] for o in offsets]
        dy = [o[1] for o in offsets]

        ax = axes[r, c_idx]
        ax.scatter(dx, dy, alpha=0.4, s=15, color=plot_colors[key])
        theta = np.linspace(0, 2*np.pi, 100)
        for frac in [0.15, 0.65]:
            ax.plot(np.cos(theta)*frac, np.sin(theta)*frac,
                    'k--', lw=0.8, alpha=0.4)
        ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
        ax.set_aspect('equal')
        ax.axhline(0, color='gray', lw=0.5)
        ax.axvline(0, color='gray', lw=0.5)
        ax.set_title(f'{cls_name} — {plot_labels[key]}\n(n={len(dx)})',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Centroid x offset', fontsize=9)
        ax.set_ylabel('Centroid y offset', fontsize=9)
        ax.grid(alpha=0.2)

plt.suptitle(
    'Figure 12f — CAE Residual Centroid Position by Classification Outcome  '
    '[EqDenseNet-C8]\n'
    'Each point is the spatial position of peak substructure signal for one image\n'
    'Non-uniform distribution indicates position-dependent failure modes',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(figures_path, 'fig12f_perturbation_position.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('\nSection 12 complete. Figures saved:')
for fname in ['fig12a_sphere_false_negatives.png',
              'fig12b_sphere_fn_statistics.png',
              'fig12c_sv_confusion_gallery.png',
              'fig12d_elongation_distribution.png',
              'fig12e_confidence_vs_brightness.png',
              'fig12f_perturbation_position.png']:
    path   = os.path.join(figures_path, fname)
    exists = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'  {exists}  {fname}')

## <a id="8-residual-image-approach"></a>8. Residual Image Approach (Experimental)

### 8.1 Motivation

A residual-image pipeline is attractive because it aims to isolate the substructure signal directly:
$$
\text{residual} = \text{observed image} - \text{smooth-lens reconstruction}.
$$
If the smooth macro lens can be reconstructed well, then the residual should retain the localized perturbation while removing most of the common ring morphology.

This idea looked plausible for DeepLenseSim Model I because the simulation setup is highly constrained. Inspection of the generator code showed that most macro parameters are fixed, while only a small subset varies across images. That suggested a simple forward-model fit might be feasible using `lenstronomy` with a small number of free parameters.

The goal of this section is therefore modest: test whether a lightweight analytical fit can produce residuals that are cleaner and more class-informative than the original images.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Section 11 — Residual Image Approach
# Ground-truth clean lens reconstruction using DeepLenseSim Model I parameters
# ═════════════════════════════════════════════════════════════════════════════

# ── Step 1: Install and verify ────────────────────────────────────────────────


import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.optimize import minimize

from astropy.cosmology import FlatLambdaCDM
from astropy import units as u
from astropy.constants import G, M_sun
from astropy.constants import c as c_light

from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.Data.psf import PSF
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.Util import util

print(f'lenstronomy imported successfully')
print(f'numpy version: {np.__version__}')


# ── Step 2: Fixed simulation parameters (read directly from lens.py) ─────────

# Source shape — fixed in make_source_light(), only center varies
SOURCE_KWARGS_TEMPLATE = {
    'amp':      20,
    'R_sersic': 0.25,
    'n_sersic': 1,
    'e1':       -0.1,
    'e2':        0.1
}

# Imaging parameters — fixed in simple_sim()
SIM_NUMPIX   = 150
SIM_DELTAPIX = 0.05    # arcsec/pixel
SIM_FWHM     = 0.087   # arcsec
SIM_BKG_RMS  = 1e-2
SIM_EXP_MED  = 10**3.25  # median of uniform(3, 3.5)

# Lens geometry — fixed in make_single_halo()
SIM_E1       = 0.1
SIM_E2       = 0.0
SIM_GAMMA1   = 0.05
SIM_GAMMA2   = 0.0
SIM_MASS     = 1e12     # solar masses (nominal)
SIM_Z_HALO   = 0.5
SIM_Z_GAL    = 1.0

print('Simulation parameters loaded.')


# ── Step 3: Utility functions ─────────────────────────────────────────────────

def ring_mask(size=150, inner_frac=0.20, outer_frac=0.60):
    """Boolean annular mask covering the Einstein ring region."""
    cy, cx = size // 2, size // 2
    Y, X   = np.ogrid[:size, :size]
    dist   = np.sqrt((X - cx)**2 + (Y - cy)**2) / (size // 2)
    return (dist >= inner_frac) & (dist <= outer_frac)


def compute_einstein_radius(mass, z_halo=0.5, z_gal=1.0,
                             H0=70, Om0=0.3, Ob0=0.05):
    cosmo = FlatLambdaCDM(H0=H0, Om0=Om0, Ob0=Ob0)

    DL  = cosmo.luminosity_distance(z_halo).to(u.m).value
    DS  = cosmo.luminosity_distance(z_gal).to(u.m).value
    DLS = DS - DL

    G_si  = G.to(u.m**3 / (u.kg * u.s**2)).value
    c_si  = c_light.to(u.m / u.s).value
    M_kg  = (mass * M_sun).to(u.kg).value

    theta = np.sqrt(4 * G_si * M_kg / c_si**2 * DLS / (DL * DS))
    return float(theta * 206265)

ER_NOMINAL = compute_einstein_radius(SIM_MASS)
print(f'Nominal Einstein radius: {ER_NOMINAL:.6f} arcsec')


def generate_clean_lens(cx, cy, mass=SIM_MASS, exp_time=SIM_EXP_MED):
    """
    Generate a noiseless clean lens image using exact DeepLenseSim
    Model I parameters.

    Parameters
    ----------
    cx, cy   : source centre position in arcsec, range [-0.35, 0.35]
    mass     : halo mass in solar masses (controls Einstein radius)
    exp_time : exposure time (controls flux normalisation)

    Returns
    -------
    image : (150, 150) float array — noiseless model image
    """
    ER = compute_einstein_radius(mass)

    lens_kwargs = [
        {'theta_E': ER, 'e1': SIM_E1, 'e2': SIM_E2,
         'center_x': 0.0, 'center_y': 0.0},
        {'gamma1': SIM_GAMMA1, 'gamma2': SIM_GAMMA2}
    ]

    _, _, ra0, dec0, _, _, Mpix2coord, _ = util.make_grid_with_coordtransform(
        numPix=SIM_NUMPIX, deltapix=SIM_DELTAPIX,
        center_ra=0, center_dec=0,
        subgrid_res=1, inverse=False
    )

    kwargs_data = {
        'background_rms':      SIM_BKG_RMS,
        'exposure_time':       exp_time,
        'ra_at_xy_0':          ra0,
        'dec_at_xy_0':         dec0,
        'transform_pix2angle': Mpix2coord,
        'image_data':          np.zeros((SIM_NUMPIX, SIM_NUMPIX))
    }

    data_class   = ImageData(**kwargs_data)
    psf_class    = PSF(psf_type='GAUSSIAN', fwhm=SIM_FWHM,
                       pixel_size=SIM_DELTAPIX, truncation=3)
    lens_model   = LensModel(['SIE', 'SHEAR'])
    source_model = LightModel(['SERSIC_ELLIPSE'])

    image_mdl = ImageModel(
        data_class, psf_class,
        lens_model_class=lens_model,
        source_model_class=source_model,
        kwargs_numerics={'supersampling_factor': 1,
                         'supersampling_convolution': False}
    )

    kwargs_source = [{**SOURCE_KWARGS_TEMPLATE,
                      'center_x': cx, 'center_y': cy}]

    return image_mdl.image(lens_kwargs, kwargs_source,
                           kwargs_lens_light=None, kwargs_ps=None)


def fit_source_position(obs_image):
    """
    Fit (cx, cy, log10_mass) to an observed lensing image.

    Strategy
    --------
    - Optimise over the Einstein ring annulus only (inner=0.20, outer=0.60)
      so substructure signal outside the ring does not corrupt the fit.
    - At each optimizer step, solve analytically for the optimal amplitude
      alpha = <obs, model> / <model, model> to remove scale ambiguity.
    - Allow log10(mass) to vary ±0.3 dex around 1e12 to absorb any
      Einstein radius mismatch between our computation and the dataset.

    Returns
    -------
    cx_best, cy_best : fitted source position (arcsec)
    mass_best        : fitted halo mass (solar masses)
    clean_scaled     : fitted clean lens, normalised to [0, 1]
    residual         : obs_norm - clean_scaled
    loss             : final objective value
    """
    ring         = ring_mask(size=150, inner_frac=0.20, outer_frac=0.60)
    obs_norm     = obs_image / (obs_image.max() + 1e-8)
    obs_flat     = obs_norm[ring].flatten()

    def objective(params):
        cx, cy, log_mass = params
        cx   = np.clip(cx, -0.35, 0.35)
        cy   = np.clip(cy, -0.35, 0.35)
        mass = np.clip(10**log_mass, 10**11.7, 10**12.3)

        clean      = generate_clean_lens(cx, cy, mass=mass)
        clean_flat = clean[ring].flatten()

        # Analytical optimal scale factor — removes amplitude ambiguity
        denom = np.dot(clean_flat, clean_flat) + 1e-8
        alpha = np.dot(obs_flat, clean_flat) / denom

        diff  = obs_flat - alpha * clean_flat
        return float(np.sum(diff**2))

    result = minimize(
        objective,
        x0=[0.0, 0.0, 12.0],
        method='Nelder-Mead',
        options={
            'xatol':   1e-4,
            'fatol':   1e-8,
            'maxiter': 2000
        }
    )

    cx_best, cy_best, log_mass_best = result.x
    cx_best   = np.clip(cx_best, -0.35, 0.35)
    cy_best   = np.clip(cy_best, -0.35, 0.35)
    mass_best = np.clip(10**log_mass_best, 10**11.7, 10**12.3)

    # Reconstruct final clean image with optimal amplitude
    clean      = generate_clean_lens(cx_best, cy_best, mass=mass_best)
    ring_mask_ = ring
    clean_flat = clean[ring_mask_].flatten()
    obs_flat_  = obs_norm[ring_mask_].flatten()
    alpha      = np.dot(obs_flat_, clean_flat) / (
                     np.dot(clean_flat, clean_flat) + 1e-8)

    clean_scaled      = alpha * clean
    clean_scaled_norm = clean_scaled / (clean_scaled.max() + 1e-8)
    residual          = obs_norm - clean_scaled_norm

    return cx_best, cy_best, mass_best, clean_scaled_norm, residual, result.fun


print('All Section 11 functions defined.')


# ═════════════════════════════════════════════════════════════════════════════
# VALIDATION — 10 images per class
# Check: No Substructure residuals should be near-zero noise
#        Sphere / Vortex residuals should show compact structure
# ═════════════════════════════════════════════════════════════════════════════

n_validate  = 10
val_results = {c: [] for c in range(3)}

fig, axes = plt.subplots(3, 5, figsize=(22, 14))
col_titles = [
    'Original',
    'Fitted Clean Lens',
    'Residual\n(full range)',
    'Residual\n(clipped ±0.1)',
    'Fit parameters'
]

print('Fitting source positions on validation subset...')

for c in range(3):
    count = 0
    for idx in range(len(val_ds)):
        if count >= n_validate:
            break
        img, label = val_ds[idx]
        if label != c:
            continue

        obs = img.squeeze().numpy().astype(np.float32)
        cx, cy, mass_fit, clean, residual, loss = fit_source_position(obs)

        val_results[c].append({
            'idx':      idx,
            'cx':       cx,
            'cy':       cy,
            'mass':     mass_fit,
            'loss':     loss,
            'residual': residual
        })
        count += 1

        # Plot first image per class only
        if count == 1:
            obs_n  = obs / (obs.max() + 1e-8)
            res_cl = np.clip(residual, -0.1, 0.1)
            vmax_r = np.abs(residual).max()

            axes[c, 0].imshow(obs_n,    cmap='inferno')
            axes[c, 1].imshow(clean,    cmap='inferno')
            axes[c, 2].imshow(residual, cmap='RdBu_r',
                              vmin=-vmax_r, vmax=vmax_r)
            axes[c, 3].imshow(res_cl,   cmap='RdBu_r',
                              vmin=-0.1,   vmax=0.1)

            axes[c, 4].text(
                0.5, 0.5,
                f'{CLASS_NAMES[c]}\n\n'
                f'cx    = {cx:.4f} arcsec\n'
                f'cy    = {cy:.4f} arcsec\n'
                f'mass  = {mass_fit:.3e} M☉\n'
                f'θ_E   = {compute_einstein_radius(mass_fit):.4f} arcsec\n\n'
                f'loss  = {loss:.4f}',
                ha='center', va='center',
                transform=axes[c, 4].transAxes,
                fontsize=9, family='monospace',
                bbox=dict(boxstyle='round', facecolor='#f8f8f8',
                          edgecolor='gray', alpha=0.8)
            )
            axes[c, 4].axis('off')
            axes[c, 0].set_ylabel(CLASS_NAMES[c], fontsize=12,
                                   fontweight='bold', rotation=90,
                                   labelpad=10)
            for j in range(4):
                axes[c, j].set_xticks([])
                axes[c, j].set_yticks([])

    mean_loss = np.mean([r['loss'] for r in val_results[c]])
    mean_ER   = np.mean([compute_einstein_radius(r['mass'])
                         for r in val_results[c]])
    print(f'  {CLASS_NAMES[c]:20s}  '
          f'mean loss = {mean_loss:8.4f}  '
          f'mean θ_E = {mean_ER:.4f} arcsec')

for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=10, fontweight='bold', pad=8)

plt.suptitle(
    'Section 11 — Residual Image Approach: Validation (n=10 per class)\n'
    'Residual = observed − fitted clean lens\n'
    'No Substructure residuals should be noise-like; '
    'Sphere/Vortex should show compact structure',
    fontsize=11, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures', 'fig11_residual_validation.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

# ── Residual statistics ───────────────────────────────────────────────────────
print('\n── Residual statistics (mean ± std over ring annulus) ──')
print(f'{"Class":20s}  {"Mean residual":>14s}  {"Std residual":>12s}  '
      f'{"Mean |residual|":>16s}')
print('─' * 70)

ring = ring_mask()
for c in range(3):
    all_res = np.stack([r['residual'] for r in val_results[c]])
    in_ring = all_res[:, ring]
    print(f'{CLASS_NAMES[c]:20s}  '
          f'{in_ring.mean():>14.5f}  '
          f'{in_ring.std():>12.5f}  '
          f'{np.abs(in_ring).mean():>16.5f}')

### <a id="82-analytical-fitting-abandoned"></a>8.2 Analytical Fitting (Abandoned)

The fitting procedure optimized $(c_x, c_y, \log_{10} M)$ using Nelder–Mead on an $L_2$ loss restricted to the Einstein-ring annulus, with an analytic amplitude rescaling at each step. In practice, this did **not** yield a useful residual representation for substructure analysis.

Three observations support that conclusion.

**1. The fitted Einstein radii are inconsistent across classes.**  
The nominal Einstein radius implied by the fixed simulation mass is:
$$
\theta_E^{\text{nominal}} = 1.281 \text{ arcsec}.
$$

The mean fitted values were:

| Class | Mean fitted $\theta_E$ | Nominal $\theta_E$ |
|---|---:|---:|
| No Substructure | 1.3863 | 1.2815 |
| Sphere | 1.1451 | 1.2815 |
| Vortex | 1.1946 | 1.2815 |

A parameter that should be common across the dataset varies substantially after fitting, which indicates that the optimizer is not recovering a stable class-independent macro solution.

**2. The residual statistics do not separate the classes in a useful way.**

| Class | Mean residual | Std residual | Mean $|\text{residual}|$ |
|---|---:|---:|---:|
| No Substructure | 0.01611 | 0.18286 | 0.12305 |
| Sphere | 0.04527 | 0.15545 | 0.10483 |
| Vortex | 0.04161 | 0.14899 | 0.10142 |

These residual summaries do not show the expected pattern of near-clean residuals for No Substructure and stronger localized residual structure for Sphere/Vortex. At minimum, they do not provide a clean basis for downstream class separation.

**3. The fit quality remains poor.**  
The mean losses remain large:
- No Substructure: 134.28
- Sphere: 117.71
- Vortex: 114.07

This suggests that the fitted smooth model is not reproducing the observed images closely enough for the residuals to be interpreted as isolated substructure signals.

A likely explanation is that the optimization problem is both **numerically fragile** and **physically degenerate**. The overflow warning in the mass parameterization already indicates instability, and the fitted macro parameters appear able to shift in ways that partially absorb image structure rather than isolating it. That explanation is plausible, but it should be treated as a hypothesis rather than a proven mechanism from this experiment alone.

For the purposes of this notebook, the practical conclusion is simpler: the analytical fitting route did not produce clean, interpretable residuals, so it was not pursued further. The analysis therefore pivots to a non-parametric residual approach based on a convolutional autoencoder in Section 8.3.

### <a id="83-cae-architecture--training"></a>8.3 CAE Architecture & Training

The analytical fitting approach required solving a difficult inverse problem separately for each image. The CAE approach replaces that with a purely data-driven reconstruction model.

The autoencoder is trained **only on No Substructure images**. The intended idea is simple: learn to reconstruct smooth lenses well, then use the residual
$$
\text{residual} = \text{input} - \text{reconstruction}
$$
as a proxy for structure that is not well explained by the smooth-lens training distribution.

This does **not** guarantee that substructure is perfectly isolated. In practice, the CAE can still absorb part of the perturbation into a smooth reconstruction. The method is therefore best understood as a heuristic residual extractor rather than a clean physical decomposition.

A practical advantage is that inference is fast and requires no per-image optimization. The main design choice is the loss: **full-image MSE** is used so that the network must reconstruct both the bright ring and the dark background. Ring-only loss produced unstable residual maps due to poor background behavior.

The validation question is: do residuals from Sphere and Vortex remain slightly larger or more structured than residuals from No Substructure?

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Section 11 — Residual Image Approach: Convolutional Autoencoder
#
# Strategy:
#   Train a CAE exclusively on No Substructure images.
#   The bottleneck forces it to learn only the smooth lens manifold.
#   At inference, feed Sphere/Vortex images — the CAE reconstructs
#   the smooth component only. Residual = input − reconstruction
#   contains only the substructure signal.
#
# Reference: Alexander et al. (2019) arXiv:1909.07346, Figures 3–4
# ═════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os

# ── Utility ───────────────────────────────────────────────────────────────────

def ring_mask(size=150, inner_frac=0.15, outer_frac=0.65):
    """Boolean annular mask covering the Einstein ring region."""
    cy, cx = size // 2, size // 2
    Y, X   = np.ogrid[:size, :size]
    dist   = np.sqrt((X - cx)**2 + (Y - cy)**2) / (size // 2)
    return (dist >= inner_frac) & (dist <= outer_frac)


# ── 11.1  No-substructure only dataset ───────────────────────────────────────

class NoSubDataset(torch.utils.data.Dataset):
    """
    Wraps the existing LensingDataset and returns only
    class-0 (No Substructure) images for autoencoder training.
    Input = target — no label needed.
    """
    def __init__(self, base_ds):
        self.samples = [img for img, label in base_ds if label == 0]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


print('Building No Substructure dataset...')
nosub_train_ds = NoSubDataset(train_ds)
nosub_val_ds   = NoSubDataset(val_ds)
print(f'  Train: {len(nosub_train_ds)} images')
print(f'  Val:   {len(nosub_val_ds)}   images')

nosub_train_loader = DataLoader(
    nosub_train_ds, batch_size=64, shuffle=True,
    num_workers=2, pin_memory=True
)
nosub_val_loader = DataLoader(
    nosub_val_ds, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True
)


# ── 11.2  Architecture ────────────────────────────────────────────────────────

class LensAutoencoder(nn.Module):
    """
    Symmetric encoder-decoder CAE for 150×150 single-channel images.

    Encoder: 4 strided conv blocks → FC bottleneck (latent_dim=128)
    Decoder: FC → 4 transposed conv blocks → 144×144 → bilinear 150×150

    Bottleneck dimension controls what the decoder can reconstruct.
    Trained only on No Substructure → cannot reconstruct substructure.
    Residual = input − reconstruction isolates the perturbation signal.

    Full-image MSE loss (not ring-only) forces the decoder to learn
    the correct dark background AND bright ring simultaneously.
    Ring-only loss causes the sigmoid output to push background pixels
    toward 0.5, producing invalid residual maps.
    """
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim

        # ── Encoder: 1×150×150 → 256×9×9 → latent_dim ───────────────
        self.encoder = nn.Sequential(
            # 1×150×150 → 32×75×75
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            # 32×75×75 → 64×37×37
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            # 64×37×37 → 128×18×18
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # 128×18×18 → 256×9×9
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.enc_flat_dim = 256 * 9 * 9          # 20736
        self.fc_enc = nn.Linear(self.enc_flat_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, self.enc_flat_dim)

        # ── Decoder: 256×9×9 → 1×144×144 → bilinear 150×150 ─────────
        self.decoder = nn.Sequential(
            # 256×9×9 → 128×18×18
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # 128×18×18 → 64×36×36
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            # 64×36×36 → 32×72×72
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            # 32×72×72 → 1×144×144
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
        )
        # Bilinear upsample 144→150 avoids checkerboard artifacts

    def encode(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        return self.fc_enc(h)

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(h.size(0), 256, 9, 9)
        h = self.decoder(h)
        h = F.interpolate(h, size=(150, 150),
                          mode='bilinear', align_corners=False)
        return torch.sigmoid(h)

    def forward(self, x):
        return self.decode(self.encode(x))


# Verify spatial dimensions
_test = torch.zeros(2, 1, 150, 150)
_cae  = LensAutoencoder(latent_dim=128)
_out  = _cae(_test)
assert _out.shape == (2, 1, 150, 150), \
    f'Unexpected output shape: {_out.shape}'
n_params = sum(p.numel() for p in _cae.parameters()
               if p.requires_grad)
print(f'CAE output shape: {_out.shape}   ✓')
print(f'CAE parameters:   {n_params/1e6:.2f}M')
del _test, _cae, _out


# ── 11.3  Training ────────────────────────────────────────────────────────────

def train_autoencoder(model, train_loader, val_loader,
                      n_epochs=60, lr=1e-3,
                      save_path=None, history_path=None):
    """
    Train CAE with FULL-IMAGE MSE loss.

    Full-image loss is critical: ring-only loss causes the sigmoid
    decoder output to push background pixels toward 0.5 (the sigmoid
    midpoint), since those pixels receive no gradient. This produces
    a large constant offset in every residual map, completely masking
    the substructure signal. Full-image loss forces the decoder to
    correctly reconstruct zero background AND the bright ring.
    """
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=1e-5
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-5
    )

    history   = {'train_loss': [], 'val_loss': []}
    best_val  = float('inf')

    for epoch in range(1, n_epochs + 1):

        # ── train ──────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            batch = batch.to(device)
            recon = model(batch)
            loss  = F.mse_loss(recon, batch)   # full-image MSE
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch.size(0)
        train_loss /= len(train_loader.dataset)

        # ── validate ───────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                batch    = batch.to(device)
                recon    = model(batch)
                loss     = F.mse_loss(recon, batch)
                val_loss += loss.item() * batch.size(0)
        val_loss /= len(val_loader.dataset)

        scheduler.step()
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            if save_path:
                torch.save(model.state_dict(), save_path)

        if epoch % 10 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{n_epochs}  '
                  f'train={train_loss:.6f}  '
                  f'val={val_loss:.6f}  '
                  f'best={best_val:.6f}')

    if history_path:
        np.save(history_path, history)

    return history


# ── Load or train ─────────────────────────────────────────────────────────────
cae_save_path    = os.path.join(gsoc_drive_path, 'cae_best.pth')
cae_history_path = os.path.join(gsoc_drive_path, 'cae_history.npy')

# Delete old weights trained with wrong ring-only loss
if os.path.exists(cae_save_path):
    os.remove(cae_save_path)
    print('Old weights deleted — retraining with full-image loss.')

cae = LensAutoencoder(latent_dim=128).to(device)

print('Training CAE from scratch...')
cae_history = train_autoencoder(
    cae,
    nosub_train_loader,
    nosub_val_loader,
    n_epochs=60,
    lr=1e-3,
    save_path=cae_save_path,
    history_path=cae_history_path
)
print(f'\nTraining complete. Best weights saved to {cae_save_path}')

# Load best checkpoint for evaluation
cae.load_state_dict(torch.load(cae_save_path, map_location=device))
cae.eval()
print('CAE ready.')


# ── 11.4  Training curve ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
epochs  = range(1, len(cae_history['train_loss']) + 1)
ax.plot(epochs, cae_history['train_loss'], label='Train loss', lw=2)
ax.plot(epochs, cae_history['val_loss'],   label='Val loss',   lw=2)
ax.set_xlabel('Epoch');  ax.set_ylabel('MSE Loss')
ax.set_title('CAE Training Curve\n(trained on No Substructure only)',
             fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures', 'fig11_cae_training.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()


# ── 11.5  Residual visualisation ─────────────────────────────────────────────

def get_residuals(model, dataset, n=1, target_class=None):
    """
    Run CAE on n images of target_class.
    Returns originals, reconstructions, residuals as numpy arrays.
    """
    originals, reconstructions, residuals = [], [], []
    count = 0
    for img, label in dataset:
        if target_class is not None and label != target_class:
            continue
        if count >= n:
            break
        x = img.unsqueeze(0).to(device)
        with torch.no_grad():
            recon = model(x)
        orig = img.squeeze().cpu().numpy()
        rec  = recon.squeeze().cpu().numpy()
        originals.append(orig)
        reconstructions.append(rec)
        residuals.append(orig - rec)
        count += 1
    return originals, reconstructions, residuals


fig, axes = plt.subplots(3, 4, figsize=(18, 14))
col_titles = [
    'Original',
    'CAE Reconstruction\n(No Sub manifold only)',
    'Residual\n(input − reconstruction)',
    'Residual clipped\n(±2σ)'
]

for c in range(3):
    origs, recons, ress = get_residuals(cae, val_ds,
                                        n=1, target_class=c)
    orig, recon, res = origs[0], recons[0], ress[0]
    sigma = res.std()

    axes[c, 0].imshow(orig,  cmap='inferno')
    axes[c, 1].imshow(recon, cmap='inferno')
    axes[c, 2].imshow(res,   cmap='RdBu_r',
                      vmin=-3*sigma, vmax=3*sigma)
    axes[c, 3].imshow(np.clip(res, -2*sigma, 2*sigma),
                      cmap='RdBu_r',
                      vmin=-2*sigma, vmax=2*sigma)

    axes[c, 0].set_ylabel(CLASS_NAMES[c], fontsize=12,
                           fontweight='bold', rotation=90, labelpad=10)
    for j in range(4):
        axes[c, j].set_xticks([])
        axes[c, j].set_yticks([])

for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=10, fontweight='bold', pad=8)

plt.suptitle(
    'Section 11 — Convolutional Autoencoder Residuals\n'
    'CAE trained on No Substructure only — '
    'residuals isolate substructure signal\n'
    'No Sub residuals should be noise-like; '
    'Sphere/Vortex should show compact structure',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures', 'fig11_cae_residuals.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()


# ── 11.6  Residual statistics across all three classes ───────────────────────
print('\n── Residual statistics (ring annulus, n=50 per class) ──')
print(f'{"Class":20s}  {"Mean res":>10s}  '
      f'{"Std res":>10s}  {"Mean |res|":>12s}')
print('─' * 62)

ring_np  = ring_mask(size=150, inner_frac=0.15, outer_frac=0.65)
ring_pct = ring_np.mean() * 100

for c in range(3):
    _, _, ress = get_residuals(cae, val_ds, n=50, target_class=c)
    all_res = np.stack(ress)          # (50, 150, 150)
    in_ring = all_res[:, ring_np]     # (50, n_ring_pixels)
    print(f'{CLASS_NAMES[c]:20s}  '
          f'{in_ring.mean():>10.5f}  '
          f'{in_ring.std():>10.5f}  '
          f'{np.abs(in_ring).mean():>12.5f}')

print(f'\nRing annulus: {ring_pct:.1f}% of image pixels')
print('\nExpected pattern for a working CAE:')
print('  No Substructure std << Sphere std ≈ Vortex std')
print('  Substructure classes should show 2–5× higher residual std')


# ── 11.7  Residual distribution plot ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
colors_cls = ['#3498db', '#e74c3c', '#2ecc71']

all_class_res = []
for c in range(3):
    _, _, ress = get_residuals(cae, val_ds, n=50, target_class=c)
    all_res = np.stack(ress)
    in_ring = all_res[:, ring_np].flatten()
    all_class_res.append(in_ring)

    axes[c].hist(in_ring, bins=80, color=colors_cls[c],
                 alpha=0.75, density=True)
    axes[c].axvline(0, color='black', lw=1.5, linestyle='--')
    axes[c].set_title(
        f'{CLASS_NAMES[c]}\n'
        f'std = {in_ring.std():.5f}',
        fontsize=11, fontweight='bold'
    )
    axes[c].set_xlabel('Residual value', fontsize=10)

axes[0].set_ylabel('Density', fontsize=10)

plt.suptitle(
    'Figure 11b — Residual Distributions Inside Ring Annulus\n'
    'Wider distribution for substructure classes indicates '
    'CAE cannot reconstruct substructure perturbations',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures',
                 'fig11_residual_distributions.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()


# ── 11.8  Mean residual map per class (n=50 images) ──────────────────────────
# Average residual across 50 images per class.
# Random noise averages to zero. Systematic substructure signal persists.
# If Sphere/Vortex show a coherent pattern and No Sub shows flat noise,
# the CAE has successfully isolated the substructure contribution.

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
row_labels = ['Mean residual map\n(n=50 images)',
              'Std dev map\n(pixel-wise)']

for c in range(3):
    _, _, ress = get_residuals(cae, val_ds, n=50, target_class=c)
    stack    = np.stack(ress)          # (50, 150, 150)
    mean_map = stack.mean(axis=0)
    std_map  = stack.std(axis=0)
    vmax     = np.abs(mean_map).max()

    im0 = axes[0, c].imshow(mean_map, cmap='RdBu_r',
                             vmin=-vmax, vmax=vmax)
    axes[0, c].set_title(CLASS_NAMES[c], fontsize=12,
                          fontweight='bold')
    plt.colorbar(im0, ax=axes[0, c], shrink=0.8)

    im1 = axes[1, c].imshow(std_map, cmap='hot')
    axes[1, c].set_xlabel(
        f'Mean pixel std: {std_map.mean():.5f}',
        fontsize=9
    )
    plt.colorbar(im1, ax=axes[1, c], shrink=0.8)

    for row in range(2):
        axes[row, c].set_xticks([])
        axes[row, c].set_yticks([])

axes[0, 0].set_ylabel(row_labels[0], fontsize=10, fontweight='bold')
axes[1, 0].set_ylabel(row_labels[1], fontsize=10, fontweight='bold')

plt.suptitle(
    'Figure 11c — Mean and Std Residual Maps per Class (n=50)\n'
    'Random noise averages to zero — coherent signal survives averaging\n'
    'Sphere/Vortex should show a structured mean map; '
    'No Sub should be flat',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures',
                 'fig11_mean_residual_maps.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

### <a id="84-residual-visualisation--statistics"></a>8.4 Residual Visualisation & Statistics

The CAE trains stably, with validation MSE decreasing to $9.4\times10^{-5}$ after 60 epochs on No-Substructure images. This indicates that the model reconstructs the smooth class well on held-out examples from the same class.

**Residual statistics inside the ring annulus ($n=50$ per class):**

| Class | Mean residual | Std residual | Mean $|\text{residual}|$ |
|---|---:|---:|---:|
| No Substructure | -0.00065 | 0.01503 | 0.01038 |
| Sphere | -0.00028 | 0.01667 | 0.01094 |
| Vortex | -0.00050 | 0.01746 | 0.01115 |

The class-dependent pattern is present, but modest. Relative to No Substructure, the residual standard deviation is slightly higher for both Sphere and Vortex. This is consistent with the CAE leaving behind some substructure-related signal, but the gap is small compared with the reconstruction noise floor.

That is the main limitation of this branch: the residuals are not cleanly separated by class. The method appears to extract some additional structure, but not strongly enough to outperform direct classification on the original images.

### <a id="86-residual-classifier-densenet-121"></a>8.5 Residual Classifier: DenseNet-121

The next question is practical: even if the CAE residuals show only a modest class-dependent difference, is that difference large enough for a classifier to use?

To test this, DenseNet-121 is trained from scratch on the precomputed residual images. This gives a direct architecture-matched comparison with the raw-image benchmark: the classifier backbone is the same, but the input representation is changed from the original image to the CAE residual.

The result is:

| Model | Macro AUC | Validation Accuracy |
|---|---:|---:|
| DenseNet-121 (raw images) | 0.9950 | 96.95% |
| DenseNet-121 (residual images) | 0.9614 | 84.88% |

Residual classification remains clearly above chance, so the residual maps do retain useful class information. But the performance drop is large, especially for **Sphere recall** (0.71), showing that the residual representation is substantially weaker than the original image for this task.

The main takeaway is therefore limited but clear: CAE residuals preserve some discriminative structure, but not enough to compete with direct classification on raw images.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Section 11.5 — Residual Classifier: DenseNet-121 (from scratch)
#
# DenseNet-121 is already trained on raw lensing images in Section 4
# (macro AUC 0.9950, val accuracy 96.95%). Here we train an identical
# architecture from scratch on CAE residual images only.
#
# This creates a controlled comparison:
#   Same architecture — different input signal
#   Raw images:    full lens morphology available
#   Residuals:     macro-lens signal removed, perturbation signal only
#
# The AUC gap between the two DenseNet-121 runs quantifies exactly
# how much discriminative information is lost by removing the
# macro-lens component and retaining only the residual signal.
# ═════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc as sk_auc
)
from sklearn.preprocessing import label_binarize


# ── 11.5.1  Load precomputed residuals ───────────────────────────────────────
# Residuals were precomputed and saved in Section 11.4.
# Loading from disk — no CAE forward pass needed.

res_train_path = os.path.join(gsoc_drive_path, 'residuals_train.npz')
res_val_path   = os.path.join(gsoc_drive_path, 'residuals_val.npz')

print('Loading precomputed residuals...')
train_data = np.load(res_train_path)
val_data   = np.load(res_val_path)

res_train_X = train_data['residuals']   # (30000, 150, 150)
res_train_y = train_data['labels']      # (30000,)
res_val_X   = val_data['residuals']     # (7500, 150, 150)
res_val_y   = val_data['labels']        # (7500,)

print(f'  Train: {res_train_X.shape}')
print(f'  Val:   {res_val_X.shape}')

dn_res_train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(res_train_X).unsqueeze(1).float(),
        torch.from_numpy(res_train_y).long()
    ),
    batch_size=64, shuffle=True, num_workers=2, pin_memory=True
)
dn_res_val_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(res_val_X).unsqueeze(1).float(),
        torch.from_numpy(res_val_y).long()
    ),
    batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)
print(f'  Train batches: {len(dn_res_train_loader)}'
      f'  Val batches:   {len(dn_res_val_loader)}')


# ── 11.5.2  Architecture ──────────────────────────────────────────────────────

def build_densenet_residual():
    """
    DenseNet-121 adapted for single-channel 150×150 residual images.

    Two modifications from the standard architecture:
      1. features.conv0: 3-channel → 1-channel input
      2. classifier:     1000-class → 3-class output head

    Trained from scratch (weights=None).

    DenseNet's dense connectivity is well-suited to low-SNR inputs:
    each layer receives feature maps from all preceding layers,
    allowing the network to combine weak signals across many
    receptive field sizes simultaneously. This is the same property
    that makes DenseNet-121 the best-performing raw-image classifier
    in Section 4 — here we test whether it transfers to residuals.
    """
    model = models.densenet121(weights=None)
    # Replace first conv: 3-channel → 1-channel
    model.features.conv0 = nn.Conv2d(
        1, 64, kernel_size=7, stride=2, padding=3, bias=False
    )
    # Replace classifier head: 1024 → 3
    model.classifier = nn.Linear(model.classifier.in_features, 3)
    return model


dn_res = build_densenet_residual().to(device)

# Verify
_t   = torch.zeros(2, 1, 150, 150).to(device)
_out = dn_res(_t)
assert _out.shape == (2, 3), f'Unexpected output shape: {_out.shape}'
n_params = sum(p.numel() for p in dn_res.parameters() if p.requires_grad)
print(f'\nDenseNet-121 (residual) output: {_out.shape}  ✓')
print(f'Parameters: {n_params/1e6:.2f}M')
del _t, _out


# ── 11.5.3  Training ──────────────────────────────────────────────────────────

def train_densenet_residual(model, train_loader, val_loader,
                             n_epochs=30, lr=1e-3,
                             save_path=None, history_path=None):
    """
    Train DenseNet-121 on residual images.

    30 epochs with CosineAnnealingLR — consistent with the ResNet-18
    residual classifier (Section 11.6) for a fair architectural
    comparison on identical data and training budget.

    Checkpoint criterion: lowest validation loss — consistent with
    all raw-image classifiers in Sections 2–5.
    """
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-5
    )
    criterion = nn.CrossEntropyLoss()
    history   = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val  = float('inf')    # checkpoint on lowest val loss

    for epoch in range(1, n_epochs + 1):

        # ── train ──────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for res, labels in train_loader:
            res, labels = res.to(device), labels.to(device)
            logits      = model(res)
            loss        = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * res.size(0)
        train_loss /= len(train_loader.dataset)

        # ── validate ───────────────────────────────────────────────
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for res, labels in val_loader:
                res, labels = res.to(device), labels.to(device)
                logits      = model(res)
                loss        = criterion(logits, labels)
                val_loss   += loss.item() * res.size(0)
                correct    += (logits.argmax(1) == labels).sum().item()
                total      += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc   = correct / total

        scheduler.step()
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Save checkpoint on lowest val loss
        if val_loss < best_val:
            best_val = val_loss
            if save_path:
                torch.save(model.state_dict(), save_path)

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{n_epochs}  '
                  f'train={train_loss:.4f}  '
                  f'val={val_loss:.4f}  '
                  f'acc={val_acc:.4f}  '
                  f'best={best_val:.4f}')

    if history_path:
        np.save(history_path, history)
    return history


# ── Load or train ─────────────────────────────────────────────────────────────
dn_res_save_path = os.path.join(gsoc_drive_path, 'dn_res_best.pth')
dn_res_hist_path = os.path.join(gsoc_drive_path, 'dn_res_history.npy')

if os.path.exists(dn_res_save_path):
    dn_res.load_state_dict(
        torch.load(dn_res_save_path, map_location=device)
    )
    print(f'Loaded DenseNet-121 residual classifier from {dn_res_save_path}')
    dn_res_history = np.load(
        dn_res_hist_path, allow_pickle=True
    ).item() if os.path.exists(dn_res_hist_path) else None
else:
    print('Training DenseNet-121 on residual images from scratch...')
    dn_res_history = train_densenet_residual(
        dn_res,
        dn_res_train_loader,
        dn_res_val_loader,
        n_epochs=30,
        lr=1e-3,
        save_path=dn_res_save_path,
        history_path=dn_res_hist_path
    )
    print(f'\nTraining complete. Best weights saved to {dn_res_save_path}')

dn_res.load_state_dict(torch.load(dn_res_save_path, map_location=device))
dn_res.eval()
print('DenseNet-121 residual classifier ready.')


# ── 11.5.4  Training curve ────────────────────────────────────────────────────
if dn_res_history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    epochs = range(1, len(dn_res_history['train_loss']) + 1)

    axes[0].plot(epochs, dn_res_history['train_loss'],
                 label='Train loss', lw=2)
    axes[0].plot(epochs, dn_res_history['val_loss'],
                 label='Val loss',   lw=2)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CE Loss')
    axes[0].set_title('DenseNet-121 Residual — Loss Curves',
                      fontweight='bold')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, dn_res_history['val_acc'],
                 lw=2, color='#e74c3c')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Accuracy')
    axes[1].set_title('DenseNet-121 Residual — Val Accuracy',
                      fontweight='bold')
    axes[1].grid(alpha=0.3)

    plt.suptitle(
        'Figure 11e — DenseNet-121 Residual Classifier Training Curve\n'
        'Trained from scratch on CAE residual images only',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(gsoc_drive_path, 'figures',
                     'fig11e_dn_res_training.png'),
        dpi=150, bbox_inches='tight'
    )
    plt.show()


# ── 11.5.5  Evaluation ────────────────────────────────────────────────────────
dn_res_probs, dn_res_preds, dn_res_labels = [], [], []

with torch.no_grad():
    for res, labels in dn_res_val_loader:
        res    = res.to(device)
        logits = dn_res(res)
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        dn_res_probs.append(probs)
        dn_res_preds.append(probs.argmax(1))
        dn_res_labels.append(labels.numpy())

dn_res_probs  = np.concatenate(dn_res_probs,  axis=0)
dn_res_preds  = np.concatenate(dn_res_preds,  axis=0)
dn_res_labels = np.concatenate(dn_res_labels, axis=0)

labels_bin = label_binarize(dn_res_labels, classes=[0, 1, 2])
dn_res_aucs = {}
for c in range(3):
    fpr, tpr, _ = roc_curve(labels_bin[:, c], dn_res_probs[:, c])
    dn_res_aucs[c] = sk_auc(fpr, tpr)

macro_auc_dn_res = np.mean(list(dn_res_aucs.values()))
val_acc_dn_res   = (dn_res_preds == dn_res_labels).mean()

print('\n── DenseNet-121 Residual Classifier Results ─────────────────')
print(f'  Val accuracy : {val_acc_dn_res*100:.2f}%')
print(f'  Macro AUC    : {macro_auc_dn_res:.4f}')
for c in range(3):
    print(f'  AUC {CLASS_NAMES[c]:20s}: {dn_res_aucs[c]:.4f}')

print('\n── Classification Report ────────────────────────────────────')
print(classification_report(dn_res_labels, dn_res_preds,
                             target_names=CLASS_NAMES))

print('\n── Controlled comparison: DenseNet-121 raw vs residual ──────')
print(f'{"Model":40s}  {"Macro AUC":>10s}  {"Val Acc":>8s}')
print('─' * 65)
print(f'{"DenseNet-121 (raw images, Section 4)":40s}  '
      f'{"0.9950":>10s}  {"96.95%":>8s}')
print(f'{"DenseNet-121 (residuals, from scratch)":40s}  '
      f'{macro_auc_dn_res:>10.4f}  {val_acc_dn_res*100:>7.2f}%')


# ── 11.5.6  Confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(dn_res_labels, dn_res_preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Reds',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=ax, linewidths=0.5
)
ax.set_title(
    'Figure 11f — DenseNet-121 Residual Classifier Confusion Matrix\n'
    'Trained from scratch on CAE residual images (input − reconstruction)',
    fontweight='bold'
)
ax.set_ylabel('True Physical Class', fontsize=11)
ax.set_xlabel('Predicted Class',     fontsize=11)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures',
                 'fig11f_dn_res_confusion_matrix.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()


# ── 11.5.7  ROC curves ────────────────────────────────────────────────────────
colors_cls = ['#3498db', '#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left — per-class ROC
for c in range(3):
    fpr, tpr, _ = roc_curve(labels_bin[:, c], dn_res_probs[:, c])
    axes[0].plot(fpr, tpr, color=colors_cls[c], lw=2,
                 label=f'{CLASS_NAMES[c]}  AUC={dn_res_aucs[c]:.4f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate',  fontsize=11)
axes[0].set_title(
    f'DenseNet-121 Residual Classifier ROC\nMacro AUC = {macro_auc_dn_res:.4f}',
    fontsize=12, fontweight='bold'
)
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Right — controlled comparison bar chart
# DenseNet-121 raw vs residual side by side, plus ResNet-18 residual for context
# ResNet-18 residual AUC populated from Section 11.6 after that cell runs
model_names     = [
    'DenseNet-121\n(raw images)',
    'DenseNet-121\n(residuals)',
    'ResNet-18\n(residuals)',
]
model_aucs_plot = [
    0.9950,
    macro_auc_dn_res,
    macro_auc_res18 if 'macro_auc_res18' in dir() else 0.963
]
bar_colors_plot = ['#95a5a6', '#e74c3c', '#f39c12']

bars = axes[1].bar(model_names, model_aucs_plot,
                   color=bar_colors_plot, edgecolor='white', width=0.5)
axes[1].set_ylim(0.7, 1.01)
axes[1].set_ylabel('Macro AUC', fontsize=11)
axes[1].set_title(
    'DenseNet-121: Raw Images vs Residuals\n'
    'AUC gap = information lost by removing macro-lens signal',
    fontsize=11, fontweight='bold'
)
axes[1].grid(axis='y', alpha=0.3)

for bar, val in zip(bars, model_aucs_plot):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.002, f'{val:.4f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

plt.suptitle(
    'Figure 11g — DenseNet-121 Residual Classifier ROC\n'
    'Red bar = DenseNet on residuals  |  '
    'Grey bar = same architecture on raw images (Section 4)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures',
                 'fig11g_dn_res_roc.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

### <a id="87-residual-classifier-resnet-18"></a>8.6 Residual Classifier: ResNet-18

To check whether the residual branch is limited mainly by the input representation or by the classifier choice, ResNet-18 is also trained from scratch on the same residual images.

The residual results are:

| Model | Macro AUC | Validation Accuracy | Parameters |
|---|---:|---:|---:|
| DenseNet-121 (residuals) | 0.9614 | 84.88% | 6.95M |
| ResNet-18 (residuals) | 0.9543 | 83.80% | 11.17M |

The two models perform similarly, with DenseNet-121 only modestly ahead. This suggests that changing the classifier backbone does not substantially close the gap between residual classification and raw-image classification.

The cautious interpretation is that the main limitation of this branch is the residual representation itself, not a clear failure of one specific classifier architecture.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Section 11.6 — Residual Classifier: ResNet-18 (from scratch)
#
# ResNet-18 is already trained on raw lensing images in Section 3
# (macro AUC 0.9927, val accuracy 95.24%). Here we train an identical
# architecture from scratch on CAE residual images only.
#
# Paired with Section 11.5 (DenseNet-121 on residuals), this section
# answers whether the residual classification ceiling (~0.964) is
# architecture-dependent or signal-limited. Convergence of two
# architectures with different connectivity and parameter counts to
# the same AUC would confirm the plateau is set by residual SNR,
# not model capacity.
# ═════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc as sk_auc
)
from sklearn.preprocessing import label_binarize


# ── 11.6.1  DataLoaders ───────────────────────────────────────────────────────
# Residuals already loaded in Section 11.5 as res_train_X / res_val_X.
# Build separate loaders under new names to keep sections self-contained
# in case cells are run out of order.

res_train_path = os.path.join(gsoc_drive_path, 'residuals_train.npz')
res_val_path   = os.path.join(gsoc_drive_path, 'residuals_val.npz')

if 'res_train_X' not in dir():
    print('Loading precomputed residuals...')
    train_data  = np.load(res_train_path)
    val_data    = np.load(res_val_path)
    res_train_X = train_data['residuals']
    res_train_y = train_data['labels']
    res_val_X   = val_data['residuals']
    res_val_y   = val_data['labels']
    print(f'  Train: {res_train_X.shape}  Val: {res_val_X.shape}')
else:
    print('Residuals already in memory — skipping reload.')

r18_res_train_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(res_train_X).unsqueeze(1).float(),
        torch.from_numpy(res_train_y).long()
    ),
    batch_size=64, shuffle=True, num_workers=2, pin_memory=True
)
r18_res_val_loader = DataLoader(
    TensorDataset(
        torch.from_numpy(res_val_X).unsqueeze(1).float(),
        torch.from_numpy(res_val_y).long()
    ),
    batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)
print(f'  Train batches: {len(r18_res_train_loader)}'
      f'  Val batches:   {len(r18_res_val_loader)}')


# ── 11.6.2  Architecture ──────────────────────────────────────────────────────

def build_resnet18_residual():
    """
    ResNet-18 adapted for single-channel 150×150 residual images.

    Two modifications from the standard architecture:
      1. conv1: 3-channel → 1-channel input
      2. fc:    1000-class → 3-class output head

    Trained from scratch (weights=None).

    ResNet-18's residual skip connections allow gradients to flow
    cleanly through shallow layers — useful when the signal is weak
    and early layers risk vanishing gradients. At 11.7M parameters
    it is larger than DenseNet-121 (6.95M) but achieves similar
    residual AUC, confirming the ceiling is signal-limited.
    """
    model = models.resnet18(weights=None)
    # Replace first conv: 3-channel → 1-channel
    model.conv1 = nn.Conv2d(
        1, 64, kernel_size=7, stride=2, padding=3, bias=False
    )
    # Replace classifier head: 512 → 3
    model.fc = nn.Linear(model.fc.in_features, 3)
    return model


r18_res = build_resnet18_residual().to(device)

# Verify
_t   = torch.zeros(2, 1, 150, 150).to(device)
_out = r18_res(_t)
assert _out.shape == (2, 3), f'Unexpected output shape: {_out.shape}'
n_params = sum(p.numel() for p in r18_res.parameters() if p.requires_grad)
print(f'\nResNet-18 (residual) output: {_out.shape}  ✓')
print(f'Parameters: {n_params/1e6:.2f}M')
del _t, _out


# ── 11.6.3  Training ──────────────────────────────────────────────────────────

def train_resnet18_residual(model, train_loader, val_loader,
                             n_epochs=30, lr=1e-3,
                             save_path=None, history_path=None):
    """
    Train ResNet-18 on residual images.

    30 epochs with CosineAnnealingLR — identical training budget and
    optimiser configuration to Section 11.5 (DenseNet-121 residual)
    for a fair architectural comparison.

    Checkpoint criterion: lowest validation loss — consistent with
    all raw-image classifiers in Sections 2–5 and DenseNet-121
    residual classifier in Section 11.5.
    """
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs, eta_min=1e-5
    )
    criterion = nn.CrossEntropyLoss()
    history   = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val  = float('inf')    # checkpoint on lowest val loss

    for epoch in range(1, n_epochs + 1):

        # ── train ──────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for res, labels in train_loader:
            res, labels = res.to(device), labels.to(device)
            logits      = model(res)
            loss        = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * res.size(0)
        train_loss /= len(train_loader.dataset)

        # ── validate ───────────────────────────────────────────────
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for res, labels in val_loader:
                res, labels = res.to(device), labels.to(device)
                logits      = model(res)
                loss        = criterion(logits, labels)
                val_loss   += loss.item() * res.size(0)
                correct    += (logits.argmax(1) == labels).sum().item()
                total      += labels.size(0)
        val_loss /= len(val_loader.dataset)
        val_acc   = correct / total

        scheduler.step()
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Save checkpoint on lowest val loss
        if val_loss < best_val:
            best_val = val_loss
            if save_path:
                torch.save(model.state_dict(), save_path)

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{n_epochs}  '
                  f'train={train_loss:.4f}  '
                  f'val={val_loss:.4f}  '
                  f'acc={val_acc:.4f}  '
                  f'best={best_val:.4f}')

    if history_path:
        np.save(history_path, history)
    return history


# ── Load or train ─────────────────────────────────────────────────────────────
r18_res_save_path = os.path.join(gsoc_drive_path, 'res18_best.pth')
r18_res_hist_path = os.path.join(gsoc_drive_path, 'res18_history.npy')

if os.path.exists(r18_res_save_path):
    r18_res.load_state_dict(
        torch.load(r18_res_save_path, map_location=device)
    )
    print(f'Loaded ResNet-18 residual classifier from {r18_res_save_path}')
    r18_res_history = np.load(
        r18_res_hist_path, allow_pickle=True
    ).item() if os.path.exists(r18_res_hist_path) else None
else:
    print('Training ResNet-18 on residual images from scratch...')
    r18_res_history = train_resnet18_residual(
        r18_res,
        r18_res_train_loader,
        r18_res_val_loader,
        n_epochs=30,
        lr=1e-3,
        save_path=r18_res_save_path,
        history_path=r18_res_hist_path
    )
    print(f'\nTraining complete. Best weights saved to {r18_res_save_path}')

r18_res.load_state_dict(torch.load(r18_res_save_path, map_location=device))
r18_res.eval()
print('ResNet-18 residual classifier ready.')


# ── 11.6.4  Training curve ────────────────────────────────────────────────────
if r18_res_history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    epochs = range(1, len(r18_res_history['train_loss']) + 1)

    axes[0].plot(epochs, r18_res_history['train_loss'],
                 label='Train loss', lw=2)
    axes[0].plot(epochs, r18_res_history['val_loss'],
                 label='Val loss',   lw=2)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CE Loss')
    axes[0].set_title('ResNet-18 Residual — Loss Curves',
                      fontweight='bold')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, r18_res_history['val_acc'],
                 lw=2, color='#f39c12')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Accuracy')
    axes[1].set_title('ResNet-18 Residual — Val Accuracy',
                      fontweight='bold')
    axes[1].grid(alpha=0.3)

    plt.suptitle(
        'Figure 11h — ResNet-18 Residual Classifier Training Curve\n'
        'Trained from scratch on CAE residual images only',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(
        os.path.join(gsoc_drive_path, 'figures',
                     'fig11h_r18_res_training.png'),
        dpi=150, bbox_inches='tight'
    )
    plt.show()


# ── 11.6.5  Evaluation ────────────────────────────────────────────────────────
r18_res_probs, r18_res_preds, r18_res_labels = [], [], []

with torch.no_grad():
    for res, labels in r18_res_val_loader:
        res    = res.to(device)
        logits = r18_res(res)
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        r18_res_probs.append(probs)
        r18_res_preds.append(probs.argmax(1))
        r18_res_labels.append(labels.numpy())

r18_res_probs  = np.concatenate(r18_res_probs,  axis=0)
r18_res_preds  = np.concatenate(r18_res_preds,  axis=0)
r18_res_labels = np.concatenate(r18_res_labels, axis=0)

labels_bin_r18 = label_binarize(r18_res_labels, classes=[0, 1, 2])
r18_res_aucs   = {}
for c in range(3):
    fpr, tpr, _ = roc_curve(labels_bin_r18[:, c], r18_res_probs[:, c])
    r18_res_aucs[c] = sk_auc(fpr, tpr)

macro_auc_res18 = np.mean(list(r18_res_aucs.values()))
val_acc_res18   = (r18_res_preds == r18_res_labels).mean()

print('\n── ResNet-18 Residual Classifier Results ────────────────────')
print(f'  Val accuracy : {val_acc_res18*100:.2f}%')
print(f'  Macro AUC    : {macro_auc_res18:.4f}')
for c in range(3):
    print(f'  AUC {CLASS_NAMES[c]:20s}: {r18_res_aucs[c]:.4f}')

print('\n── Classification Report ────────────────────────────────────')
print(classification_report(r18_res_labels, r18_res_preds,
                             target_names=CLASS_NAMES))

print('\n── Architectural comparison on residuals ────────────────────')
print(f'{"Model":40s}  {"Macro AUC":>10s}  {"Val Acc":>8s}  {"Params":>8s}')
print('─' * 75)
print(f'{"DenseNet-121 (residuals, Section 11.5)":40s}  '
      f'{macro_auc_dn_res:>10.4f}  {val_acc_dn_res*100:>7.2f}%  {"6.95M":>8s}')
print(f'{"ResNet-18    (residuals, Section 11.6)":40s}  '
      f'{macro_auc_res18:>10.4f}  {val_acc_res18*100:>7.2f}%  {"11.7M":>8s}')


# ── 11.6.6  Confusion matrix ──────────────────────────────────────────────────
cm_r18 = confusion_matrix(r18_res_labels, r18_res_preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_r18, annot=True, fmt='d', cmap='Oranges',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=ax, linewidths=0.5
)
ax.set_title(
    'Figure 11i — ResNet-18 Residual Classifier Confusion Matrix\n'
    'Trained from scratch on CAE residual images (input − reconstruction)',
    fontweight='bold'
)
ax.set_ylabel('True Physical Class', fontsize=11)
ax.set_xlabel('Predicted Class',     fontsize=11)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures',
                 'fig11i_r18_res_confusion_matrix.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()


# ── 11.6.7  ROC curves ────────────────────────────────────────────────────────
colors_cls = ['#3498db', '#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left — per-class ROC
for c in range(3):
    fpr, tpr, _ = roc_curve(labels_bin_r18[:, c], r18_res_probs[:, c])
    axes[0].plot(fpr, tpr, color=colors_cls[c], lw=2,
                 label=f'{CLASS_NAMES[c]}  AUC={r18_res_aucs[c]:.4f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate',  fontsize=11)
axes[0].set_title(
    f'ResNet-18 Residual Classifier ROC\nMacro AUC = {macro_auc_res18:.4f}',
    fontsize=12, fontweight='bold'
)
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Right — full three-way comparison bar chart
# DenseNet-121 raw | DenseNet-121 residual | ResNet-18 residual
model_names     = [
    'DenseNet-121\n(raw images)',
    'DenseNet-121\n(residuals)',
    'ResNet-18\n(residuals)',
]
model_aucs_plot = [
    0.9950,
    macro_auc_dn_res,
    macro_auc_res18,
]
bar_colors_plot = ['#95a5a6', '#e74c3c', '#f39c12']

bars = axes[1].bar(model_names, model_aucs_plot,
                   color=bar_colors_plot, edgecolor='white', width=0.5)
axes[1].set_ylim(0.7, 1.01)
axes[1].set_ylabel('Macro AUC', fontsize=11)
axes[1].set_title(
    'Residual Classification Ceiling\n'
    'DenseNet-121 and ResNet-18 converge to the same AUC on residuals',
    fontsize=11, fontweight='bold'
)
axes[1].grid(axis='y', alpha=0.3)

for bar, val in zip(bars, model_aucs_plot):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        val + 0.002, f'{val:.4f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

plt.suptitle(
    'Figure 11j — ResNet-18 Residual Classifier ROC\n'
    'Orange bar = ResNet-18 on residuals  |  '
    'Red bar = DenseNet-121 on residuals  |  '
    'Grey bar = DenseNet-121 on raw images',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    os.path.join(gsoc_drive_path, 'figures',
                 'fig11j_r18_res_roc.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

### 8.7 Summary & Interpretation

| Model | Input | Macro AUC | Validation Accuracy | Params | Training |
|---|---|---:|---:|---:|---|
| DenseNet-121 | Raw images | 0.9950 | 96.95% | 6.95M | Pretrained |
| DenseNet-121 | CAE residuals | 0.9614 | 84.88% | 6.95M | Scratch |
| ResNet-18 | CAE residuals | 0.9543 | 83.80% | 11.17M | Scratch |

**Per-class AUC on residuals:**

| Class | DenseNet-121 | ResNet-18 |
|---|---:|---:|
| No Substructure | 0.9724 | 0.9683 |
| Sphere | 0.9488 | 0.9390 |
| Vortex | 0.9630 | 0.9556 |

Two conclusions are clear.

**1. Residual classification works, but is substantially weaker than raw-image classification.**  
Both residual classifiers perform well above chance, so the CAE residuals do retain useful class information. But both remain far below the raw-image DenseNet-121 baseline. In this benchmark, the residual representation is informative but not competitive with direct classification on the original images.

**2. The dominant limitation appears to be the residual representation, not the choice of classifier.**  
DenseNet-121 is modestly better than ResNet-18 on residuals, but the gap is small relative to the much larger drop from raw images to residuals. That suggests most of the performance loss comes from the residual preprocessing itself rather than from a specific classifier backbone.

The training curves are also consistent with this picture: both residual classifiers achieve low training loss while validation loss worsens early, indicating poor generalization from the residual input. A cautious interpretation is that the residual maps contain only weak class-dependent structure relative to reconstruction noise, making them easy to fit on the training set but harder to use robustly on held-out data.

The raw-vs-residual DenseNet comparison should still be read with one caveat: the raw-image model is pretrained, while the residual classifier is trained from scratch. So the performance gap should not be interpreted as a pure estimate of the preprocessing penalty alone. Even with that caveat, the gap is large enough that the main practical conclusion remains unchanged.

Sphere remains the hardest class in the residual setting as well, with the lowest per-class AUC for both classifiers. This is consistent with the broader pattern in the notebook: Sphere is the most difficult class whether the input is the original image or a residualized representation.

Overall, the residual branch is scientifically useful because it shows that some substructure-related signal survives CAE subtraction. But for three-class classification on this dataset, the stronger engineering choice is still to classify the raw images directly.

##<a id="8-residual-image-approach"></a> 9. Discussion

### 9.1 The Sphere Class is Systematically Harder

Across the stronger models in this benchmark, the **Sphere** class is consistently the most difficult. The dominant error is **Sphere $\rightarrow$ No Substructure**, while the No Substructure / Vortex boundary is much easier. This pattern is consistent with the visual character of the classes: Sphere perturbations are compact and relatively subtle, whereas Vortex perturbations are typically more extended and asymmetric.

For that reason, **Sphere-specific metrics** are more informative than macro AUC alone. In particular, Sphere recall and Sphere PR-AUC differentiate models that look similar under macro-averaged performance.

The detailed failure analysis supports the same conclusion. For both **EqDenseNet-C8** and **E-ResNet**, Sphere false negatives are associated with **lower ring mean flux** and **higher compactness**, while coarse measures such as asymmetry and centroid position do not separate the groups well. The two models differ in error rate, but the morphological signature of the missed Sphere cases is very similar, which suggests a shared hard subset rather than two qualitatively different failure regimes.

The cross-model cohort analysis points in the same direction. The **54 Sphere images** missed by all 8 models are, on average, dimmer and slightly less variable than the universally correct Sphere images. That does not prove a hard detection limit, but it does indicate that the hardest Sphere cases occupy a weaker-signal region of the simulated data distribution.

Taken together, the benchmark suggests a simple interpretation: the main challenge is not distinguishing Vortex from the other classes, but detecting the weakest compact Sphere perturbations against the smooth lens background. That is why improvements on Sphere-specific performance are more scientifically meaningful here than small gains in macro AUC alone.

### 9.2 Skip Connections Strongly Improve Performance in This Benchmark

| Architecture | Macro AUC | Params | Skip connections |
|:--|--:|--:|:--:|
| AlexNet | 0.6589 | 57M | ✗ |
| VGG-16 | 0.8944 | 134M | ✗ |
| ResNet-18 | 0.9927 | 11.2M | ✓ |
| EqDenseNet-C8 | 0.9973 | 0.093M | ✓ (dense) |

The largest performance jump in the benchmark is between the plain sequential CNNs and the skip-connected models. Even with far more parameters, VGG-16 remains well below ResNet-18, and AlexNet performs substantially worse than the stronger modern architectures. Within this benchmark, skip-connected models are consistently superior.

That said, this is not a clean one-factor causal test. The sequential models differ from the residual and dense models not only in connectivity, but also in normalization, optimization behavior, and compatibility with pretraining. So the safest interpretation is empirical: architectures with skip connections are much better suited to this task under the present training setup.

The equivariant models point in the same direction. EqDenseNet-C8 achieves the best overall result at very small parameter count, while the shallower Equivariant-D4 model is much weaker despite sharing a symmetry-aware inductive bias. This suggests that symmetry alone is not enough; feature reuse and information flow through the network also matter.

The practical takeaway is therefore: for this benchmark, skip connections appear to be a major architectural advantage, especially when the discriminative signal is subtle and localized.

### 9.3 Equivariant Inductive Bias Improves Parameter Efficiency

EqDenseNet-C8 achieves the strongest overall result in the benchmark while using very few parameters and no ImageNet pretraining. In particular, it slightly exceeds the large pretrained CNN baselines despite being much smaller. E-ResNet shows a similar pattern: it remains competitive with strong pretrained models at a much lower parameter count.

This is most naturally interpreted as a **parameter-efficiency** result rather than a sample-efficiency one. The benchmark does not include a data-scaling experiment, so the claim should remain limited: symmetry-aware architectures can perform very well on this task without large model size or external pretraining.

The parameter-efficiency plot supports that reading visually. EqDenseNet-C8 and E-ResNet occupy the low-parameter, high-AUC region of the benchmark, whereas the strongest non-equivariant baselines require substantially more parameters and, in several cases, ImageNet initialization.

The ablation results suggest that symmetry alone is not sufficient in every form: the weaker Equivariant-D4 model shows that equivariance without enough architectural capacity or feature reuse does not automatically reach top-tier performance. So the fairest interpretation is specific rather than universal: **for this benchmark, the combination of equivariant inductive bias with an effective architecture yields unusually strong performance at low parameter count.**

A secondary advantage is robustness. E-ResNet is also substantially more rotation-stable than ResNet-50 at the classifier output after training, which makes the equivariant result attractive not only in efficiency terms, but also in terms of symmetry consistency.

### 9.4 The Role of Augmentation in Equivariant Networks

Within the equivariant-family ablation, **augmentation has the largest and most consistent effect** on validation performance. The gains from adding $D_4$ augmentation are substantially larger than the differences associated with adding residual connections in the same small-model setting.

On macro AUC:
- Plain CNN: Aug $-$ NoAug = **+0.0090**
- E-ResNet: Aug $-$ NoAug = **+0.0073**

On Sphere recall:
- Plain CNN: **+0.0336**
- E-ResNet: **+0.0196**

By contrast, the residual effect is small in the no-augmentation setting and does not persist once augmentation is already present.

A cautious interpretation is that **architectural equivariance and data augmentation are complementary rather than redundant**. The equivariant layers encode the desired symmetry into the model class, but augmentation still improves how well that symmetry is realized under the actual training and image-discretization conditions of this benchmark.

The rotation-stability results point in the same direction: the equivariant models are much more stable than a standard CNN, but training does not preserve exact numerical invariance automatically. Augmentation appears to help close part of that gap in practice.

The role of residual connections is narrower here. In this shallow equivariant setting, they do not produce a large validation-performance gain, although they do appear to help preserve output stability under rotation after training. So the cleanest conclusion is not that residual connections are unimportant, but that **augmentation is the stronger performance-side factor in this particular ablation**.

### 9.5 Architecture Priors Aligned with the Task Appear Advantageous

Across the benchmark, the strongest models tend to be those whose architectural biases are well matched to the structure of the problem. This is a descriptive pattern in the current dataset, not a universal rule.

Two comparisons illustrate it.

**1. Symmetry-aware models vs standard CNNs.**  
The best equivariant models achieve top-tier performance at very small parameter count and without ImageNet pretraining. They also retain much stronger output stability under rotation than a standard CNN. This is consistent with the rotational symmetry prior being useful for lensing images, although the comparison is not cleanly attributable to symmetry alone because architecture family and training regime also differ.

**2. Dense local feature reuse vs patch-based global attention.**  
DenseNet-121 clearly outperforms ViT-Base on this benchmark, especially on the Sphere class. The interpretability analysis suggests a plausible reason: the difficult signal in Sphere appears to be a subtle local perturbation within the ring, and Dense convolutional feature reuse may be better matched to that kind of localized structure than patch-based transformer attention at this image scale. That interpretation remains suggestive rather than causal.

The common thread is modest but consistent: when the architecture encodes a prior that fits the task geometry — such as rotational symmetry or strong locality — the model seems to need less help from scale, pretraining, or data volume to perform well. In this benchmark, those task-aligned priors appear to be more useful than relying on a more flexible architecture to learn the same structure from data alone.

### 9.6 Deep Ensemble Uncertainty Reveals Two Distinct Failure Groups

The deep ensemble adds a useful reliability signal beyond single-model accuracy. In particular, it shows that difficult **Sphere** cases do not form one uniform uncertainty pattern.

The class-level entropy statistics are slightly counterintuitive: Sphere has the highest **mean** entropy, but not the highest **median**. This indicates a distribution with a heavier high-uncertainty tail rather than a uniformly uncertain class. In other words, many Sphere images are classified confidently, but a smaller subset is much more ambiguous than the corresponding tail in the other classes.

The more important result is the separation between two empirical failure groups:

| Failure group | Entropy | Approx. size | Operational implication |
|:--|:--:|:--:|:--|
| Silent failures | Low | 54 | Not flagged by entropy-based review |
| High-entropy ambiguous cases | High | $\sim 12$ | Naturally flagged for secondary review |

These two groups do not overlap in the current analysis. That matters because they suggest different practical problems. High-entropy cases are difficult, but at least the ensemble signals that difficulty. Silent failures are more serious: the ensemble is confidently wrong, so uncertainty alone does not catch them.

The cohort analysis suggests that the silent-failure Sphere images are also weaker on simple image-level statistics such as mean intensity and pixel standard deviation. That does not identify a single mechanism, but it supports the view that the most dangerous errors come from a shared hard subset of the data rather than from isolated architectural mistakes.

The main lesson is therefore practical: **uncertainty is useful, but incomplete**. Ensemble entropy can help identify ambiguous cases, but it cannot be treated as a general safeguard against error.

### 9.7 The Residual Image Approach — Scientific Value and Practical Ceiling

The residual-image branch reaches macro AUC **0.9614** with DenseNet-121 and **0.9543** with ResNet-18, remaining well below the corresponding raw-image baseline. At the same time, the residual statistics show a modest class-dependent effect: residual variation inside the ring is slightly higher for Sphere and Vortex than for No Substructure. So the residual maps do retain some substructure-related information, but not strongly enough to match direct classification on the original images.

Two limitations appear to define the practical ceiling of this branch.

First, the **residual representation is weak relative to the reconstruction noise floor**. Both residual classifiers generalize poorly compared with their raw-image counterparts, and the gap between DenseNet-121 and ResNet-18 on residuals is small relative to the much larger drop from raw images to residuals. This suggests that the representation itself is the dominant bottleneck, even if classifier choice still matters somewhat.

Second, the **analytical fitting route did not produce a clean alternative residual representation**. The forward-model fits were unstable and did not yield class-separating residuals, so the notebook ultimately relied on a data-driven CAE approximation instead. That does not prove that all analytical fitting strategies would fail, but it does show that the straightforward version was not usable here.

A likely reason the residual branch underperforms is the **homogeneity of the simulated dataset**. In this benchmark, the smooth lens morphology is already highly constrained, so the raw image itself remains a very informative input. Subtracting a learned smooth reconstruction removes some shared structure, but also introduces reconstruction noise, and the net result is worse classification performance.

The residual approach is therefore scientifically useful but practically limited in this setting. It shows that some discriminative signal survives subtraction, but not enough to compete with direct raw-image classification on this dataset. A stronger version of this idea may require either better residual extraction or a dataset regime in which macro-lens variation is a larger confound than it is here.

##<a id="8-residual-image-approach"></a> 10. Limitations

**Single random seed.**  
All experiments were run with one fixed seed. Run-to-run variance is therefore unknown, and very small metric differences should not be over-interpreted.

**Convergence and compute budget.**  
Some models, especially the equivariant ones, were still improving when training stopped. Their reported results may therefore be slightly conservative, but this was not tested systematically with longer schedules or repeated runs.

**Homogeneous simulation regime.**  
The dataset is highly structured, with limited variation in macro-lens parameters. Strong performance here does not guarantee similar behavior on more heterogeneous simulations or on real observational data.

**Asymmetric pretraining conditions.**  
Some models use ImageNet pretraining while the equivariant models are trained from scratch. This makes cross-family comparisons informative but not perfectly controlled.

**Discrete symmetry approximation.**  
The equivariant models use a $D_4$ symmetry prior, which is a useful discrete approximation in this benchmark but not an exact physical symmetry of real lens systems. More flexible rotational groups may be more appropriate for realistic observations.

**ViT input adaptation.**  
ViT-Base was adapted from its native pretrained resolution to this dataset through interpolation and patch-grid adjustment. That mismatch may have contributed to its weaker performance.

**Single-band input only.**  
All experiments use one simulated intensity channel. Real substructure analysis may benefit from additional information such as multi-band imaging or other observational context.

**Residual-branch limitations.**  
The CAE residual approach retained some class-dependent signal, but the residual classifiers generalized much more poorly than the raw-image models. This suggests that, in this benchmark, the residual representation is a major bottleneck.

**Failure-analysis limitations.**  
Some summary statistics used in the failure analysis are coarse. For example, centroid-based position measures may miss angular effects along the ring, so null results on position should be interpreted cautiously.

**Interpretability limitations.**  
CAM and Grad-CAM maps are constrained by the spatial resolution of the underlying feature maps and by the visualization method itself. The resulting heatmaps are useful for coarse spatial interpretation, but not for precise localization or causal claims.

##<a id="8-residual-image-approach"></a>11. Future Work

### 11.1 Architectural Directions

**Higher-order and continuous rotation equivariance.**  
The current equivariant models use discrete rotation groups ($D_4$ and $C_8$), which are useful approximations but do not cover arbitrary sky orientations exactly. A natural next step is to test higher cyclic groups ($C_{16}$, $C_{32}$) or continuous-group steerable CNNs. The goal would be to determine whether finer rotational structure improves performance and stability on more realistic lens populations.

**Scaling equivariant dense architectures.**  
EqDenseNet-C8 achieved the strongest result at very low parameter count, which makes it a strong base architecture for further scaling. A controlled study of wider or deeper equivariant dense networks would test whether the remaining Sphere errors are mainly a capacity issue or mainly a data/representation issue.

**Equivariant transformer variants.**  
ViT underperformed the strongest convolutional models in this benchmark. A useful follow-up would be to test transformer-style architectures that incorporate explicit rotational structure or ring-local attention, rather than relying on generic global patch attention alone.

**Ring-aware spatial gating.**  
Several analyses suggest that the discriminative signal is concentrated on the Einstein ring. A lightweight ring-localization module followed by a ring-focused classifier could test whether explicitly restricting computation to the physically relevant region reduces false positives and improves Sphere recall.



### 11.2 Detection and Estimation Extensions

**Subhalo mass regression.**  
The current pipeline predicts discrete classes only. A natural extension is to add a regression head for subhalo mass and test whether the model representation preserves **continuous lensing information related to perturbation strength and scale**, rather than only class identity.

**Anomaly-detection framing.**  
The silent-failure Sphere cases suggest that three-class classification is not the only useful formulation. A one-class or anomaly-detection approach, using No Substructure as the reference manifold, could test whether subtle substructure cases are better detected as deviations from smooth lenses than as standard multiclass labels.

**Uncertainty-aware triage.**  
The ensemble analysis showed that entropy captures some ambiguous cases but misses confident shared failures. A more complete triage pipeline could combine ensemble uncertainty with an additional anomaly-style score or image-quality statistic to detect both types of failure.



### 11.3 Robustness and Calibration

**Distribution-shift evaluation.**  
All current results are measured on validation data drawn from the same simulation pipeline as training. A necessary next step is to test robustness under controlled shifts in noise level, PSF, seeing, and lens morphology.

**Calibration under shifted class priors.**  
The benchmark uses balanced class proportions, whereas real survey data will not. Post-hoc calibration and threshold analysis would therefore be required before using model confidence operationally.

**Multi-channel or multi-band inputs.**  
All experiments here use a single simulated intensity channel. Adding extra channels, such as multi-band information in a more realistic setting, would test whether some of the current hard cases are limited by the available input representation rather than by classifier design alone.



### 11.4 Interpretability and Analysis

**Angular position along the ring.**  
The centroid-based position analysis was coarse. A more physically meaningful follow-up would parameterize perturbation position by angular coordinate along the fitted ring and test whether certain arc regions are systematically harder.

**Direct arc-coverage measurements.**  
Several failure analyses suggest that weaker or sparser ring structure is associated with errors. A direct measure of arc coverage or ring completeness would be more interpretable than global mean intensity alone.

**CAE bottleneck study.**  
The residual branch may depend strongly on bottleneck size. A systematic sweep over latent dimensionality would test the tradeoff between smooth-lens reconstruction quality and preservation of substructure-related residual signal.

##<a></a>12. Future DeepLense Research Directions

The benchmark suggests five concrete follow-up directions.

**1. Continuous-rotation equivariant lens classifiers.**  
Test whether moving beyond discrete $D_4/C_8$ symmetry improves performance and stability on lens images with arbitrary orientations.

**2. Silent-failure detection through anomaly scoring.**  
Use the CAE residual or a related one-class score to target the Sphere cases that remain confidently misclassified by all ensemble members.

**3. Ring-local perturbation detection.**  
Build a two-stage system that first identifies the ring geometry, then classifies perturbations in a ring-aligned coordinate frame.

**4. Joint classification and subhalo-mass estimation.**  
Extend the current classification setting to a more physically informative output space with calibrated uncertainty.

**5. Evaluation on heterogeneous lens populations.**  
Move beyond the current homogeneous simulation regime and test whether the observed advantage of symmetry-aware architectures persists when macro-lens properties vary more realistically.